In [ ]:
from google.colab import drive
import os, json

drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/EGFR_Tunisian_Digital_Twin"

folders = [
    "data/raw",
    "data/processed",
    "data/demo",
    "models",
    "outputs",
    "dashboard_exports",
    "reports"
]

for folder in folders:
    os.makedirs(os.path.join(BASE_DIR, folder), exist_ok=True)

print("Project folders created in:", BASE_DIR)

Mounted at /content/drive
Project folders created in: /content/drive/MyDrive/EGFR_Tunisian_Digital_Twin


In [ ]:
import pandas as pd
import os

demo_patients = [
    {
        "patient_id": "TN-LC-0001",
        "age": 61,
        "sex": "Male",
        "region": "Monastir",
        "smoking_status": "Former smoker",
        "cancer_type": "NSCLC",
        "histology": "Adenocarcinoma",
        "stage": "IV",
        "tnm": "T2N2M1",
        "metastasis": "Bone",
        "ecog": 1,
        "egfr": "Exon 19 deletion",
        "alk": "Unknown",
        "ros1": "Unknown",
        "pdl1": "Unknown",
        "current_treatment": "None",
        "best_global_option": "Osimertinib",
        "tunisia_availability": "Limited / requires verification"
    },
    {
        "patient_id": "TN-LC-0002",
        "age": 55,
        "sex": "Female",
        "region": "Tunis",
        "smoking_status": "Never smoker",
        "cancer_type": "NSCLC",
        "histology": "Adenocarcinoma",
        "stage": "III",
        "tnm": "T3N2M0",
        "metastasis": "None",
        "ecog": 0,
        "egfr": "L858R",
        "alk": "Negative",
        "ros1": "Unknown",
        "pdl1": "Unknown",
        "current_treatment": "Chemotherapy",
        "best_global_option": "Osimertinib",
        "tunisia_availability": "Limited / requires verification"
    },
    {
        "patient_id": "TN-LC-0003",
        "age": 68,
        "sex": "Male",
        "region": "Sousse",
        "smoking_status": "Current smoker",
        "cancer_type": "NSCLC",
        "histology": "Squamous cell carcinoma",
        "stage": "IV",
        "tnm": "T4N2M1",
        "metastasis": "Liver",
        "ecog": 2,
        "egfr": "Negative",
        "alk": "Unknown",
        "ros1": "Unknown",
        "pdl1": "High",
        "current_treatment": "None",
        "best_global_option": "Immunotherapy + chemotherapy",
        "tunisia_availability": "Requires verification"
    }
]

df = pd.DataFrame(demo_patients)

csv_path = os.path.join(BASE_DIR, "data/demo/tunisian_patients_demo.csv")
json_path = os.path.join(BASE_DIR, "data/demo/tunisian_patients_demo.json")

df.to_csv(csv_path, index=False)
df.to_json(json_path, orient="records", indent=2)

df

,patient_id,age,sex,region,smoking_status,cancer_type,histology,stage,tnm,metastasis,ecog,egfr,alk,ros1,pdl1,current_treatment,best_global_option,tunisia_availability
0,TN-LC-0001,61,Male,Monastir,Former smoker,NSCLC,Adenocarcinoma,IV,T2N2M1,Bone,1,Exon 19 deletion,Unknown,Unknown,Unknown,None,Osimertinib,Limited / requires verification
1,TN-LC-0002,55,Female,Tunis,Never smoker,NSCLC,Adenocarcinoma,III,T3N2M0,None,0,L858R,Negative,Unknown,Unknown,Chemotherapy,Osimertinib,Limited / requires verification
2,TN-LC-0003,68,Male,Sousse,Current smoker,NSCLC,Squamous cell carcinoma,IV,T4N2M1,Liver,2,Negative,Unknown,Unknown,High,None,Immunotherapy + chemotherapy,Requires verification


In [ ]:
def predict_patient(row):
    score = 50
    reasons = []

    if row["egfr"] in ["Exon 19 deletion", "L858R"]:
        score += 25
        reasons.append("EGFR sensitizing mutation detected")

    if row["stage"] == "IV":
        score -= 10
        reasons.append("Advanced metastatic stage")

    if row["ecog"] >= 2:
        score -= 10
        reasons.append("Reduced performance status")

    if row["histology"] == "Adenocarcinoma":
        score += 5
        reasons.append("Adenocarcinoma profile compatible with molecular targeting")

    score = max(0, min(100, score))

    if score >= 75:
        response = "High predicted benefit"
    elif score >= 55:
        response = "Intermediate predicted benefit"
    else:
        response = "Low predicted benefit"

    return {
        "patient_id": row["patient_id"],
        "predicted_efficiency": score,
        "predicted_response": response,
        "recommended_treatment": row["best_global_option"],
        "tunisia_availability": row["tunisia_availability"],
        "explanation": reasons
    }

predictions = [predict_patient(row) for _, row in df.iterrows()]

predictions_path = os.path.join(BASE_DIR, "data/demo/tunisian_predictions.json")

with open(predictions_path, "w") as f:
    json.dump(predictions, f, indent=2)

predictions

[{'patient_id': 'TN-LC-0001',
  'predicted_efficiency': 70,
  'predicted_response': 'Intermediate predicted benefit',
  'recommended_treatment': 'Osimertinib',
  'tunisia_availability': 'Limited / requires verification',
  'explanation': ['EGFR sensitizing mutation detected',
   'Advanced metastatic stage',
   'Adenocarcinoma profile compatible with molecular targeting']},
 {'patient_id': 'TN-LC-0002',
  'predicted_efficiency': 80,
  'predicted_response': 'High predicted benefit',
  'recommended_treatment': 'Osimertinib',
  'tunisia_availability': 'Limited / requires verification',
  'explanation': ['EGFR sensitizing mutation detected',
   'Adenocarcinoma profile compatible with molecular targeting']},
 {'patient_id': 'TN-LC-0003',
  'predicted_efficiency': 30,
  'predicted_response': 'Low predicted benefit',
  'recommended_treatment': 'Immunotherapy + chemotherapy',
  'tunisia_availability': 'Requires verification',
  'explanation': ['Advanced metastatic stage', 'Reduced performance s

In [ ]:
drug_availability = [
    {
        "drug": "Osimertinib",
        "indication": "EGFR-mutated NSCLC",
        "global_rank": 1,
        "tunisia_status": "Limited / requires hospital pharmacy verification",
        "alternatives": ["Erlotinib", "Gefitinib", "Afatinib", "Platinum-based chemotherapy"]
    },
    {
        "drug": "Erlotinib",
        "indication": "EGFR-mutated NSCLC",
        "global_rank": 2,
        "tunisia_status": "Possible local alternative / verify",
        "alternatives": ["Gefitinib", "Platinum-based chemotherapy"]
    },
    {
        "drug": "Platinum-based chemotherapy",
        "indication": "Advanced NSCLC",
        "global_rank": 3,
        "tunisia_status": "Likely more accessible / verify locally",
        "alternatives": ["Pemetrexed-based regimen", "Taxane-based regimen"]
    }
]

availability_path = os.path.join(BASE_DIR, "data/demo/drug_availability.json")

with open(availability_path, "w") as f:
    json.dump(drug_availability, f, indent=2)

drug_availability

[{'drug': 'Osimertinib',
  'indication': 'EGFR-mutated NSCLC',
  'global_rank': 1,
  'tunisia_status': 'Limited / requires hospital pharmacy verification',
  'alternatives': ['Erlotinib',
   'Gefitinib',
   'Afatinib',
   'Platinum-based chemotherapy']},
 {'drug': 'Erlotinib',
  'indication': 'EGFR-mutated NSCLC',
  'global_rank': 2,
  'tunisia_status': 'Possible local alternative / verify',
  'alternatives': ['Gefitinib', 'Platinum-based chemotherapy']},
 {'drug': 'Platinum-based chemotherapy',
  'indication': 'Advanced NSCLC',
  'global_rank': 3,
  'tunisia_status': 'Likely more accessible / verify locally',
  'alternatives': ['Pemetrexed-based regimen', 'Taxane-based regimen']}]

In [ ]:
drug_design_candidates = [
    {
        "target": "EGFR Exon 19 deletion",
        "candidate": "Osimertinib-like scaffold",
        "module_type": "Research only",
        "predicted_binding": "High",
        "docking_score": -9.1,
        "admet_risk": "Moderate",
        "evidence_level": "Computational hypothesis"
    },
    {
        "target": "EGFR L858R",
        "candidate": "Quinazoline-based EGFR inhibitor scaffold",
        "module_type": "Research only",
        "predicted_binding": "Medium",
        "docking_score": -8.2,
        "admet_risk": "Unknown",
        "evidence_level": "Computational hypothesis"
    },
    {
        "target": "EGFR resistance mutation",
        "candidate": "Allosteric EGFR inhibitor candidate",
        "module_type": "Research only",
        "predicted_binding": "Experimental",
        "docking_score": -7.6,
        "admet_risk": "Unknown",
        "evidence_level": "Early computational hypothesis"
    }
]

design_path = os.path.join(BASE_DIR, "data/demo/drug_design_candidates.json")

with open(design_path, "w") as f:
    json.dump(drug_design_candidates, f, indent=2)

drug_design_candidates

[{'target': 'EGFR Exon 19 deletion',
  'candidate': 'Osimertinib-like scaffold',
  'module_type': 'Research only',
  'predicted_binding': 'High',
  'docking_score': -9.1,
  'admet_risk': 'Moderate',
  'evidence_level': 'Computational hypothesis'},
 {'target': 'EGFR L858R',
  'candidate': 'Quinazoline-based EGFR inhibitor scaffold',
  'module_type': 'Research only',
  'predicted_binding': 'Medium',
  'docking_score': -8.2,
  'admet_risk': 'Unknown',
  'evidence_level': 'Computational hypothesis'},
 {'target': 'EGFR resistance mutation',
  'candidate': 'Allosteric EGFR inhibitor candidate',
  'module_type': 'Research only',
  'predicted_binding': 'Experimental',
  'docking_score': -7.6,
  'admet_risk': 'Unknown',
  'evidence_level': 'Early computational hypothesis'}]

In [ ]:
EXPORT_DIR = os.path.join(BASE_DIR, "dashboard_exports/demo")
os.makedirs(EXPORT_DIR, exist_ok=True)

files_to_copy = [
    "tunisian_patients_demo.json",
    "tunisian_predictions.json",
    "drug_availability.json",
    "drug_design_candidates.json"
]

for file in files_to_copy:
    src = os.path.join(BASE_DIR, "data/demo", file)
    dst = os.path.join(EXPORT_DIR, file)
    with open(src, "r") as f:
        data = json.load(f)
    with open(dst, "w") as f:
        json.dump(data, f, indent=2)

print("Dashboard demo files exported to:")
print(EXPORT_DIR)
print(os.listdir(EXPORT_DIR))

Dashboard demo files exported to:
/content/drive/MyDrive/EGFR_Tunisian_Digital_Twin/dashboard_exports/demo
['tunisian_patients_demo.json', 'tunisian_predictions.json', 'drug_availability.json', 'drug_design_candidates.json']


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

SEARCH_ROOT = "/content/drive/MyDrive"

matches = []

for root, dirs, files in os.walk(SEARCH_ROOT):
    if "package.json" in files:
        matches.append(root)

print("Found React/dashboard folders:")
for i, path in enumerate(matches):
    print(i, ":", path)

Mounted at /content/drive
Found React/dashboard folders:


In [ ]:
import os

SEARCH_ROOT = "/content/drive/MyDrive"

keywords = [
    "dashboard",
    "frontend",
    "digital",
    "twin",
    "egfr",
    "pfe",
    "lung",
    "cancer"
]

found = []

for root, dirs, files in os.walk(SEARCH_ROOT):
    # skip heavy/cache folders
    if "node_modules" in root or ".git" in root or "__pycache__" in root:
        continue

    folder_name = os.path.basename(root).lower()

    score = 0

    # important files
    if "package.json" in files:
        score += 10
    if "index.html" in files:
        score += 5
    if "vite.config.js" in files or "vite.config.ts" in files:
        score += 8
    if "App.jsx" in files or "App.js" in files:
        score += 8

    # folder name hints
    for k in keywords:
        if k in folder_name:
            score += 2

    if score >= 5:
        found.append((score, root, files[:10]))

found = sorted(found, reverse=True)

print("Possible dashboard folders:")
for i, item in enumerate(found[:30]):
    score, root, files = item
    print("\n", i, "| score:", score)
    print(root)
    print("files:", files)

Possible dashboard folders:

 0 | score: 7
/content/drive/MyDrive/pfe_digital_twin_app/frontend
files: ['index.gdoc', 'index.html']

 1 | score: 6
/content/drive/MyDrive/pfe_digital_twin_app
files: ['07_backend_api_colab.ipynb']

 2 | score: 6
/content/drive/MyDrive/pfe_digital_twin/pfe_digital_twin
files: ['S_Table_6_Copy_Number.xlsx', 'S_Table_13_RPPA.xlsx', 'EGFR_Mutations_Table7_Cleaned.xlsx', 'gdc_manifest.2026-04-15.002227.txt', 'gdc_manifest_full.2026-04-27.232209.txt']

 3 | score: 6
/content/drive/MyDrive/pfe_digital_twin
files: ['EGFR_Mutations_Table7_Cleaned.xlsx', 'Master_Twin_Data_Final.csv', 'data_rppa.txt', 'data_cna.txt', 'Master_MultiOmics_27.csv', 'gdc_manifest.2026-04-15.002227.gdoc', 'gdc_manifest.2026-04-15.002227.txt', 'DigitalTwin_Master_Final.csv', 'Twin_Predictive_Final.csv']

 4 | score: 6
/content/drive/MyDrive/PFE_DigitalTwin
files: ['clinical.cart.2026-04-28.tar.gz', 'Clinical_Cleaned.csv', 'Master_Patient_Profiles.csv']

 5 | score: 6
/content/drive/MyDriv

In [ ]:
from google.colab import drive
import os, json, shutil, re
from datetime import datetime

drive.mount('/content/drive')

FRONTEND_PATH = "/content/drive/MyDrive/pfe_digital_twin_app/frontend"

if not os.path.exists(FRONTEND_PATH):
    raise Exception("Frontend folder not found.")

INDEX_PATH = os.path.join(FRONTEND_PATH, "index.html")
if not os.path.exists(INDEX_PATH):
    raise Exception("index.html not found in frontend folder.")

DEMO_PATH = os.path.join(FRONTEND_PATH, "demo")
os.makedirs(DEMO_PATH, exist_ok=True)

print("Working in:", FRONTEND_PATH)
print("Demo data folder:", DEMO_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working in: /content/drive/MyDrive/pfe_digital_twin_app/frontend
Demo data folder: /content/drive/MyDrive/pfe_digital_twin_app/frontend/demo


In [ ]:
tunisian_patients = [
    {
        "patient_id": "TN-LC-0001",
        "age": 61,
        "sex": "Male",
        "region": "Monastir",
        "smoking_status": "Former smoker",
        "cancer_type": "NSCLC",
        "histology": "Adenocarcinoma",
        "stage": "IV",
        "tnm": "T2N2M1",
        "metastasis": "Bone",
        "ecog": 1,
        "egfr": "Exon 19 deletion",
        "alk": "Unknown",
        "ros1": "Unknown",
        "pdl1": "Unknown",
        "best_global_option": "Osimertinib",
        "tunisia_availability": "Limited / requires hospital pharmacy verification",
        "local_strategy": "Use EGFR-targeted therapy if available. If not accessible, consider locally available EGFR-TKI or platinum-based chemotherapy with close follow-up."
    },
    {
        "patient_id": "TN-LC-0002",
        "age": 55,
        "sex": "Female",
        "region": "Tunis",
        "smoking_status": "Never smoker",
        "cancer_type": "NSCLC",
        "histology": "Adenocarcinoma",
        "stage": "III",
        "tnm": "T3N2M0",
        "metastasis": "None",
        "ecog": 0,
        "egfr": "L858R",
        "alk": "Negative",
        "ros1": "Unknown",
        "pdl1": "Unknown",
        "best_global_option": "Osimertinib",
        "tunisia_availability": "Limited / requires hospital pharmacy verification",
        "local_strategy": "Molecularly guided EGFR therapy with monitoring for resistance and progression."
    },
    {
        "patient_id": "TN-LC-0003",
        "age": 68,
        "sex": "Male",
        "region": "Sousse",
        "smoking_status": "Current smoker",
        "cancer_type": "NSCLC",
        "histology": "Squamous cell carcinoma",
        "stage": "IV",
        "tnm": "T4N2M1",
        "metastasis": "Liver",
        "ecog": 2,
        "egfr": "Negative",
        "alk": "Unknown",
        "ros1": "Unknown",
        "pdl1": "High",
        "best_global_option": "Immunotherapy + chemotherapy",
        "tunisia_availability": "Requires verification",
        "local_strategy": "If immunotherapy access is limited, use chemotherapy-based local protocol and follow progression carefully."
    }
]

tunisian_predictions = [
    {
        "patient_id": "TN-LC-0001",
        "predicted_efficiency": 80,
        "predicted_response": "High predicted benefit",
        "resistance_risk": 42,
        "recommended_treatment": "Osimertinib",
        "explanation": [
            "EGFR exon 19 deletion detected",
            "Adenocarcinoma profile supports EGFR targeting",
            "Advanced stage requires systemic therapy",
            "Availability in Tunisia must be verified"
        ]
    },
    {
        "patient_id": "TN-LC-0002",
        "predicted_efficiency": 85,
        "predicted_response": "High predicted benefit",
        "resistance_risk": 35,
        "recommended_treatment": "Osimertinib",
        "explanation": [
            "EGFR L858R sensitizing mutation detected",
            "Good ECOG score",
            "Stage III disease may benefit from guided therapy"
        ]
    },
    {
        "patient_id": "TN-LC-0003",
        "predicted_efficiency": 58,
        "predicted_response": "Intermediate predicted benefit",
        "resistance_risk": 61,
        "recommended_treatment": "Immunotherapy + chemotherapy",
        "explanation": [
            "EGFR negative profile",
            "High PD-L1 may support immunotherapy",
            "ECOG 2 and metastatic disease increase risk",
            "Drug access must be checked locally"
        ]
    }
]

drug_availability = [
    {
        "drug": "Osimertinib",
        "indication": "EGFR-mutated NSCLC",
        "tunisia_status": "Limited / verify with hospital pharmacy",
        "alternatives": ["Erlotinib", "Gefitinib", "Afatinib", "Platinum-based chemotherapy"]
    },
    {
        "drug": "Erlotinib",
        "indication": "EGFR-mutated NSCLC",
        "tunisia_status": "Possible local alternative / verify",
        "alternatives": ["Gefitinib", "Afatinib", "Platinum-based chemotherapy"]
    },
    {
        "drug": "Gefitinib",
        "indication": "EGFR-mutated NSCLC",
        "tunisia_status": "Possible local alternative / verify",
        "alternatives": ["Erlotinib", "Afatinib", "Platinum-based chemotherapy"]
    },
    {
        "drug": "Platinum-based chemotherapy",
        "indication": "Advanced NSCLC",
        "tunisia_status": "More accessible local option",
        "alternatives": ["Pemetrexed-based regimen", "Taxane-based regimen"]
    }
]

drug_design_candidates = [
    {
        "target": "EGFR Exon 19 deletion",
        "candidate": "Osimertinib-like scaffold",
        "module_type": "Research only",
        "predicted_binding": "High",
        "docking_score": -9.1,
        "admet_risk": "Moderate",
        "evidence_level": "Computational hypothesis"
    },
    {
        "target": "EGFR L858R",
        "candidate": "Quinazoline-based EGFR inhibitor scaffold",
        "module_type": "Research only",
        "predicted_binding": "Medium",
        "docking_score": -8.2,
        "admet_risk": "Unknown",
        "evidence_level": "Computational hypothesis"
    },
    {
        "target": "EGFR resistance / MET bypass",
        "candidate": "Allosteric EGFR or MET-combination candidate",
        "module_type": "Research only",
        "predicted_binding": "Experimental",
        "docking_score": -7.6,
        "admet_risk": "Unknown",
        "evidence_level": "Early computational hypothesis"
    }
]

files = {
    "tunisian_patients_demo.json": tunisian_patients,
    "tunisian_predictions.json": tunisian_predictions,
    "drug_availability.json": drug_availability,
    "drug_design_candidates.json": drug_design_candidates
}

for filename, data in files.items():
    with open(os.path.join(DEMO_PATH, filename), "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)

print("Created:")
for f in os.listdir(DEMO_PATH):
    print("-", f)

Created:
- tunisian_patients_demo.json
- tunisian_predictions.json
- drug_availability.json
- drug_design_candidates.json


In [ ]:
tunisia_page = """
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>Tunisian Multimodal Digital Twin</title>
  <style>
    body { margin:0; font-family:Arial,sans-serif; background:#0f172a; color:#e5e7eb; }
    header { padding:28px 36px; background:#111827; border-bottom:1px solid #334155; }
    h1 { margin:0; font-size:32px; color:white; }
    .subtitle { color:#94a3b8; margin-top:8px; font-size:16px; }
    .container { padding:28px 36px; }
    .warning { background:#451a03; border-left:5px solid #f97316; padding:16px; border-radius:12px; margin-bottom:24px; color:#fed7aa; }
    .grid { display:grid; grid-template-columns:repeat(3,1fr); gap:18px; margin-bottom:24px; }
    .grid2 { display:grid; grid-template-columns:repeat(2,1fr); gap:18px; margin-bottom:24px; }
    .card { background:#111827; border:1px solid #334155; border-radius:18px; padding:20px; box-shadow:0 8px 20px rgba(0,0,0,.25); }
    .card h2 { margin-top:0; color:#f8fafc; font-size:20px; }
    .label { color:#94a3b8; font-size:14px; }
    .value { color:#38bdf8; font-size:24px; font-weight:bold; margin:6px 0 14px; }
    .pill { display:inline-block; background:#1d4ed8; color:white; padding:6px 10px; border-radius:20px; margin:4px 4px 4px 0; font-size:13px; }
    .green { background:#166534; }
    .orange { background:#b45309; }
    .red { background:#991b1b; }
    select { background:#020617; color:white; padding:12px; border-radius:10px; border:1px solid #334155; margin-bottom:20px; min-width:280px; font-size:15px; }
    table { width:100%; border-collapse:collapse; background:#111827; border-radius:18px; overflow:hidden; border:1px solid #334155; margin-bottom:24px; }
    th,td { padding:14px; border-bottom:1px solid #334155; text-align:left; vertical-align:top; }
    th { background:#1e293b; color:white; }
    td { color:#d1d5db; }
    .section-title { color:white; margin-top:35px; margin-bottom:12px; font-size:24px; }
    .bar-bg { background:#334155; height:26px; border-radius:20px; overflow:hidden; margin-top:8px; }
    .bar { background:#38bdf8; height:26px; line-height:26px; padding-left:10px; color:#020617; font-weight:bold; }
    a { color:#38bdf8; text-decoration:none; }
    @media(max-width:900px){ .grid,.grid2{ grid-template-columns:1fr; } }
  </style>
</head>
<body>
<header>
  <h1>Tunisian Multimodal Digital Twin for Lung Cancer</h1>
  <div class="subtitle">Precision medicine prototype adapted to Tunisia: clinical data, molecular profile, pathology reports, treatment availability and research drug-design support.</div>
  <p><a href="index.html">← Back to main dashboard</a></p>
</header>

<div class="container">
  <div class="warning">
    In Tunisia, digitized SVS pathology slides are not widely available. This version does not depend on SVS images.
    It uses clinical files, molecular tests, pathology reports, treatment history and follow-up. SVS and CT/radiomics can be added later.
  </div>

  <label class="label">Select Tunisian patient twin:</label><br>
  <select id="patientSelect"></select>

  <div class="grid">
    <div class="card">
      <h2>Patient Summary</h2>
      <div class="label">Patient ID</div>
      <div class="value" id="patientId"></div>
      <p id="patientDetails"></p>
    </div>

    <div class="card">
      <h2>Molecular Profile</h2>
      <div class="label">EGFR</div>
      <div class="value" id="egfr"></div>
      <div id="molecularPills"></div>
    </div>

    <div class="card">
      <h2>Recommended Strategy</h2>
      <div class="label">Best global option</div>
      <div class="value" id="bestTreatment"></div>
      <p id="localStrategy"></p>
    </div>
  </div>

  <div class="grid2">
    <div class="card">
      <h2>AI Prediction</h2>
      <div class="label">Predicted efficiency</div>
      <div class="bar-bg"><div class="bar" id="efficiencyBar"></div></div>
      <p><b>Predicted response:</b> <span id="predictedResponse"></span></p>
      <p><b>Resistance risk:</b> <span id="resistanceRisk"></span>%</p>
    </div>

    <div class="card">
      <h2>Explanation</h2>
      <ul id="explanationList"></ul>
    </div>
  </div>

  <h2 class="section-title">Multimodal Inputs</h2>
  <div class="grid2">
    <div class="card">
      <h2>Available Now</h2>
      <span class="pill green">Clinical data</span>
      <span class="pill green">Molecular tests</span>
      <span class="pill green">Pathology reports</span>
      <span class="pill green">Treatment history</span>
      <span class="pill green">Follow-up</span>
    </div>
    <div class="card">
      <h2>Future Optional Inputs</h2>
      <span class="pill orange">SVS pathology slides</span>
      <span class="pill orange">CT imaging</span>
      <span class="pill orange">Radiomics</span>
      <span class="pill orange">Proteomics</span>
      <span class="pill orange">Hospital API integration</span>
    </div>
  </div>

  <h2 class="section-title">Treatment Availability Layer</h2>
  <table id="availabilityTable">
    <thead>
      <tr><th>Drug</th><th>Indication</th><th>Tunisia Status</th><th>Alternatives</th></tr>
    </thead>
    <tbody></tbody>
  </table>

  <h2 class="section-title">Drug Design Research Module</h2>
  <div class="warning">
    This module is research-only. It is not a clinical prescription engine. It prioritizes candidate molecules for docking, ADMET filtering and laboratory validation.
  </div>
  <table id="designTable">
    <thead>
      <tr><th>Target</th><th>Candidate</th><th>Predicted Binding</th><th>Docking Score</th><th>Evidence Level</th></tr>
    </thead>
    <tbody></tbody>
  </table>

  <h2 class="section-title">Hospital / Syslab Integration Plan</h2>
  <div class="grid2">
    <div class="card">
      <h2>Data Entry MVP</h2>
      <p>The first Tunisian version can work with manual patient entry, Excel import, pathology report upload, molecular test upload and follow-up forms.</p>
    </div>
    <div class="card">
      <h2>Real Hospital Version</h2>
      <p>The future system can connect to hospital information systems, laboratory systems, oncology files and pharmacy availability to continuously update the digital twin.</p>
    </div>
  </div>
</div>

<script>
let patients = [];
let predictions = [];
let availability = [];
let designCandidates = [];

async function loadJSON(path) {
  const response = await fetch(path);
  if (!response.ok) throw new Error("Failed to load " + path);
  return await response.json();
}

function findPrediction(patientId) {
  return predictions.find(p => p.patient_id === patientId) || predictions[0];
}

function renderPatient(patient) {
  const prediction = findPrediction(patient.patient_id);

  document.getElementById("patientId").textContent = patient.patient_id;
  document.getElementById("patientDetails").innerHTML =
    `${patient.age} years old, ${patient.sex}<br>
     Region: ${patient.region}<br>
     Histology: ${patient.histology}<br>
     Stage: ${patient.stage} | TNM: ${patient.tnm}<br>
     ECOG: ${patient.ecog}`;

  document.getElementById("egfr").textContent = patient.egfr;
  document.getElementById("molecularPills").innerHTML =
    `<span class="pill">ALK: ${patient.alk}</span>
     <span class="pill">ROS1: ${patient.ros1}</span>
     <span class="pill">PD-L1: ${patient.pdl1}</span>`;

  document.getElementById("bestTreatment").textContent = patient.best_global_option;
  document.getElementById("localStrategy").innerHTML =
    `<b>Tunisia availability:</b> ${patient.tunisia_availability}<br><br>${patient.local_strategy}`;

  const efficiency = prediction.predicted_efficiency || 0;
  const bar = document.getElementById("efficiencyBar");
  bar.style.width = efficiency + "%";
  bar.textContent = efficiency + "%";

  document.getElementById("predictedResponse").textContent = prediction.predicted_response;
  document.getElementById("resistanceRisk").textContent = prediction.resistance_risk;

  document.getElementById("explanationList").innerHTML =
    prediction.explanation.map(x => `<li>${x}</li>`).join("");
}

function renderAvailability() {
  const tbody = document.querySelector("#availabilityTable tbody");
  tbody.innerHTML = availability.map(d => `
    <tr>
      <td><b>${d.drug}</b></td>
      <td>${d.indication}</td>
      <td>${d.tunisia_status}</td>
      <td>${d.alternatives.join(", ")}</td>
    </tr>
  `).join("");
}

function renderDesign() {
  const tbody = document.querySelector("#designTable tbody");
  tbody.innerHTML = designCandidates.map(d => `
    <tr>
      <td>${d.target}</td>
      <td><b>${d.candidate}</b><br><span class="pill red">${d.module_type}</span></td>
      <td>${d.predicted_binding}</td>
      <td>${d.docking_score}</td>
      <td>${d.evidence_level}</td>
    </tr>
  `).join("");
}

async function init() {
  patients = await loadJSON("demo/tunisian_patients_demo.json");
  predictions = await loadJSON("demo/tunisian_predictions.json");
  availability = await loadJSON("demo/drug_availability.json");
  designCandidates = await loadJSON("demo/drug_design_candidates.json");

  const select = document.getElementById("patientSelect");
  select.innerHTML = patients.map(p =>
    `<option value="${p.patient_id}">${p.patient_id} — ${p.histology}, ${p.stage}, EGFR: ${p.egfr}</option>`
  ).join("");

  select.addEventListener("change", () => {
    const patient = patients.find(p => p.patient_id === select.value);
    renderPatient(patient);
  });

  renderPatient(patients[0]);
  renderAvailability();
  renderDesign();
}

init().catch(error => {
  document.body.innerHTML = "<h1 style='color:white;padding:40px'>Failed to load Tunisia module data</h1><pre style='color:red;padding:40px'>" + error + "</pre>";
});
</script>
</body>
</html>
"""

page_path = os.path.join(FRONTEND_PATH, "tunisia_multimodal.html")
with open(page_path, "w", encoding="utf-8") as f:
    f.write(tunisia_page)

print("Created:", page_path)

Created: /content/drive/MyDrive/pfe_digital_twin_app/frontend/tunisia_multimodal.html


In [ ]:
from datetime import datetime
import shutil, os

backup_path = INDEX_PATH + ".backup_" + datetime.now().strftime("%Y%m%d_%H%M%S")
shutil.copy2(INDEX_PATH, backup_path)
print("Backup created:", backup_path)

with open(INDEX_PATH, "r", encoding="utf-8", errors="ignore") as f:
    html = f.read()

button_html = """
<div style="padding:16px; text-align:center; background:#0f172a;">
  <a href="tunisia_multimodal.html"
     style="display:inline-block; padding:12px 18px; border-radius:12px; background:#2563eb; color:white; text-decoration:none; font-weight:bold; font-family:Arial, sans-serif;">
    Open Tunisian Multimodal Precision Medicine Module
  </a>
</div>
"""

if "tunisia_multimodal.html" not in html:
    if "<body" in html.lower():
        body_match = re.search(r"<body[^>]*>", html, flags=re.IGNORECASE)
        insert_pos = body_match.end()
        html = html[:insert_pos] + button_html + html[insert_pos:]
    else:
        html = button_html + html

    with open(INDEX_PATH, "w", encoding="utf-8") as f:
        f.write(html)

    print("Button added to index.html")
else:
    print("Button already exists. No change made.")

Backup created: /content/drive/MyDrive/pfe_digital_twin_app/frontend/index.html.backup_20260514_124933
Button added to index.html


In [ ]:
%cd /content/drive/MyDrive/pfe_digital_twin_app/frontend
!python -m http.server 8080

/content/drive/MyDrive/pfe_digital_twin_app/frontend
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/lib/python3.12/http/server.py", line 1329, in <module>
    test(
  File "/usr/lib/python3.12/http/server.py", line 1276, in test
    with ServerClass(addr, HandlerClass) as httpd:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socketserver.py", line 457, in __init__
    self.server_bind()
  File "/usr/lib/python3.12/http/server.py", line 1323, in server_bind
    return super().server_bind()
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/http/server.py", line 140, in server_bind
    socketserver.TCPServer.server_bind(self)
  File "/usr/lib/python3.12/socketserver.py", line 478, in server_bind
    self.socket.bind(self.server_address)
OSError: [Errno 98] Address already in use


In [ ]:
%cd /content/drive/MyDrive/pfe_digital_twin_app/frontend
!python -m http.server 8081

/content/drive/MyDrive/pfe_digital_twin_app/frontend
Serving HTTP on 0.0.0.0 port 8081 (http://0.0.0.0:8081/) ...

Keyboard interrupt received, exiting.
^C


In [ ]:
import os, subprocess, time
from google.colab import output

FRONTEND_PATH = "/content/drive/MyDrive/pfe_digital_twin_app/frontend"

os.chdir(FRONTEND_PATH)

# Start server in background
server = subprocess.Popen(["python3", "-m", "http.server", "8081"])

time.sleep(2)

print("Server started in background on port 8081")
output.serve_kernel_port_as_window(8081)

Server started in background on port 8081
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [ ]:
from google.colab import output
output.serve_kernel_port_as_window(8081, path="/tunisia_multimodal.html")

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [ ]:
%cd /content/drive/MyDrive/pfe_digital_twin_app/frontend
!python -m http.server 8082

/content/drive/MyDrive/pfe_digital_twin_app/frontend
Serving HTTP on 0.0.0.0 port 8082 (http://0.0.0.0:8082/) ...

Keyboard interrupt received, exiting.


In [ ]:
from google.colab import drive
import os, shutil
from datetime import datetime

drive.mount('/content/drive')

FRONTEND_PATH = "/content/drive/MyDrive/pfe_digital_twin_app/frontend"

if not os.path.exists(FRONTEND_PATH):
    raise Exception("Frontend path not found")

print("Working in:", FRONTEND_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working in: /content/drive/MyDrive/pfe_digital_twin_app/frontend


In [ ]:
import os

hospital_page = r"""
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>Hospital Anapath Digital Twin System</title>
  <style>
    body {
      margin: 0;
      font-family: Arial, sans-serif;
      background: #f3f6fb;
      color: #111827;
    }

    header {
      background: #0f172a;
      color: white;
      padding: 26px 36px;
    }

    header h1 {
      margin: 0;
      font-size: 30px;
    }

    header p {
      color: #cbd5e1;
      margin-bottom: 0;
    }

    .container {
      padding: 28px 36px;
      max-width: 1250px;
      margin: auto;
    }

    .alert {
      background: #fff7ed;
      border-left: 5px solid #f97316;
      padding: 16px;
      border-radius: 12px;
      margin-bottom: 22px;
      color: #7c2d12;
    }

    .grid {
      display: grid;
      grid-template-columns: repeat(2, 1fr);
      gap: 18px;
      margin-bottom: 22px;
    }

    .grid3 {
      display: grid;
      grid-template-columns: repeat(3, 1fr);
      gap: 18px;
      margin-bottom: 22px;
    }

    .card {
      background: white;
      border-radius: 18px;
      padding: 20px;
      box-shadow: 0 6px 18px rgba(15,23,42,0.08);
      border: 1px solid #e5e7eb;
    }

    h2 {
      margin-top: 0;
      color: #0f172a;
      font-size: 21px;
    }

    label {
      font-size: 13px;
      color: #475569;
      font-weight: bold;
      display: block;
      margin-top: 12px;
      margin-bottom: 5px;
    }

    input, select, textarea {
      width: 100%;
      box-sizing: border-box;
      padding: 10px;
      border-radius: 10px;
      border: 1px solid #cbd5e1;
      font-size: 14px;
      background: white;
    }

    textarea {
      min-height: 85px;
      resize: vertical;
    }

    button {
      border: none;
      padding: 12px 16px;
      border-radius: 12px;
      font-weight: bold;
      cursor: pointer;
      margin-top: 12px;
      margin-right: 8px;
    }

    .primary {
      background: #2563eb;
      color: white;
    }

    .success {
      background: #16a34a;
      color: white;
    }

    .danger {
      background: #dc2626;
      color: white;
    }

    .secondary {
      background: #e2e8f0;
      color: #0f172a;
    }

    .pill {
      display: inline-block;
      padding: 6px 10px;
      border-radius: 999px;
      color: white;
      background: #2563eb;
      margin: 4px 4px 0 0;
      font-size: 13px;
    }

    .green {
      background: #15803d;
    }

    .orange {
      background: #c2410c;
    }

    .red {
      background: #b91c1c;
    }

    .purple {
      background: #7e22ce;
    }

    .value {
      font-size: 28px;
      color: #2563eb;
      font-weight: bold;
      margin: 6px 0 12px;
    }

    table {
      width: 100%;
      border-collapse: collapse;
      background: white;
      border-radius: 16px;
      overflow: hidden;
      box-shadow: 0 6px 18px rgba(15,23,42,0.08);
      margin-top: 16px;
    }

    th, td {
      padding: 12px;
      border-bottom: 1px solid #e5e7eb;
      text-align: left;
      vertical-align: top;
      font-size: 14px;
    }

    th {
      background: #0f172a;
      color: white;
    }

    .small {
      color: #64748b;
      font-size: 13px;
    }

    a {
      color: #38bdf8;
      text-decoration: none;
      font-weight: bold;
    }

    .toplink {
      color: #bfdbfe;
    }

    pre {
      white-space: pre-wrap;
      background: #020617;
      color: #d1fae5;
      padding: 15px;
      border-radius: 12px;
      overflow: auto;
      max-height: 300px;
    }

    @media(max-width: 900px) {
      .grid, .grid3 {
        grid-template-columns: 1fr;
      }
    }
  </style>
</head>

<body>
<header>
  <h1>Hospital Anapath Digital Twin System</h1>
  <p>Mini LIS / Syslab-like prototype for Tunisian lung cancer precision medicine</p>
  <p><a class="toplink" href="index.html">← Back to main dashboard</a> | <a class="toplink" href="tunisia_multimodal.html">Open Tunisian Multimodal Twin</a></p>
</header>

<div class="container">

  <div class="alert">
    <b>Prototype rule:</b> Do not enter real patient names, CIN, phone numbers or identifiable data.
    Use anonymized IDs only, for example <b>HM-TN-LC-0001</b>.
    This MVP is for research/demo until hospital authorization, ethics approval and data-protection validation are obtained.
  </div>

  <div class="grid">
    <div class="card">
      <h2>1. Patient Registry</h2>

      <label>Anonymous Patient ID</label>
      <input id="patient_id" placeholder="HM-TN-LC-0001">

      <label>Age</label>
      <input id="age" type="number" placeholder="61">

      <label>Sex</label>
      <select id="sex">
        <option>Male</option>
        <option>Female</option>
      </select>

      <label>Region</label>
      <input id="region" placeholder="Tunis / Monastir / Sousse">

      <label>Smoking Status</label>
      <select id="smoking">
        <option>Unknown</option>
        <option>Never smoker</option>
        <option>Former smoker</option>
        <option>Current smoker</option>
      </select>

      <label>ECOG Performance Status</label>
      <select id="ecog">
        <option value="0">0 - Fully active</option>
        <option value="1">1 - Restricted strenuous activity</option>
        <option value="2">2 - Ambulatory >50%</option>
        <option value="3">3 - Limited self-care</option>
        <option value="4">4 - Disabled</option>
      </select>
    </div>

    <div class="card">
      <h2>2. Anapath Sample / Biopsy</h2>

      <label>Sample ID</label>
      <input id="sample_id" placeholder="ANA-2026-0001">

      <label>Sample Type</label>
      <select id="sample_type">
        <option>Bronchial biopsy</option>
        <option>CT-guided biopsy</option>
        <option>Surgical specimen</option>
        <option>FFPE block</option>
        <option>Cytology</option>
      </select>

      <label>Anatomical Site</label>
      <input id="site" placeholder="Lung upper lobe / lymph node / pleura">

      <label>Reception Date</label>
      <input id="reception_date" type="date">

      <label>Specimen Quality</label>
      <select id="quality">
        <option>Good</option>
        <option>Limited material</option>
        <option>Necrotic</option>
        <option>Insufficient for molecular testing</option>
      </select>

      <label>Pathologist Comment</label>
      <textarea id="pathologist_comment" placeholder="Short anapath note..."></textarea>
    </div>
  </div>

  <div class="grid">
    <div class="card">
      <h2>3. Pathology Report</h2>

      <label>Cancer Type</label>
      <select id="cancer_type">
        <option>NSCLC</option>
        <option>SCLC</option>
        <option>Other</option>
      </select>

      <label>Histology</label>
      <select id="histology">
        <option>Adenocarcinoma</option>
        <option>Squamous cell carcinoma</option>
        <option>Large cell carcinoma</option>
        <option>NSCLC NOS</option>
        <option>Small cell carcinoma</option>
      </select>

      <label>Grade</label>
      <select id="grade">
        <option>Unknown</option>
        <option>Well differentiated</option>
        <option>Moderately differentiated</option>
        <option>Poorly differentiated</option>
      </select>

      <label>Stage</label>
      <select id="stage">
        <option>I</option>
        <option>II</option>
        <option>III</option>
        <option>IV</option>
      </select>

      <label>TNM</label>
      <input id="tnm" placeholder="T2N2M1">

      <label>Metastasis</label>
      <input id="metastasis" placeholder="None / bone / liver / brain">
    </div>

    <div class="card">
      <h2>4. Molecular / IHC Profile</h2>

      <label>EGFR</label>
      <select id="egfr">
        <option>Unknown</option>
        <option>Negative</option>
        <option>Exon 19 deletion</option>
        <option>L858R</option>
        <option>T790M</option>
        <option>Exon 20 insertion</option>
        <option>Other EGFR mutation</option>
      </select>

      <label>ALK</label>
      <select id="alk">
        <option>Unknown</option>
        <option>Negative</option>
        <option>Positive</option>
      </select>

      <label>ROS1</label>
      <select id="ros1">
        <option>Unknown</option>
        <option>Negative</option>
        <option>Positive</option>
      </select>

      <label>PD-L1</label>
      <select id="pdl1">
        <option>Unknown</option>
        <option>Negative</option>
        <option>Low</option>
        <option>High</option>
      </select>

      <label>IHC Markers</label>
      <input id="ihc" placeholder="TTF-1+, Napsin A+, p40-">

      <label>Molecular Test Status</label>
      <select id="molecular_status">
        <option>Not requested</option>
        <option>Requested</option>
        <option>Completed</option>
        <option>Insufficient material</option>
      </select>
    </div>
  </div>

  <div class="grid3">
    <div class="card">
      <h2>Digital Twin Prediction</h2>
      <div class="small">Predicted benefit score</div>
      <div class="value" id="benefitScore">--</div>
      <div id="predictionPills"></div>
    </div>

    <div class="card">
      <h2>Tunisian Treatment Strategy</h2>
      <p id="treatmentStrategy">Fill the form and click Predict.</p>
    </div>

    <div class="card">
      <h2>Drug Design Research</h2>
      <p id="drugDesign">Research module appears when rare/resistant mutation or unavailable treatment is detected.</p>
    </div>
  </div>

  <button class="primary" onclick="predictTwin()">Predict Digital Twin</button>
  <button class="success" onclick="saveCase()">Save Case Locally</button>
  <button class="secondary" onclick="exportJSON()">Export Current Case JSON</button>
  <button class="danger" onclick="clearCases()">Clear Local Cases</button>

  <h2>Saved Hospital Cases</h2>
  <table id="casesTable">
    <thead>
      <tr>
        <th>Patient ID</th>
        <th>Sample ID</th>
        <th>Histology</th>
        <th>Stage</th>
        <th>EGFR</th>
        <th>Prediction</th>
        <th>Treatment Strategy</th>
      </tr>
    </thead>
    <tbody></tbody>
  </table>

  <h2>Export Preview</h2>
  <pre id="jsonPreview">No case exported yet.</pre>

</div>

<script>
function val(id) {
  return document.getElementById(id).value;
}

function buildCase() {
  return {
    metadata: {
      system: "Hospital Anapath Digital Twin MVP",
      hospital_site: "Military Hospital - Anapath Service",
      data_mode: "Anonymized prototype",
      created_at: new Date().toISOString()
    },
    patient: {
      patient_id: val("patient_id"),
      age: Number(val("age")),
      sex: val("sex"),
      region: val("region"),
      smoking_status: val("smoking"),
      ecog: Number(val("ecog"))
    },
    sample: {
      sample_id: val("sample_id"),
      sample_type: val("sample_type"),
      anatomical_site: val("site"),
      reception_date: val("reception_date"),
      specimen_quality: val("quality"),
      pathologist_comment: val("pathologist_comment")
    },
    pathology: {
      cancer_type: val("cancer_type"),
      histology: val("histology"),
      grade: val("grade"),
      stage: val("stage"),
      tnm: val("tnm"),
      metastasis: val("metastasis")
    },
    molecular: {
      egfr: val("egfr"),
      alk: val("alk"),
      ros1: val("ros1"),
      pdl1: val("pdl1"),
      ihc_markers: val("ihc"),
      molecular_test_status: val("molecular_status")
    }
  };
}

function runPrediction(c) {
  let score = 50;
  let reasons = [];

  const egfr = c.molecular.egfr;
  const histology = c.pathology.histology;
  const stage = c.pathology.stage;
  const ecog = c.patient.ecog;
  const pdl1 = c.molecular.pdl1;
  const quality = c.sample.specimen_quality;

  if (egfr === "Exon 19 deletion" || egfr === "L858R") {
    score += 28;
    reasons.push("EGFR sensitizing mutation detected");
  }

  if (egfr === "T790M") {
    score += 20;
    reasons.push("EGFR resistance mutation suggests need for advanced targeted strategy");
  }

  if (egfr === "Exon 20 insertion") {
    score += 5;
    reasons.push("EGFR exon 20 insertion needs specific treatment strategy and access verification");
  }

  if (histology === "Adenocarcinoma") {
    score += 7;
    reasons.push("Adenocarcinoma supports molecular testing priority");
  }

  if (stage === "IV") {
    score -= 10;
    reasons.push("Metastatic stage increases progression risk");
  }

  if (ecog >= 2) {
    score -= 10;
    reasons.push("Reduced ECOG performance status increases treatment risk");
  }

  if (pdl1 === "High" && egfr === "Negative") {
    score += 12;
    reasons.push("High PD-L1 may support immunotherapy strategy if accessible");
  }

  if (quality === "Insufficient for molecular testing") {
    score -= 15;
    reasons.push("Insufficient material limits precision medicine confidence");
  }

  score = Math.max(0, Math.min(100, score));

  let label = "Intermediate predicted benefit";
  if (score >= 75) label = "High predicted benefit";
  if (score < 55) label = "Low predicted benefit";

  let treatment = "";
  let design = "";

  if (egfr === "Exon 19 deletion" || egfr === "L858R") {
    treatment = "Best global option: Osimertinib. Tunisia layer: verify hospital pharmacy access. If unavailable, consider erlotinib/gefitinib/afatinib or platinum-based chemotherapy according to local protocol.";
    design = "Research only: EGFR inhibitor scaffold prioritization can be explored using PyMOL, AutoDock Vina, RDKit and ADMET filtering.";
  } else if (egfr === "T790M") {
    treatment = "Resistance profile: consider advanced EGFR-targeted strategy if accessible; request full resistance workup including MET amplification if possible.";
    design = "Research only: prioritize molecules targeting resistant EGFR conformations and possible combination with MET pathway candidates.";
  } else if (egfr === "Exon 20 insertion") {
    treatment = "Rare EGFR alteration: verify availability of specific targeted therapy; otherwise use local systemic protocol and consider research referral.";
    design = "Research only: exon 20 insertion requires candidate molecule docking and literature-supported scaffold screening.";
  } else if (egfr === "Negative" && pdl1 === "High") {
    treatment = "EGFR negative with high PD-L1: immunotherapy + chemotherapy may be globally indicated; Tunisia access must be verified. If unavailable, use chemotherapy-based local protocol.";
    design = "Research only: no EGFR-targeted design priority; explore pathway-based candidates only after expanded molecular profiling.";
  } else {
    treatment = "Incomplete molecular profile: request EGFR, ALK, ROS1, PD-L1 testing if possible. Use pathology and stage to guide standard local protocol.";
    design = "Research only: insufficient molecular data for rational docking. Complete molecular profiling first.";
  }

  return {
    score,
    label,
    reasons,
    treatment,
    design
  };
}

function predictTwin() {
  const c = buildCase();
  const pred = runPrediction(c);
  c.prediction = pred;

  document.getElementById("benefitScore").textContent = pred.score + "%";
  document.getElementById("predictionPills").innerHTML =
    `<span class="pill green">${pred.label}</span>` +
    pred.reasons.map(r => `<span class="pill">${r}</span>`).join("");

  document.getElementById("treatmentStrategy").textContent = pred.treatment;
  document.getElementById("drugDesign").textContent = pred.design;

  document.getElementById("jsonPreview").textContent = JSON.stringify(c, null, 2);
  return c;
}

function getCases() {
  return JSON.parse(localStorage.getItem("hospital_anapath_cases") || "[]");
}

function setCases(cases) {
  localStorage.setItem("hospital_anapath_cases", JSON.stringify(cases));
}

function saveCase() {
  const c = predictTwin();

  if (!c.patient.patient_id || !c.sample.sample_id) {
    alert("Please enter at least Anonymous Patient ID and Sample ID.");
    return;
  }

  const cases = getCases();
  cases.push(c);
  setCases(cases);
  renderCases();
  alert("Case saved locally in browser storage.");
}

function renderCases() {
  const cases = getCases();
  const tbody = document.querySelector("#casesTable tbody");

  tbody.innerHTML = cases.map(c => `
    <tr>
      <td>${c.patient.patient_id}</td>
      <td>${c.sample.sample_id}</td>
      <td>${c.pathology.histology}</td>
      <td>${c.pathology.stage}</td>
      <td>${c.molecular.egfr}</td>
      <td>${c.prediction ? c.prediction.score + "% - " + c.prediction.label : "Not predicted"}</td>
      <td>${c.prediction ? c.prediction.treatment : ""}</td>
    </tr>
  `).join("");
}

function exportJSON() {
  const c = predictTwin();
  const dataStr = "data:text/json;charset=utf-8," + encodeURIComponent(JSON.stringify(c, null, 2));
  const a = document.createElement("a");
  a.setAttribute("href", dataStr);
  a.setAttribute("download", (c.patient.patient_id || "hospital_case") + ".json");
  document.body.appendChild(a);
  a.click();
  a.remove();
}

function clearCases() {
  if (confirm("Clear all locally saved cases?")) {
    localStorage.removeItem("hospital_anapath_cases");
    renderCases();
  }
}

renderCases();
</script>
</body>
</html>
"""

page_path = os.path.join(FRONTEND_PATH, "hospital_anapath_system.html")

with open(page_path, "w", encoding="utf-8") as f:
    f.write(hospital_page)

print("Created:", page_path)

Created: /content/drive/MyDrive/pfe_digital_twin_app/frontend/hospital_anapath_system.html


In [ ]:
import re, shutil, os
from datetime import datetime

INDEX_PATH = os.path.join(FRONTEND_PATH, "index.html")

backup_path = INDEX_PATH + ".backup_" + datetime.now().strftime("%Y%m%d_%H%M%S")
shutil.copy2(INDEX_PATH, backup_path)
print("Backup created:", backup_path)

with open(INDEX_PATH, "r", encoding="utf-8", errors="ignore") as f:
    html = f.read()

button_html = """
<div style="padding:16px; text-align:center; background:#111827;">
  <a href="hospital_anapath_system.html"
     style="display:inline-block; padding:12px 18px; border-radius:12px; background:#16a34a; color:white; text-decoration:none; font-weight:bold; font-family:Arial, sans-serif; margin-right:10px;">
    Open Hospital Anapath System
  </a>
  <a href="tunisia_multimodal.html"
     style="display:inline-block; padding:12px 18px; border-radius:12px; background:#2563eb; color:white; text-decoration:none; font-weight:bold; font-family:Arial, sans-serif;">
    Open Tunisian Multimodal Twin
  </a>
</div>
"""

if "hospital_anapath_system.html" not in html:
    body_match = re.search(r"<body[^>]*>", html, flags=re.IGNORECASE)
    if body_match:
        insert_pos = body_match.end()
        html = html[:insert_pos] + button_html + html[insert_pos:]
    else:
        html = button_html + html

    with open(INDEX_PATH, "w", encoding="utf-8") as f:
        f.write(html)

    print("Hospital system button added.")
else:
    print("Button already exists.")

Backup created: /content/drive/MyDrive/pfe_digital_twin_app/frontend/index.html.backup_20260514_132711
Hospital system button added.


In [ ]:
import re, shutil, os
from datetime import datetime

INDEX_PATH = os.path.join(FRONTEND_PATH, "index.html")

backup_path = INDEX_PATH + ".backup_" + datetime.now().strftime("%Y%m%d_%H%M%S")
shutil.copy2(INDEX_PATH, backup_path)
print("Backup created:", backup_path)

with open(INDEX_PATH, "r", encoding="utf-8", errors="ignore") as f:
    html = f.read()

button_html = """
<div style="padding:16px; text-align:center; background:#111827;">
  <a href="hospital_anapath_system.html"
     style="display:inline-block; padding:12px 18px; border-radius:12px; background:#16a34a; color:white; text-decoration:none; font-weight:bold; font-family:Arial, sans-serif; margin-right:10px;">
    Open Hospital Anapath System
  </a>
  <a href="tunisia_multimodal.html"
     style="display:inline-block; padding:12px 18px; border-radius:12px; background:#2563eb; color:white; text-decoration:none; font-weight:bold; font-family:Arial, sans-serif;">
    Open Tunisian Multimodal Twin
  </a>
</div>
"""

if "hospital_anapath_system.html" not in html:
    body_match = re.search(r"<body[^>]*>", html, flags=re.IGNORECASE)
    if body_match:
        insert_pos = body_match.end()
        html = html[:insert_pos] + button_html + html[insert_pos:]
    else:
        html = button_html + html

    with open(INDEX_PATH, "w", encoding="utf-8") as f:
        f.write(html)

    print("Hospital system button added.")
else:
    print("Button already exists.")

Backup created: /content/drive/MyDrive/pfe_digital_twin_app/frontend/index.html.backup_20260514_133212
Button already exists.


In [ ]:
import re, shutil, os
from datetime import datetime

INDEX_PATH = os.path.join(FRONTEND_PATH, "index.html")

backup_path = INDEX_PATH + ".backup_" + datetime.now().strftime("%Y%m%d_%H%M%S")
shutil.copy2(INDEX_PATH, backup_path)
print("Backup created:", backup_path)

with open(INDEX_PATH, "r", encoding="utf-8", errors="ignore") as f:
    html = f.read()

button_html = """
<div style="padding:16px; text-align:center; background:#111827;">
  <a href="hospital_anapath_system.html"
     style="display:inline-block; padding:12px 18px; border-radius:12px; background:#16a34a; color:white; text-decoration:none; font-weight:bold; font-family:Arial, sans-serif; margin-right:10px;">
    Open Hospital Anapath System
  </a>
  <a href="tunisia_multimodal.html"
     style="display:inline-block; padding:12px 18px; border-radius:12px; background:#2563eb; color:white; text-decoration:none; font-weight:bold; font-family:Arial, sans-serif;">
    Open Tunisian Multimodal Twin
  </a>
</div>
"""

if "hospital_anapath_system.html" not in html:
    body_match = re.search(r"<body[^>]*>", html, flags=re.IGNORECASE)
    if body_match:
        insert_pos = body_match.end()
        html = html[:insert_pos] + button_html + html[insert_pos:]
    else:
        html = button_html + html

    with open(INDEX_PATH, "w", encoding="utf-8") as f:
        f.write(html)

    print("Hospital system button added.")
else:
    print("Button already exists.")

Backup created: /content/drive/MyDrive/pfe_digital_twin_app/frontend/index.html.backup_20260514_133237
Button already exists.


In [ ]:
from pathlib import Path

app_code = r'''
from fastapi import FastAPI, Body, HTTPException
from fastapi.responses import JSONResponse
from fastapi.staticfiles import StaticFiles
from pathlib import Path
from datetime import datetime, timezone
from typing import Dict, Any
import sqlite3
import json

PROJECT_DIR = Path(__file__).resolve().parent
DB_PATH = PROJECT_DIR / "data" / "hospital_twin.db"
FRONTEND_DIR = PROJECT_DIR / "frontend"

app = FastAPI(
    title="Hospital Anapath Digital Twin System",
    description="Dynamic anapath/LIS-like system for Tunisian lung cancer precision medicine prototype",
    version="0.1.0"
)

def now():
    return datetime.now(timezone.utc).isoformat()

def get_conn():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

def init_db():
    DB_PATH.parent.mkdir(parents=True, exist_ok=True)

    with get_conn() as conn:
        conn.execute("""
        CREATE TABLE IF NOT EXISTS cases (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            patient_id TEXT UNIQUE NOT NULL,
            payload TEXT NOT NULL,
            prediction TEXT NOT NULL,
            created_at TEXT NOT NULL,
            updated_at TEXT NOT NULL
        )
        """)

        conn.execute("""
        CREATE TABLE IF NOT EXISTS audit_log (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            action TEXT NOT NULL,
            patient_id TEXT,
            details TEXT,
            created_at TEXT NOT NULL
        )
        """)

        conn.commit()

def audit(action: str, patient_id: str = "", details: Dict[str, Any] | None = None):
    with get_conn() as conn:
        conn.execute(
            "INSERT INTO audit_log(action, patient_id, details, created_at) VALUES (?, ?, ?, ?)",
            (action, patient_id, json.dumps(details or {}), now())
        )
        conn.commit()

def get_nested(payload, group, field, default="Unknown"):
    return payload.get(group, {}).get(field, default)

def predict_case(payload: Dict[str, Any]) -> Dict[str, Any]:
    patient = payload.get("patient", {})
    sample = payload.get("sample", {})
    pathology = payload.get("pathology", {})
    molecular = payload.get("molecular", {})

    age = patient.get("age")
    ecog = int(patient.get("ecog") or 0)

    histology = pathology.get("histology", "Unknown")
    stage = pathology.get("stage", "Unknown")
    grade = pathology.get("grade", "Unknown")

    egfr = molecular.get("egfr", "Unknown")
    alk = molecular.get("alk", "Unknown")
    ros1 = molecular.get("ros1", "Unknown")
    pdl1 = molecular.get("pdl1", "Unknown")
    molecular_status = molecular.get("molecular_test_status", "Unknown")

    specimen_quality = sample.get("specimen_quality", "Unknown")

    score = 50
    resistance_risk = 35
    confidence = 70
    reasons = []
    missing_data = []

    if egfr in ["Exon 19 deletion", "L858R"]:
        score += 30
        resistance_risk += 5
        reasons.append("EGFR sensitizing mutation detected")
    elif egfr == "T790M":
        score += 18
        resistance_risk += 30
        reasons.append("EGFR resistance mutation detected")
    elif egfr == "Exon 20 insertion":
        score += 8
        resistance_risk += 20
        reasons.append("EGFR exon 20 insertion requires specific strategy")
    elif egfr == "Negative":
        reasons.append("EGFR negative profile")
    else:
        confidence -= 15
        missing_data.append("EGFR mutation result")

    if histology == "Adenocarcinoma":
        score += 7
        reasons.append("Adenocarcinoma supports molecular profiling priority")

    if stage == "IV":
        score -= 10
        resistance_risk += 15
        reasons.append("Metastatic disease increases progression risk")
    elif stage == "III":
        score -= 3
        resistance_risk += 5

    if ecog >= 2:
        score -= 10
        confidence -= 5
        reasons.append("Reduced ECOG performance status increases treatment risk")

    if pdl1 == "High" and egfr == "Negative":
        score += 12
        reasons.append("High PD-L1 may support immunotherapy strategy if accessible")
    elif pdl1 == "Unknown":
        confidence -= 5
        missing_data.append("PD-L1 result")

    if alk == "Unknown":
        missing_data.append("ALK status")
        confidence -= 3

    if ros1 == "Unknown":
        missing_data.append("ROS1 status")
        confidence -= 3

    if specimen_quality == "Insufficient for molecular testing":
        score -= 15
        confidence -= 20
        reasons.append("Insufficient sample limits precision-medicine confidence")

    if molecular_status in ["Not requested", "Insufficient material"]:
        confidence -= 15
        missing_data.append("Complete molecular testing")

    score = max(0, min(100, score))
    resistance_risk = max(0, min(100, resistance_risk))
    confidence = max(0, min(100, confidence))

    if score >= 75:
        predicted_response = "High predicted benefit"
    elif score >= 55:
        predicted_response = "Intermediate predicted benefit"
    else:
        predicted_response = "Low predicted benefit"

    if egfr in ["Exon 19 deletion", "L858R"]:
        recommended = "Osimertinib"
        tunisia_strategy = (
            "Best global option: Osimertinib. Tunisia layer: verify hospital pharmacy access. "
            "If unavailable, consider erlotinib/gefitinib/afatinib or platinum-based chemotherapy according to local protocol."
        )
        drug_design = (
            "Research only: EGFR inhibitor scaffold prioritization can be explored using PyMOL, AutoDock Vina, RDKit and ADMET filtering."
        )
    elif egfr == "T790M":
        recommended = "Advanced EGFR resistance strategy"
        tunisia_strategy = (
            "Resistance profile: verify access to appropriate EGFR-targeted therapy; request resistance workup including MET amplification if possible."
        )
        drug_design = (
            "Research only: prioritize molecules targeting resistant EGFR conformations and possible EGFR/MET combination strategies."
        )
    elif egfr == "Exon 20 insertion":
        recommended = "Specific EGFR exon 20 strategy / local systemic therapy"
        tunisia_strategy = (
            "Rare EGFR alteration: verify targeted therapy availability. If unavailable, use local systemic protocol and consider research referral."
        )
        drug_design = (
            "Research only: exon 20 insertion should trigger docking/scaffold screening for candidate molecules."
        )
    elif egfr == "Negative" and pdl1 == "High":
        recommended = "Immunotherapy + chemotherapy if accessible"
        tunisia_strategy = (
            "EGFR negative with high PD-L1: immunotherapy + chemotherapy may be globally indicated. "
            "Tunisia access must be verified; if unavailable, use chemotherapy-based local protocol."
        )
        drug_design = (
            "Research only: no EGFR-targeted design priority; expand molecular profiling before rational docking."
        )
    else:
        recommended = "Complete molecular testing first"
        tunisia_strategy = (
            "Incomplete profile: request EGFR, ALK, ROS1 and PD-L1 testing if possible. "
            "Use pathology, stage and local protocols while waiting for molecular results."
        )
        drug_design = (
            "Research only: insufficient molecular data for rational drug design. Complete profiling first."
        )

    return {
        "predicted_efficiency": score,
        "predicted_response": predicted_response,
        "resistance_risk": resistance_risk,
        "confidence": confidence,
        "recommended_treatment": recommended,
        "tunisia_strategy": tunisia_strategy,
        "drug_design_research": drug_design,
        "reasons": reasons,
        "missing_data": missing_data,
        "model_version": "rule_based_v0.1_replaceable_by_ml",
        "generated_at": now()
    }

def extract_patient_id(payload: Dict[str, Any]) -> str:
    patient_id = payload.get("patient", {}).get("patient_id") or payload.get("patient_id")
    if not patient_id:
        raise HTTPException(status_code=400, detail="patient.patient_id is required")
    return str(patient_id).strip()

@app.on_event("startup")
def startup():
    init_db()

@app.get("/api/health")
def health():
    return {
        "status": "online",
        "system": "Hospital Anapath Digital Twin",
        "database": str(DB_PATH),
        "time": now()
    }

@app.post("/api/predict")
def predict(payload: Dict[str, Any] = Body(...)):
    prediction = predict_case(payload)
    return {
        "payload": payload,
        "prediction": prediction
    }

@app.post("/api/cases")
def save_case(payload: Dict[str, Any] = Body(...)):
    patient_id = extract_patient_id(payload)
    prediction = predict_case(payload)

    payload_json = json.dumps(payload, ensure_ascii=False)
    prediction_json = json.dumps(prediction, ensure_ascii=False)

    with get_conn() as conn:
        existing = conn.execute("SELECT id, created_at FROM cases WHERE patient_id = ?", (patient_id,)).fetchone()

        if existing:
            conn.execute(
                "UPDATE cases SET payload=?, prediction=?, updated_at=? WHERE patient_id=?",
                (payload_json, prediction_json, now(), patient_id)
            )
            case_id = existing["id"]
            action = "update_case"
        else:
            created = now()
            cur = conn.execute(
                "INSERT INTO cases(patient_id, payload, prediction, created_at, updated_at) VALUES (?, ?, ?, ?, ?)",
                (patient_id, payload_json, prediction_json, created, created)
            )
            case_id = cur.lastrowid
            action = "create_case"

        conn.commit()

    audit(action, patient_id, {"case_id": case_id})

    return {
        "message": "case_saved",
        "case_id": case_id,
        "patient_id": patient_id,
        "prediction": prediction
    }

@app.get("/api/cases")
def list_cases():
    with get_conn() as conn:
        rows = conn.execute("SELECT * FROM cases ORDER BY updated_at DESC").fetchall()

    cases = []
    for r in rows:
        cases.append({
            "id": r["id"],
            "patient_id": r["patient_id"],
            "payload": json.loads(r["payload"]),
            "prediction": json.loads(r["prediction"]),
            "created_at": r["created_at"],
            "updated_at": r["updated_at"]
        })

    return cases

@app.get("/api/cases/{case_id}")
def get_case(case_id: int):
    with get_conn() as conn:
        r = conn.execute("SELECT * FROM cases WHERE id=?", (case_id,)).fetchone()

    if not r:
        raise HTTPException(status_code=404, detail="Case not found")

    return {
        "id": r["id"],
        "patient_id": r["patient_id"],
        "payload": json.loads(r["payload"]),
        "prediction": json.loads(r["prediction"]),
        "created_at": r["created_at"],
        "updated_at": r["updated_at"]
    }

@app.delete("/api/cases/{case_id}")
def delete_case(case_id: int):
    with get_conn() as conn:
        r = conn.execute("SELECT patient_id FROM cases WHERE id=?", (case_id,)).fetchone()
        if not r:
            raise HTTPException(status_code=404, detail="Case not found")

        patient_id = r["patient_id"]
        conn.execute("DELETE FROM cases WHERE id=?", (case_id,))
        conn.commit()

    audit("delete_case", patient_id, {"case_id": case_id})
    return {"message": "case_deleted", "case_id": case_id}

@app.get("/api/stats")
def stats():
    cases = list_cases()
    total = len(cases)

    egfr_mutated = 0
    stage_iv = 0
    high_benefit = 0

    for c in cases:
        payload = c["payload"]
        pred = c["prediction"]

        egfr = payload.get("molecular", {}).get("egfr", "")
        stage = payload.get("pathology", {}).get("stage", "")

        if egfr in ["Exon 19 deletion", "L858R", "T790M", "Exon 20 insertion", "Other EGFR mutation"]:
            egfr_mutated += 1

        if stage == "IV":
            stage_iv += 1

        if pred.get("predicted_efficiency", 0) >= 75:
            high_benefit += 1

    return {
        "total_cases": total,
        "egfr_mutated_cases": egfr_mutated,
        "stage_iv_cases": stage_iv,
        "high_predicted_benefit_cases": high_benefit
    }

@app.get("/api/export/cases")
def export_cases():
    return {
        "exported_at": now(),
        "system": "Hospital Anapath Digital Twin",
        "cases": list_cases()
    }

@app.get("/api/export/fhir/{case_id}")
def export_fhir_diagnostic_report(case_id: int):
    case = get_case(case_id)
    payload = case["payload"]
    prediction = case["prediction"]

    patient = payload.get("patient", {})
    sample = payload.get("sample", {})
    pathology = payload.get("pathology", {})
    molecular = payload.get("molecular", {})

    report = {
        "resourceType": "DiagnosticReport",
        "id": f"diagnostic-report-{case_id}",
        "status": "final",
        "code": {
            "text": "Anatomopathology lung cancer precision medicine report"
        },
        "subject": {
            "identifier": {
                "value": patient.get("patient_id", case["patient_id"])
            }
        },
        "specimen": [
            {
                "identifier": {
                    "value": sample.get("sample_id", "")
                },
                "type": {
                    "text": sample.get("sample_type", "")
                }
            }
        ],
        "conclusion": (
            f"Histology: {pathology.get('histology', 'Unknown')}. "
            f"Stage: {pathology.get('stage', 'Unknown')}. "
            f"EGFR: {molecular.get('egfr', 'Unknown')}. "
            f"Predicted response: {prediction.get('predicted_response')}. "
            f"Recommended strategy: {prediction.get('tunisia_strategy')}"
        ),
        "presentedForm": [
            {
                "contentType": "application/json",
                "title": "Digital Twin Prediction JSON",
                "data": prediction
            }
        ]
    }

    return report

@app.post("/api/seed-demo")
def seed_demo():
    demo_cases = [
        {
            "patient": {
                "patient_id": "HM-TN-LC-0001",
                "age": 61,
                "sex": "Male",
                "region": "Tunis",
                "smoking_status": "Former smoker",
                "ecog": 1
            },
            "sample": {
                "sample_id": "ANA-2026-0001",
                "sample_type": "Bronchial biopsy",
                "anatomical_site": "Lung upper lobe",
                "reception_date": "2026-05-14",
                "specimen_quality": "Good",
                "pathologist_comment": "Tumor material adequate for molecular testing."
            },
            "pathology": {
                "cancer_type": "NSCLC",
                "histology": "Adenocarcinoma",
                "grade": "Moderately differentiated",
                "stage": "IV",
                "tnm": "T2N2M1",
                "metastasis": "Bone"
            },
            "molecular": {
                "egfr": "Exon 19 deletion",
                "alk": "Unknown",
                "ros1": "Unknown",
                "pdl1": "Unknown",
                "ihc_markers": "TTF-1+, Napsin A+",
                "molecular_test_status": "Completed"
            }
        },
        {
            "patient": {
                "patient_id": "HM-TN-LC-0002",
                "age": 68,
                "sex": "Male",
                "region": "Sousse",
                "smoking_status": "Current smoker",
                "ecog": 2
            },
            "sample": {
                "sample_id": "ANA-2026-0002",
                "sample_type": "CT-guided biopsy",
                "anatomical_site": "Lung mass",
                "reception_date": "2026-05-14",
                "specimen_quality": "Limited material",
                "pathologist_comment": "Small biopsy."
            },
            "pathology": {
                "cancer_type": "NSCLC",
                "histology": "Squamous cell carcinoma",
                "grade": "Poorly differentiated",
                "stage": "IV",
                "tnm": "T4N2M1",
                "metastasis": "Liver"
            },
            "molecular": {
                "egfr": "Negative",
                "alk": "Unknown",
                "ros1": "Unknown",
                "pdl1": "High",
                "ihc_markers": "p40+, TTF-1-",
                "molecular_test_status": "Completed"
            }
        }
    ]

    saved = []
    for c in demo_cases:
        result = save_case(c)
        saved.append(result)

    return {
        "message": "demo_cases_seeded",
        "saved": saved
    }

# Serve frontend last
app.mount("/", StaticFiles(directory=str(FRONTEND_DIR), html=True), name="frontend")
'''

PROJECT_DIR = Path("/content/drive/MyDrive/pfe_digital_twin_app")
app_path = PROJECT_DIR / "app.py"
app_path.write_text(app_code, encoding="utf-8")

print("Backend created:", app_path)


Backend created: /content/drive/MyDrive/pfe_digital_twin_app/app.py


In [ ]:
from pathlib import Path

FRONTEND_DIR = Path("/content/drive/MyDrive/pfe_digital_twin_app/frontend")
FRONTEND_DIR.mkdir(parents=True, exist_ok=True)

frontend_code = r'''
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>Hospital Anapath Digital Twin System</title>
  <style>
    body {
      margin: 0;
      font-family: Arial, sans-serif;
      background: #f3f6fb;
      color: #0f172a;
    }

    header {
      background: #0f172a;
      color: white;
      padding: 24px 34px;
    }

    header h1 {
      margin: 0;
      font-size: 30px;
    }

    header p {
      color: #cbd5e1;
      margin-bottom: 0;
    }

    .container {
      max-width: 1300px;
      margin: auto;
      padding: 26px 34px;
    }

    .alert {
      background: #fff7ed;
      border-left: 5px solid #f97316;
      padding: 14px;
      border-radius: 12px;
      margin-bottom: 20px;
      color: #7c2d12;
    }

    .grid {
      display: grid;
      grid-template-columns: repeat(2, 1fr);
      gap: 18px;
      margin-bottom: 18px;
    }

    .grid3 {
      display: grid;
      grid-template-columns: repeat(4, 1fr);
      gap: 18px;
      margin-bottom: 18px;
    }

    .card {
      background: white;
      border-radius: 18px;
      padding: 20px;
      box-shadow: 0 6px 18px rgba(15,23,42,0.08);
      border: 1px solid #e5e7eb;
    }

    h2 {
      margin-top: 0;
      color: #0f172a;
      font-size: 20px;
    }

    label {
      font-size: 13px;
      color: #475569;
      font-weight: bold;
      display: block;
      margin-top: 11px;
      margin-bottom: 5px;
    }

    input, select, textarea {
      width: 100%;
      box-sizing: border-box;
      padding: 10px;
      border-radius: 10px;
      border: 1px solid #cbd5e1;
      font-size: 14px;
      background: white;
    }

    textarea {
      min-height: 80px;
      resize: vertical;
    }

    button {
      border: none;
      padding: 12px 16px;
      border-radius: 12px;
      font-weight: bold;
      cursor: pointer;
      margin: 5px 5px 5px 0;
    }

    .primary { background: #2563eb; color: white; }
    .success { background: #16a34a; color: white; }
    .danger { background: #dc2626; color: white; }
    .secondary { background: #e2e8f0; color: #0f172a; }
    .purple { background: #7e22ce; color: white; }

    .value {
      font-size: 30px;
      font-weight: bold;
      color: #2563eb;
    }

    .small {
      color: #64748b;
      font-size: 13px;
    }

    .pill {
      display: inline-block;
      padding: 6px 10px;
      border-radius: 999px;
      color: white;
      background: #2563eb;
      margin: 4px 4px 0 0;
      font-size: 13px;
    }

    .green { background: #15803d; }
    .orange { background: #c2410c; }
    .red { background: #b91c1c; }

    table {
      width: 100%;
      border-collapse: collapse;
      background: white;
      border-radius: 16px;
      overflow: hidden;
      box-shadow: 0 6px 18px rgba(15,23,42,0.08);
      margin-top: 16px;
    }

    th, td {
      padding: 12px;
      border-bottom: 1px solid #e5e7eb;
      text-align: left;
      vertical-align: top;
      font-size: 14px;
    }

    th {
      background: #0f172a;
      color: white;
    }

    pre {
      white-space: pre-wrap;
      background: #020617;
      color: #d1fae5;
      padding: 15px;
      border-radius: 12px;
      max-height: 350px;
      overflow: auto;
    }

    .bar-bg {
      background: #e2e8f0;
      height: 28px;
      border-radius: 20px;
      overflow: hidden;
      margin-top: 8px;
    }

    .bar {
      background: #2563eb;
      color: white;
      height: 28px;
      line-height: 28px;
      padding-left: 10px;
      font-weight: bold;
      width: 0%;
    }

    @media(max-width: 1000px) {
      .grid, .grid3 { grid-template-columns: 1fr; }
    }
  </style>
</head>

<body>
<header>
  <h1>Hospital Anapath Digital Twin System</h1>
  <p>Dynamic real-time anapath/LIS-like prototype for Tunisian lung cancer precision medicine</p>
</header>

<div class="container">

  <div class="alert">
    <b>Important:</b> Use anonymized IDs only. Do not enter real names, CIN, phone numbers, addresses, or identifiable hospital data in this prototype.
  </div>

  <div class="grid3">
    <div class="card">
      <div class="small">Total cases</div>
      <div class="value" id="statTotal">0</div>
    </div>
    <div class="card">
      <div class="small">EGFR mutated</div>
      <div class="value" id="statEgfr">0</div>
    </div>
    <div class="card">
      <div class="small">Stage IV</div>
      <div class="value" id="statStage">0</div>
    </div>
    <div class="card">
      <div class="small">High predicted benefit</div>
      <div class="value" id="statHigh">0</div>
    </div>
  </div>

  <div>
    <button class="primary" onclick="predictLive()">Run Prediction</button>
    <button class="success" onclick="saveCase()">Save / Update Case</button>
    <button class="purple" onclick="seedDemo()">Add Demo Cases</button>
    <button class="secondary" onclick="refreshCases()">Refresh Cases</button>
    <button class="secondary" onclick="openApiDocs()">Open API Docs</button>
  </div>

  <div class="grid">
    <div class="card">
      <h2>1. Patient Registry</h2>

      <label>Anonymous Patient ID</label>
      <input id="patient_id" placeholder="HM-TN-LC-0001">

      <label>Age</label>
      <input id="age" type="number" placeholder="61">

      <label>Sex</label>
      <select id="sex">
        <option>Male</option>
        <option>Female</option>
      </select>

      <label>Region</label>
      <input id="region" placeholder="Tunis / Monastir / Sousse">

      <label>Smoking Status</label>
      <select id="smoking_status">
        <option>Unknown</option>
        <option>Never smoker</option>
        <option>Former smoker</option>
        <option>Current smoker</option>
      </select>

      <label>ECOG</label>
      <select id="ecog">
        <option value="0">0</option>
        <option value="1">1</option>
        <option value="2">2</option>
        <option value="3">3</option>
        <option value="4">4</option>
      </select>
    </div>

    <div class="card">
      <h2>2. Anapath Sample</h2>

      <label>Sample ID</label>
      <input id="sample_id" placeholder="ANA-2026-0001">

      <label>Sample Type</label>
      <select id="sample_type">
        <option>Bronchial biopsy</option>
        <option>CT-guided biopsy</option>
        <option>Surgical specimen</option>
        <option>FFPE block</option>
        <option>Cytology</option>
      </select>

      <label>Anatomical Site</label>
      <input id="anatomical_site" placeholder="Lung upper lobe / lymph node / pleura">

      <label>Reception Date</label>
      <input id="reception_date" type="date">

      <label>Specimen Quality</label>
      <select id="specimen_quality">
        <option>Good</option>
        <option>Limited material</option>
        <option>Necrotic</option>
        <option>Insufficient for molecular testing</option>
      </select>

      <label>Pathologist Comment</label>
      <textarea id="pathologist_comment" placeholder="Short anapath note..."></textarea>
    </div>
  </div>

  <div class="grid">
    <div class="card">
      <h2>3. Pathology Report</h2>

      <label>Cancer Type</label>
      <select id="cancer_type">
        <option>NSCLC</option>
        <option>SCLC</option>
        <option>Other</option>
      </select>

      <label>Histology</label>
      <select id="histology">
        <option>Adenocarcinoma</option>
        <option>Squamous cell carcinoma</option>
        <option>Large cell carcinoma</option>
        <option>NSCLC NOS</option>
        <option>Small cell carcinoma</option>
      </select>

      <label>Grade</label>
      <select id="grade">
        <option>Unknown</option>
        <option>Well differentiated</option>
        <option>Moderately differentiated</option>
        <option>Poorly differentiated</option>
      </select>

      <label>Stage</label>
      <select id="stage">
        <option>I</option>
        <option>II</option>
        <option>III</option>
        <option>IV</option>
      </select>

      <label>TNM</label>
      <input id="tnm" placeholder="T2N2M1">

      <label>Metastasis</label>
      <input id="metastasis" placeholder="None / bone / liver / brain">
    </div>

    <div class="card">
      <h2>4. Molecular / IHC Profile</h2>

      <label>EGFR</label>
      <select id="egfr">
        <option>Unknown</option>
        <option>Negative</option>
        <option>Exon 19 deletion</option>
        <option>L858R</option>
        <option>T790M</option>
        <option>Exon 20 insertion</option>
        <option>Other EGFR mutation</option>
      </select>

      <label>ALK</label>
      <select id="alk">
        <option>Unknown</option>
        <option>Negative</option>
        <option>Positive</option>
      </select>

      <label>ROS1</label>
      <select id="ros1">
        <option>Unknown</option>
        <option>Negative</option>
        <option>Positive</option>
      </select>

      <label>PD-L1</label>
      <select id="pdl1">
        <option>Unknown</option>
        <option>Negative</option>
        <option>Low</option>
        <option>High</option>
      </select>

      <label>IHC Markers</label>
      <input id="ihc_markers" placeholder="TTF-1+, Napsin A+, p40-">

      <label>Molecular Test Status</label>
      <select id="molecular_test_status">
        <option>Not requested</option>
        <option>Requested</option>
        <option>Completed</option>
        <option>Insufficient material</option>
      </select>
    </div>
  </div>

  <div class="grid">
    <div class="card">
      <h2>Real-Time Digital Twin Prediction</h2>
      <div class="small">Predicted efficiency</div>
      <div class="bar-bg"><div class="bar" id="effBar">0%</div></div>
      <p><b>Response:</b> <span id="predResponse">--</span></p>
      <p><b>Resistance risk:</b> <span id="resRisk">--</span></p>
      <p><b>Confidence:</b> <span id="confidence">--</span></p>
      <div id="reasonPills"></div>
    </div>

    <div class="card">
      <h2>Treatment + Research Output</h2>
      <p><b>Recommended treatment:</b> <span id="recommended">--</span></p>
      <p><b>Tunisia strategy:</b></p>
      <p id="tnStrategy">--</p>
      <p><b>Drug design research:</b></p>
      <p id="drugDesign">--</p>
    </div>
  </div>

  <h2>Saved Dynamic Cases</h2>
  <table>
    <thead>
      <tr>
        <th>ID</th>
        <th>Patient</th>
        <th>Histology</th>
        <th>Stage</th>
        <th>EGFR</th>
        <th>Prediction</th>
        <th>Action</th>
      </tr>
    </thead>
    <tbody id="casesBody"></tbody>
  </table>

  <h2>Live JSON Output</h2>
  <pre id="jsonPreview">No prediction yet.</pre>

</div>

<script>
const API = "";

function value(id) {
  return document.getElementById(id).value;
}

function buildPayload() {
  return {
    patient: {
      patient_id: value("patient_id"),
      age: Number(value("age")),
      sex: value("sex"),
      region: value("region"),
      smoking_status: value("smoking_status"),
      ecog: Number(value("ecog"))
    },
    sample: {
      sample_id: value("sample_id"),
      sample_type: value("sample_type"),
      anatomical_site: value("anatomical_site"),
      reception_date: value("reception_date"),
      specimen_quality: value("specimen_quality"),
      pathologist_comment: value("pathologist_comment")
    },
    pathology: {
      cancer_type: value("cancer_type"),
      histology: value("histology"),
      grade: value("grade"),
      stage: value("stage"),
      tnm: value("tnm"),
      metastasis: value("metastasis")
    },
    molecular: {
      egfr: value("egfr"),
      alk: value("alk"),
      ros1: value("ros1"),
      pdl1: value("pdl1"),
      ihc_markers: value("ihc_markers"),
      molecular_test_status: value("molecular_test_status")
    }
  };
}

async function postJSON(url, data) {
  const res = await fetch(API + url, {
    method: "POST",
    headers: {"Content-Type": "application/json"},
    body: JSON.stringify(data)
  });

  if (!res.ok) {
    const msg = await res.text();
    throw new Error(msg);
  }

  return await res.json();
}

async function getJSON(url) {
  const res = await fetch(API + url);
  if (!res.ok) throw new Error(await res.text());
  return await res.json();
}

function renderPrediction(result) {
  const p = result.prediction;
  const eff = p.predicted_efficiency || 0;

  const bar = document.getElementById("effBar");
  bar.style.width = eff + "%";
  bar.textContent = eff + "%";

  document.getElementById("predResponse").textContent = p.predicted_response;
  document.getElementById("resRisk").textContent = p.resistance_risk + "%";
  document.getElementById("confidence").textContent = p.confidence + "%";
  document.getElementById("recommended").textContent = p.recommended_treatment;
  document.getElementById("tnStrategy").textContent = p.tunisia_strategy;
  document.getElementById("drugDesign").textContent = p.drug_design_research;

  const reasons = (p.reasons || []).map(x => `<span class="pill green">${x}</span>`).join("");
  const missing = (p.missing_data || []).map(x => `<span class="pill orange">Missing: ${x}</span>`).join("");
  document.getElementById("reasonPills").innerHTML = reasons + missing;

  document.getElementById("jsonPreview").textContent = JSON.stringify(result, null, 2);
}

async function predictLive() {
  const payload = buildPayload();
  const result = await postJSON("/api/predict", payload);
  renderPrediction(result);
}

async function saveCase() {
  const payload = buildPayload();

  if (!payload.patient.patient_id) {
    alert("Anonymous Patient ID is required.");
    return;
  }

  const result = await postJSON("/api/cases", payload);
  renderPrediction({payload, prediction: result.prediction});
  await refreshCases();
  alert("Case saved to SQLite database.");
}

async function refreshCases() {
  const cases = await getJSON("/api/cases");
  const body = document.getElementById("casesBody");

  body.innerHTML = cases.map(c => {
    const p = c.payload;
    const pred = c.prediction;

    return `
      <tr>
        <td>${c.id}</td>
        <td><b>${c.patient_id}</b></td>
        <td>${p.pathology?.histology || ""}</td>
        <td>${p.pathology?.stage || ""}</td>
        <td>${p.molecular?.egfr || ""}</td>
        <td>${pred.predicted_efficiency}% - ${pred.predicted_response}</td>
        <td>
          <button class="secondary" onclick="loadCase(${c.id})">Load</button>
          <button class="danger" onclick="deleteCase(${c.id})">Delete</button>
          <button class="purple" onclick="openFHIR(${c.id})">FHIR</button>
        </td>
      </tr>
    `;
  }).join("");

  await refreshStats();
}

async function refreshStats() {
  const s = await getJSON("/api/stats");
  document.getElementById("statTotal").textContent = s.total_cases;
  document.getElementById("statEgfr").textContent = s.egfr_mutated_cases;
  document.getElementById("statStage").textContent = s.stage_iv_cases;
  document.getElementById("statHigh").textContent = s.high_predicted_benefit_cases;
}

function setValue(id, v) {
  document.getElementById(id).value = v ?? "";
}

async function loadCase(id) {
  const c = await getJSON(`/api/cases/${id}`);
  const p = c.payload;

  setValue("patient_id", p.patient.patient_id);
  setValue("age", p.patient.age);
  setValue("sex", p.patient.sex);
  setValue("region", p.patient.region);
  setValue("smoking_status", p.patient.smoking_status);
  setValue("ecog", p.patient.ecog);

  setValue("sample_id", p.sample.sample_id);
  setValue("sample_type", p.sample.sample_type);
  setValue("anatomical_site", p.sample.anatomical_site);
  setValue("reception_date", p.sample.reception_date);
  setValue("specimen_quality", p.sample.specimen_quality);
  setValue("pathologist_comment", p.sample.pathologist_comment);

  setValue("cancer_type", p.pathology.cancer_type);
  setValue("histology", p.pathology.histology);
  setValue("grade", p.pathology.grade);
  setValue("stage", p.pathology.stage);
  setValue("tnm", p.pathology.tnm);
  setValue("metastasis", p.pathology.metastasis);

  setValue("egfr", p.molecular.egfr);
  setValue("alk", p.molecular.alk);
  setValue("ros1", p.molecular.ros1);
  setValue("pdl1", p.molecular.pdl1);
  setValue("ihc_markers", p.molecular.ihc_markers);
  setValue("molecular_test_status", p.molecular.molecular_test_status);

  renderPrediction({payload: p, prediction: c.prediction});
  window.scrollTo({top: 0, behavior: "smooth"});
}

async function deleteCase(id) {
  if (!confirm("Delete this case?")) return;

  await fetch(API + `/api/cases/${id}`, {method: "DELETE"});
  await refreshCases();
}

async function seedDemo() {
  await postJSON("/api/seed-demo", {});
  await refreshCases();
  alert("Demo cases added.");
}

function openFHIR(id) {
  window.open(`/api/export/fhir/${id}`, "_blank");
}

function openApiDocs() {
  window.open("/docs", "_blank");
}

let timer = null;
document.querySelectorAll("input, select, textarea").forEach(el => {
  el.addEventListener("change", () => {
    clearTimeout(timer);
    timer = setTimeout(() => {
      const patientId = value("patient_id");
      if (patientId) predictLive().catch(() => {});
    }, 400);
  });
});

refreshCases().catch(console.error);
</script>
</body>
</html>
'''

(FRONTEND_DIR / "index.html").write_text(frontend_code, encoding="utf-8")

print("Frontend created:", FRONTEND_DIR / "index.html")

Frontend created: /content/drive/MyDrive/pfe_digital_twin_app/frontend/index.html


In [ ]:
import subprocess, time, os
from google.colab import output

PROJECT_DIR = "/content/drive/MyDrive/pfe_digital_twin_app"

# Stop old servers if they exist
!pkill -f "uvicorn" || true
!pkill -f "app.py" || true

server = subprocess.Popen(
    ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=PROJECT_DIR
)

time.sleep(3)

print("Server is running on port 8000")
print("Displaying system within notebook...")
output.serve_kernel_port_as_iframe(8000, height=800)


^C
^C
Server is running on port 8000
Displaying system within notebook...


<IPython.core.display.Javascript object>

In [ ]:
from google.colab import output
# Use iframe instead of window to bypass Colab/browser restrictions
output.serve_kernel_port_as_iframe(8000, path="/docs", height=800)


<IPython.core.display.Javascript object>

In [ ]:
from google.colab import output
# Display the main interface (root path '/')
output.serve_kernel_port_as_iframe(8000, path="/", height=850)

<IPython.core.display.Javascript object>

In [ ]:
import subprocess, time, os

PROJECT_DIR = "/content/drive/MyDrive/pfe_digital_twin_app"
LOG_FILE = os.path.join(PROJECT_DIR, "api.log")

# Stop the existing server
!pkill -f "uvicorn" || true

# Start the server and redirect output to api.log
print(f"Starting server and logging to {LOG_FILE}...")
with open(LOG_FILE, "w") as log_file:
    server = subprocess.Popen(
        ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"],
        cwd=PROJECT_DIR,
        stdout=log_file,
        stderr=subprocess.STDOUT
    )

# Wait a moment for it to start and write some logs
time.sleep(3)

# Display the logs
print("\n--- API LOGS ---")
with open(LOG_FILE, "r") as log_file:
    print(log_file.read())
print("----------------")


^C
Starting server and logging to /content/drive/MyDrive/pfe_digital_twin_app/api.log...

--- API LOGS ---
INFO:     Started server process [47144]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
                        INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [46830]

----------------


In [ ]:
import os

LOG_FILE = "/content/drive/MyDrive/pfe_digital_twin_app/api.log"

if os.path.exists(LOG_FILE):
    print("--- Contenu de api.log ---")
    with open(LOG_FILE, "r") as f:
        print(f.read())
else:
    print("Le fichier api.log n'existe pas.")


--- Contenu de api.log ---
INFO:     Started server process [47144]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
                        INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [46830]



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
from pathlib import Path
from datetime import datetime

PROJECT_DIR = Path("/content/drive/MyDrive/pfe_hospital_anapath_real_system")
FRONTEND_DIR = PROJECT_DIR / "frontend"
DATA_DIR = PROJECT_DIR / "data"

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
FRONTEND_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

for file in ["app.py", "frontend/index.html"]:
    path = PROJECT_DIR / file
    if path.exists():
        backup = PROJECT_DIR / f"{file.replace('/', '_')}.backup_{timestamp}"
        shutil.copy2(path, backup)
        print("Backup:", backup)

print("Ready:", PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ready: /content/drive/MyDrive/pfe_hospital_anapath_real_system


In [ ]:
!pip -q install fastapi uvicorn

In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/pfe_hospital_anapath_real_system")
APP_PATH = PROJECT_DIR / "app.py"

app_code = r'''
from fastapi import FastAPI, Body, HTTPException, Depends, Header
from fastapi.staticfiles import StaticFiles
from pathlib import Path
from datetime import datetime, timezone
from typing import Dict, Any, Optional
import sqlite3
import json
import secrets
import hashlib
import os
import time

PROJECT_DIR = Path(__file__).resolve().parent
DB_PATH = PROJECT_DIR / "data" / "hospital_twin.db"
FRONTEND_DIR = PROJECT_DIR / "frontend"

app = FastAPI(
    title="Hospital Anapath Digital Twin System",
    description="Dynamic hospital-ready prototype for Tunisian lung cancer precision medicine",
    version="0.2.0"
)

# ----------------------------
# Database helpers
# ----------------------------

def now_iso():
    return datetime.now(timezone.utc).isoformat()

def get_conn():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

def hash_password(password: str, salt: Optional[str] = None):
    if salt is None:
        salt = secrets.token_hex(16)
    hashed = hashlib.pbkdf2_hmac(
        "sha256",
        password.encode("utf-8"),
        salt.encode("utf-8"),
        120000
    ).hex()
    return salt, hashed

def verify_password(password: str, salt: str, stored_hash: str):
    _, candidate_hash = hash_password(password, salt)
    return secrets.compare_digest(candidate_hash, stored_hash)

def init_db():
    DB_PATH.parent.mkdir(parents=True, exist_ok=True)

    with get_conn() as conn:
        conn.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE NOT NULL,
            full_name TEXT NOT NULL,
            role TEXT NOT NULL,
            salt TEXT NOT NULL,
            password_hash TEXT NOT NULL,
            is_active INTEGER NOT NULL DEFAULT 1,
            created_at TEXT NOT NULL
        )
        """)

        conn.execute("""
        CREATE TABLE IF NOT EXISTS sessions (
            token TEXT PRIMARY KEY,
            user_id INTEGER NOT NULL,
            created_at TEXT NOT NULL,
            expires_at INTEGER NOT NULL,
            FOREIGN KEY(user_id) REFERENCES users(id)
        )
        """)

        conn.execute("""
        CREATE TABLE IF NOT EXISTS cases (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            patient_id TEXT UNIQUE NOT NULL,
            payload TEXT NOT NULL,
            prediction TEXT NOT NULL,
            created_by TEXT,
            updated_by TEXT,
            created_at TEXT NOT NULL,
            updated_at TEXT NOT NULL
        )
        """)

        conn.execute("""
        CREATE TABLE IF NOT EXISTS audit_log (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            action TEXT NOT NULL,
            username TEXT,
            role TEXT,
            patient_id TEXT,
            details TEXT,
            created_at TEXT NOT NULL
        )
        """)

        conn.commit()

    seed_default_users()

def create_user(username: str, password: str, role: str, full_name: str):
    salt, password_hash = hash_password(password)

    with get_conn() as conn:
        existing = conn.execute(
            "SELECT id FROM users WHERE username=?",
            (username,)
        ).fetchone()

        if existing:
            return

        conn.execute(
            """
            INSERT INTO users(username, full_name, role, salt, password_hash, is_active, created_at)
            VALUES (?, ?, ?, ?, ?, 1, ?)
            """,
            (username, full_name, role, salt, password_hash, now_iso())
        )
        conn.commit()

def seed_default_users():
    create_user("admin", "admin123", "admin", "System Administrator")
    create_user("pathologist", "patho123", "pathologist", "Pathologist User")
    create_user("oncologist", "onco123", "oncologist", "Oncologist User")
    create_user("researcher", "research123", "researcher", "Research User")

def audit(action: str, user: Optional[Dict[str, Any]] = None, patient_id: str = "", details: Dict[str, Any] = None):
    details = details or {}

    username = user["username"] if user else ""
    role = user["role"] if user else ""

    with get_conn() as conn:
        conn.execute(
            """
            INSERT INTO audit_log(action, username, role, patient_id, details, created_at)
            VALUES (?, ?, ?, ?, ?, ?)
            """,
            (action, username, role, patient_id, json.dumps(details), now_iso())
        )
        conn.commit()

# ----------------------------
# Authentication
# ----------------------------

def get_current_user(authorization: Optional[str] = Header(default=None)):
    if not authorization:
        raise HTTPException(status_code=401, detail="Missing Authorization header")

    if not authorization.startswith("Bearer "):
        raise HTTPException(status_code=401, detail="Invalid Authorization header")

    token = authorization.replace("Bearer ", "").strip()

    with get_conn() as conn:
        session = conn.execute(
            "SELECT * FROM sessions WHERE token=?",
            (token,)
        ).fetchone()

        if not session:
            raise HTTPException(status_code=401, detail="Invalid session")

        if int(session["expires_at"]) < int(time.time()):
            conn.execute("DELETE FROM sessions WHERE token=?", (token,))
            conn.commit()
            raise HTTPException(status_code=401, detail="Session expired")

        user = conn.execute(
            "SELECT id, username, full_name, role, is_active FROM users WHERE id=?",
            (session["user_id"],)
        ).fetchone()

        if not user or int(user["is_active"]) != 1:
            raise HTTPException(status_code=401, detail="Inactive user")

    return dict(user)

def require_roles(*allowed_roles):
    def dependency(user: Dict[str, Any] = Depends(get_current_user)):
        if user["role"] not in allowed_roles:
            raise HTTPException(status_code=403, detail="Insufficient permissions")
        return user
    return dependency

# ----------------------------
# Prediction engine
# ----------------------------

def predict_case(payload: Dict[str, Any]) -> Dict[str, Any]:
    patient = payload.get("patient", {})
    sample = payload.get("sample", {})
    pathology = payload.get("pathology", {})
    molecular = payload.get("molecular", {})

    ecog = int(patient.get("ecog") or 0)

    histology = pathology.get("histology", "Unknown")
    stage = pathology.get("stage", "Unknown")
    egfr = molecular.get("egfr", "Unknown")
    alk = molecular.get("alk", "Unknown")
    ros1 = molecular.get("ros1", "Unknown")
    pdl1 = molecular.get("pdl1", "Unknown")
    molecular_status = molecular.get("molecular_test_status", "Unknown")
    specimen_quality = sample.get("specimen_quality", "Unknown")

    score = 50
    resistance_risk = 35
    confidence = 75
    reasons = []
    missing_data = []

    if egfr in ["Exon 19 deletion", "L858R"]:
        score += 30
        resistance_risk += 5
        reasons.append("EGFR sensitizing mutation detected")
    elif egfr == "T790M":
        score += 18
        resistance_risk += 30
        reasons.append("EGFR resistance mutation detected")
    elif egfr == "Exon 20 insertion":
        score += 8
        resistance_risk += 20
        reasons.append("EGFR exon 20 insertion requires specific strategy")
    elif egfr == "Negative":
        reasons.append("EGFR negative profile")
    else:
        confidence -= 15
        missing_data.append("EGFR mutation result")

    if histology == "Adenocarcinoma":
        score += 7
        reasons.append("Adenocarcinoma supports molecular profiling priority")

    if stage == "IV":
        score -= 10
        resistance_risk += 15
        reasons.append("Metastatic disease increases progression risk")
    elif stage == "III":
        score -= 3
        resistance_risk += 5

    if ecog >= 2:
        score -= 10
        confidence -= 5
        reasons.append("Reduced ECOG increases treatment risk")

    if pdl1 == "High" and egfr == "Negative":
        score += 12
        reasons.append("High PD-L1 may support immunotherapy strategy if accessible")
    elif pdl1 == "Unknown":
        confidence -= 5
        missing_data.append("PD-L1 result")

    if alk == "Unknown":
        missing_data.append("ALK status")
        confidence -= 3

    if ros1 == "Unknown":
        missing_data.append("ROS1 status")
        confidence -= 3

    if specimen_quality == "Insufficient for molecular testing":
        score -= 15
        confidence -= 20
        reasons.append("Insufficient sample limits precision-medicine confidence")

    if molecular_status in ["Not requested", "Insufficient material"]:
        confidence -= 15
        missing_data.append("Complete molecular testing")

    score = max(0, min(100, score))
    resistance_risk = max(0, min(100, resistance_risk))
    confidence = max(0, min(100, confidence))

    if score >= 75:
        predicted_response = "High predicted benefit"
    elif score >= 55:
        predicted_response = "Intermediate predicted benefit"
    else:
        predicted_response = "Low predicted benefit"

    if egfr in ["Exon 19 deletion", "L858R"]:
        recommended = "Osimertinib"
        tunisia_strategy = (
            "Best global option: Osimertinib. Tunisia layer: verify hospital pharmacy access. "
            "If unavailable, consider erlotinib/gefitinib/afatinib or platinum-based chemotherapy according to local protocol."
        )
        drug_design = (
            "Research only: EGFR inhibitor scaffold prioritization using PyMOL, AutoDock Vina, RDKit and ADMET filtering."
        )
    elif egfr == "T790M":
        recommended = "Advanced EGFR resistance strategy"
        tunisia_strategy = (
            "Resistance profile: verify access to appropriate EGFR-targeted therapy; request MET amplification/resistance workup if possible."
        )
        drug_design = (
            "Research only: prioritize resistant EGFR conformations and possible EGFR/MET combination strategies."
        )
    elif egfr == "Exon 20 insertion":
        recommended = "Specific EGFR exon 20 strategy / local systemic therapy"
        tunisia_strategy = (
            "Rare EGFR alteration: verify targeted therapy availability. If unavailable, use local systemic protocol and consider research referral."
        )
        drug_design = (
            "Research only: exon 20 insertion should trigger docking/scaffold screening."
        )
    elif egfr == "Negative" and pdl1 == "High":
        recommended = "Immunotherapy + chemotherapy if accessible"
        tunisia_strategy = (
            "EGFR negative with high PD-L1: immunotherapy + chemotherapy may be globally indicated. "
            "Tunisia access must be verified; if unavailable, use chemotherapy-based local protocol."
        )
        drug_design = (
            "Research only: expand molecular profiling before rational docking."
        )
    else:
        recommended = "Complete molecular testing first"
        tunisia_strategy = (
            "Incomplete profile: request EGFR, ALK, ROS1 and PD-L1 testing if possible. "
            "Use pathology, stage and local protocols while waiting."
        )
        drug_design = (
            "Research only: insufficient molecular data for rational drug design."
        )

    return {
        "predicted_efficiency": score,
        "predicted_response": predicted_response,
        "resistance_risk": resistance_risk,
        "confidence": confidence,
        "recommended_treatment": recommended,
        "tunisia_strategy": tunisia_strategy,
        "drug_design_research": drug_design,
        "reasons": reasons,
        "missing_data": missing_data,
        "model_version": "rule_based_secure_v0.2_replaceable_by_ml",
        "generated_at": now_iso()
    }

def extract_patient_id(payload):
    patient_id = payload.get("patient", {}).get("patient_id") or payload.get("patient_id")
    if not patient_id:
        raise HTTPException(status_code=400, detail="patient.patient_id is required")
    return str(patient_id).strip()

# ----------------------------
# Startup
# ----------------------------

@app.on_event("startup")
def startup():
    init_db()

# ----------------------------
# Auth endpoints
# ----------------------------

@app.post("/api/auth/login")
def login(data: Dict[str, Any] = Body(...)):
    username = str(data.get("username", "")).strip()
    password = str(data.get("password", ""))

    with get_conn() as conn:
        user = conn.execute(
            "SELECT * FROM users WHERE username=? AND is_active=1",
            (username,)
        ).fetchone()

        if not user:
            raise HTTPException(status_code=401, detail="Invalid credentials")

        if not verify_password(password, user["salt"], user["password_hash"]):
            raise HTTPException(status_code=401, detail="Invalid credentials")

        token = secrets.token_urlsafe(32)
        expires_at = int(time.time()) + 12 * 60 * 60

        conn.execute(
            "INSERT INTO sessions(token, user_id, created_at, expires_at) VALUES (?, ?, ?, ?)",
            (token, user["id"], now_iso(), expires_at)
        )
        conn.commit()

    user_public = {
        "id": user["id"],
        "username": user["username"],
        "full_name": user["full_name"],
        "role": user["role"]
    }

    audit("login", user_public)

    return {
        "token": token,
        "user": user_public,
        "expires_in_hours": 12
    }

@app.get("/api/auth/me")
def me(user: Dict[str, Any] = Depends(get_current_user)):
    return user

@app.post("/api/auth/logout")
def logout(authorization: Optional[str] = Header(default=None), user: Dict[str, Any] = Depends(get_current_user)):
    token = authorization.replace("Bearer ", "").strip()

    with get_conn() as conn:
        conn.execute("DELETE FROM sessions WHERE token=?", (token,))
        conn.commit()

    audit("logout", user)
    return {"message": "logged_out"}

# ----------------------------
# API endpoints
# ----------------------------

@app.get("/api/health")
def health():
    return {
        "status": "online",
        "system": "Hospital Anapath Digital Twin",
        "version": "0.2.0",
        "database": str(DB_PATH),
        "time": now_iso()
    }

@app.post("/api/predict")
def predict(payload: Dict[str, Any] = Body(...), user: Dict[str, Any] = Depends(get_current_user)):
    prediction = predict_case(payload)
    audit("predict", user, extract_patient_id(payload) if payload.get("patient", {}).get("patient_id") else "", {})
    return {"payload": payload, "prediction": prediction}

@app.post("/api/cases")
def save_case(
    payload: Dict[str, Any] = Body(...),
    user: Dict[str, Any] = Depends(require_roles("admin", "pathologist"))
):
    patient_id = extract_patient_id(payload)
    prediction = predict_case(payload)

    payload_json = json.dumps(payload, ensure_ascii=False)
    prediction_json = json.dumps(prediction, ensure_ascii=False)

    with get_conn() as conn:
        existing = conn.execute(
            "SELECT id FROM cases WHERE patient_id=?",
            (patient_id,)
        ).fetchone()

        if existing:
            conn.execute(
                """
                UPDATE cases
                SET payload=?, prediction=?, updated_by=?, updated_at=?
                WHERE patient_id=?
                """,
                (payload_json, prediction_json, user["username"], now_iso(), patient_id)
            )
            case_id = existing["id"]
            action = "update_case"
        else:
            created = now_iso()
            cur = conn.execute(
                """
                INSERT INTO cases(patient_id, payload, prediction, created_by, updated_by, created_at, updated_at)
                VALUES (?, ?, ?, ?, ?, ?, ?)
                """,
                (patient_id, payload_json, prediction_json, user["username"], user["username"], created, created)
            )
            case_id = cur.lastrowid
            action = "create_case"

        conn.commit()

    audit(action, user, patient_id, {"case_id": case_id})

    return {
        "message": "case_saved",
        "case_id": case_id,
        "patient_id": patient_id,
        "prediction": prediction
    }

@app.get("/api/cases")
def list_cases(user: Dict[str, Any] = Depends(get_current_user)):
    with get_conn() as conn:
        rows = conn.execute(
            "SELECT * FROM cases ORDER BY updated_at DESC"
        ).fetchall()

    cases = []
    for r in rows:
        cases.append({
            "id": r["id"],
            "patient_id": r["patient_id"],
            "payload": json.loads(r["payload"]),
            "prediction": json.loads(r["prediction"]),
            "created_by": r["created_by"],
            "updated_by": r["updated_by"],
            "created_at": r["created_at"],
            "updated_at": r["updated_at"]
        })

    return cases

@app.get("/api/cases/{case_id}")
def get_case(case_id: int, user: Dict[str, Any] = Depends(get_current_user)):
    with get_conn() as conn:
        r = conn.execute("SELECT * FROM cases WHERE id=?", (case_id,)).fetchone()

    if not r:
        raise HTTPException(status_code=404, detail="Case not found")

    return {
        "id": r["id"],
        "patient_id": r["patient_id"],
        "payload": json.loads(r["payload"]),
        "prediction": json.loads(r["prediction"]),
        "created_by": r["created_by"],
        "updated_by": r["updated_by"],
        "created_at": r["created_at"],
        "updated_at": r["updated_at"]
    }

@app.delete("/api/cases/{case_id}")
def delete_case(case_id: int, user: Dict[str, Any] = Depends(require_roles("admin"))):
    with get_conn() as conn:
        r = conn.execute("SELECT patient_id FROM cases WHERE id=?", (case_id,)).fetchone()
        if not r:
            raise HTTPException(status_code=404, detail="Case not found")

        patient_id = r["patient_id"]
        conn.execute("DELETE FROM cases WHERE id=?", (case_id,))
        conn.commit()

    audit("delete_case", user, patient_id, {"case_id": case_id})
    return {"message": "case_deleted", "case_id": case_id}

@app.get("/api/stats")
def stats(user: Dict[str, Any] = Depends(get_current_user)):
    cases = list_cases(user)

    total = len(cases)
    egfr_mutated = 0
    stage_iv = 0
    high_benefit = 0

    for c in cases:
        payload = c["payload"]
        pred = c["prediction"]

        egfr = payload.get("molecular", {}).get("egfr", "")
        stage = payload.get("pathology", {}).get("stage", "")

        if egfr in ["Exon 19 deletion", "L858R", "T790M", "Exon 20 insertion", "Other EGFR mutation"]:
            egfr_mutated += 1

        if stage == "IV":
            stage_iv += 1

        if pred.get("predicted_efficiency", 0) >= 75:
            high_benefit += 1

    return {
        "total_cases": total,
        "egfr_mutated_cases": egfr_mutated,
        "stage_iv_cases": stage_iv,
        "high_predicted_benefit_cases": high_benefit
    }

@app.get("/api/export/cases")
def export_cases(user: Dict[str, Any] = Depends(require_roles("admin", "researcher"))):
    return {
        "exported_at": now_iso(),
        "system": "Hospital Anapath Digital Twin",
        "cases": list_cases(user)
    }

@app.get("/api/export/fhir/{case_id}")
def export_fhir(case_id: int, user: Dict[str, Any] = Depends(get_current_user)):
    case = get_case(case_id, user)
    payload = case["payload"]
    prediction = case["prediction"]

    patient = payload.get("patient", {})
    sample = payload.get("sample", {})
    pathology = payload.get("pathology", {})
    molecular = payload.get("molecular", {})

    return {
        "resourceType": "DiagnosticReport",
        "id": f"diagnostic-report-{case_id}",
        "status": "final",
        "code": {
            "text": "Anatomopathology lung cancer precision medicine report"
        },
        "subject": {
            "identifier": {
                "value": patient.get("patient_id", case["patient_id"])
            }
        },
        "specimen": [
            {
                "identifier": {
                    "value": sample.get("sample_id", "")
                },
                "type": {
                    "text": sample.get("sample_type", "")
                }
            }
        ],
        "conclusion": (
            f"Histology: {pathology.get('histology', 'Unknown')}. "
            f"Stage: {pathology.get('stage', 'Unknown')}. "
            f"EGFR: {molecular.get('egfr', 'Unknown')}. "
            f"Predicted response: {prediction.get('predicted_response')}. "
            f"Recommended strategy: {prediction.get('tunisia_strategy')}"
        ),
        "digitalTwinPrediction": prediction
    }

@app.get("/api/audit")
def get_audit(user: Dict[str, Any] = Depends(require_roles("admin"))):
    with get_conn() as conn:
        rows = conn.execute(
            "SELECT * FROM audit_log ORDER BY id DESC LIMIT 200"
        ).fetchall()

    return [dict(r) for r in rows]

@app.post("/api/seed-demo")
def seed_demo(user: Dict[str, Any] = Depends(require_roles("admin"))):
    demo_cases = [
        {
            "patient": {
                "patient_id": "HM-TN-LC-0001",
                "age": 61,
                "sex": "Male",
                "region": "Tunis",
                "smoking_status": "Former smoker",
                "ecog": 1
            },
            "sample": {
                "sample_id": "ANA-2026-0001",
                "sample_type": "Bronchial biopsy",
                "anatomical_site": "Lung upper lobe",
                "reception_date": "2026-05-14",
                "specimen_quality": "Good",
                "pathologist_comment": "Tumor material adequate for molecular testing."
            },
            "pathology": {
                "cancer_type": "NSCLC",
                "histology": "Adenocarcinoma",
                "grade": "Moderately differentiated",
                "stage": "IV",
                "tnm": "T2N2M1",
                "metastasis": "Bone"
            },
            "molecular": {
                "egfr": "Exon 19 deletion",
                "alk": "Unknown",
                "ros1": "Unknown",
                "pdl1": "Unknown",
                "ihc_markers": "TTF-1+, Napsin A+",
                "molecular_test_status": "Completed"
            }
        },
        {
            "patient": {
                "patient_id": "HM-TN-LC-0002",
                "age": 68,
                "sex": "Male",
                "region": "Sousse",
                "smoking_status": "Current smoker",
                "ecog": 2
            },
            "sample": {
                "sample_id": "ANA-2026-0002",
                "sample_type": "CT-guided biopsy",
                "anatomical_site": "Lung mass",
                "reception_date": "2026-05-14",
                "specimen_quality": "Limited material",
                "pathologist_comment": "Small biopsy."
            },
            "pathology": {
                "cancer_type": "NSCLC",
                "histology": "Squamous cell carcinoma",
                "grade": "Poorly differentiated",
                "stage": "IV",
                "tnm": "T4N2M1",
                "metastasis": "Liver"
            },
            "molecular": {
                "egfr": "Negative",
                "alk": "Unknown",
                "ros1": "Unknown",
                "pdl1": "High",
                "ihc_markers": "p40+, TTF-1-",
                "molecular_test_status": "Completed"
            }
        }
    ]

    saved = []

    for payload in demo_cases:
        patient_id = extract_patient_id(payload)
        prediction = predict_case(payload)

        with get_conn() as conn:
            existing = conn.execute(
                "SELECT id FROM cases WHERE patient_id=?",
                (patient_id,)
            ).fetchone()

            if existing:
                conn.execute(
                    "UPDATE cases SET payload=?, prediction=?, updated_by=?, updated_at=? WHERE patient_id=?",
                    (json.dumps(payload), json.dumps(prediction), user["username"], now_iso(), patient_id)
                )
                case_id = existing["id"]
            else:
                created = now_iso()
                cur = conn.execute(
                    """
                    INSERT INTO cases(patient_id, payload, prediction, created_by, updated_by, created_at, updated_at)
                    VALUES (?, ?, ?, ?, ?, ?, ?)
                    """,
                    (patient_id, json.dumps(payload), json.dumps(prediction), user["username"], user["username"], created, created)
                )
                case_id = cur.lastrowid

            conn.commit()

        saved.append({"case_id": case_id, "patient_id": patient_id})

    audit("seed_demo", user, "", {"count": len(saved)})

    return {"message": "demo_cases_seeded", "saved": saved}

# Static frontend
app.mount("/", StaticFiles(directory=str(FRONTEND_DIR), html=True), name="frontend")
'''

APP_PATH.write_text(app_code, encoding="utf-8")
print("Created secure backend:", APP_PATH)

Created secure backend: /content/drive/MyDrive/pfe_hospital_anapath_real_system/app.py


In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/pfe_hospital_anapath_real_system")
FRONTEND_DIR = PROJECT_DIR / "frontend"
FRONTEND_DIR.mkdir(parents=True, exist_ok=True)

index_html = r'''
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>Hospital Anapath Digital Twin v0.2</title>
  <style>
    body { margin:0; font-family:Arial,sans-serif; background:#f3f6fb; color:#0f172a; }
    header { background:#0f172a; color:white; padding:24px 34px; }
    header h1 { margin:0; font-size:30px; }
    header p { color:#cbd5e1; margin-bottom:0; }
    .container { max-width:1300px; margin:auto; padding:26px 34px; }
    .alert { background:#fff7ed; border-left:5px solid #f97316; padding:14px; border-radius:12px; margin-bottom:20px; color:#7c2d12; }
    .grid { display:grid; grid-template-columns:repeat(2,1fr); gap:18px; margin-bottom:18px; }
    .grid4 { display:grid; grid-template-columns:repeat(4,1fr); gap:18px; margin-bottom:18px; }
    .card { background:white; border-radius:18px; padding:20px; box-shadow:0 6px 18px rgba(15,23,42,.08); border:1px solid #e5e7eb; }
    h2 { margin-top:0; color:#0f172a; font-size:20px; }
    label { font-size:13px; color:#475569; font-weight:bold; display:block; margin-top:11px; margin-bottom:5px; }
    input, select, textarea { width:100%; box-sizing:border-box; padding:10px; border-radius:10px; border:1px solid #cbd5e1; font-size:14px; background:white; }
    textarea { min-height:78px; resize:vertical; }
    button { border:none; padding:12px 16px; border-radius:12px; font-weight:bold; cursor:pointer; margin:5px 5px 5px 0; }
    .primary { background:#2563eb; color:white; }
    .success { background:#16a34a; color:white; }
    .danger { background:#dc2626; color:white; }
    .secondary { background:#e2e8f0; color:#0f172a; }
    .purple { background:#7e22ce; color:white; }
    .value { font-size:30px; font-weight:bold; color:#2563eb; }
    .small { color:#64748b; font-size:13px; }
    .pill { display:inline-block; padding:6px 10px; border-radius:999px; color:white; background:#2563eb; margin:4px 4px 0 0; font-size:13px; }
    .green { background:#15803d; }
    .orange { background:#c2410c; }
    .red { background:#b91c1c; }
    table { width:100%; border-collapse:collapse; background:white; border-radius:16px; overflow:hidden; box-shadow:0 6px 18px rgba(15,23,42,.08); margin-top:16px; }
    th, td { padding:12px; border-bottom:1px solid #e5e7eb; text-align:left; vertical-align:top; font-size:14px; }
    th { background:#0f172a; color:white; }
    pre { white-space:pre-wrap; background:#020617; color:#d1fae5; padding:15px; border-radius:12px; max-height:350px; overflow:auto; }
    .bar-bg { background:#e2e8f0; height:28px; border-radius:20px; overflow:hidden; margin-top:8px; }
    .bar { background:#2563eb; color:white; height:28px; line-height:28px; padding-left:10px; font-weight:bold; width:0%; }
    #appSection { display:none; }
    #loginSection { max-width:460px; margin:60px auto; }
    @media(max-width:1000px){ .grid,.grid4{grid-template-columns:1fr;} }
  </style>
</head>

<body>
<header>
  <h1>Hospital Anapath Digital Twin System v0.2</h1>
  <p>Secure dynamic prototype: login, roles, SQLite, audit log, prediction and hospital case registry</p>
</header>

<div id="loginSection" class="card">
  <h2>Login</h2>
  <p class="small">Demo accounts:</p>
  <p class="small">
    admin / admin123<br>
    pathologist / patho123<br>
    oncologist / onco123<br>
    researcher / research123
  </p>

  <label>Username</label>
  <input id="loginUsername" value="admin">

  <label>Password</label>
  <input id="loginPassword" type="password" value="admin123">

  <button class="primary" onclick="login()">Login</button>
  <p id="loginError" style="color:#dc2626;font-weight:bold;"></p>
</div>

<div id="appSection">
  <div class="container">

    <div class="alert">
      <b>Prototype rule:</b> Use anonymized IDs only. No real names, CIN, phone numbers, addresses or identifiable patient data.
      Logged in as: <b id="userBadge"></b>
      <button class="secondary" onclick="logout()">Logout</button>
    </div>

    <div class="grid4">
      <div class="card"><div class="small">Total cases</div><div class="value" id="statTotal">0</div></div>
      <div class="card"><div class="small">EGFR mutated</div><div class="value" id="statEgfr">0</div></div>
      <div class="card"><div class="small">Stage IV</div><div class="value" id="statStage">0</div></div>
      <div class="card"><div class="small">High benefit</div><div class="value" id="statHigh">0</div></div>
    </div>

    <button class="primary" onclick="predictLive()">Run Prediction</button>
    <button class="success" onclick="saveCase()">Save / Update Case</button>
    <button class="purple" onclick="seedDemo()">Add Demo Cases</button>
    <button class="secondary" onclick="refreshCases()">Refresh Cases</button>
    <button class="secondary" onclick="openApiDocs()">Open API Docs</button>
    <button class="secondary" onclick="openExport()">Export Dataset</button>
    <button class="secondary" onclick="openAudit()">Audit Log</button>

    <div class="grid">
      <div class="card">
        <h2>1. Patient Registry</h2>
        <label>Anonymous Patient ID</label>
        <input id="patient_id" placeholder="HM-TN-LC-0001">

        <label>Age</label>
        <input id="age" type="number" placeholder="61">

        <label>Sex</label>
        <select id="sex"><option>Male</option><option>Female</option></select>

        <label>Region</label>
        <input id="region" placeholder="Tunis / Monastir / Sousse">

        <label>Smoking Status</label>
        <select id="smoking_status">
          <option>Unknown</option><option>Never smoker</option><option>Former smoker</option><option>Current smoker</option>
        </select>

        <label>ECOG</label>
        <select id="ecog"><option value="0">0</option><option value="1">1</option><option value="2">2</option><option value="3">3</option><option value="4">4</option></select>
      </div>

      <div class="card">
        <h2>2. Anapath Sample</h2>
        <label>Sample ID</label>
        <input id="sample_id" placeholder="ANA-2026-0001">

        <label>Sample Type</label>
        <select id="sample_type">
          <option>Bronchial biopsy</option><option>CT-guided biopsy</option><option>Surgical specimen</option><option>FFPE block</option><option>Cytology</option>
        </select>

        <label>Anatomical Site</label>
        <input id="anatomical_site" placeholder="Lung upper lobe / lymph node / pleura">

        <label>Reception Date</label>
        <input id="reception_date" type="date">

        <label>Specimen Quality</label>
        <select id="specimen_quality">
          <option>Good</option><option>Limited material</option><option>Necrotic</option><option>Insufficient for molecular testing</option>
        </select>

        <label>Pathologist Comment</label>
        <textarea id="pathologist_comment" placeholder="Short anapath note..."></textarea>
      </div>
    </div>

    <div class="grid">
      <div class="card">
        <h2>3. Pathology Report</h2>
        <label>Cancer Type</label>
        <select id="cancer_type"><option>NSCLC</option><option>SCLC</option><option>Other</option></select>

        <label>Histology</label>
        <select id="histology">
          <option>Adenocarcinoma</option><option>Squamous cell carcinoma</option><option>Large cell carcinoma</option><option>NSCLC NOS</option><option>Small cell carcinoma</option>
        </select>

        <label>Grade</label>
        <select id="grade">
          <option>Unknown</option><option>Well differentiated</option><option>Moderately differentiated</option><option>Poorly differentiated</option>
        </select>

        <label>Stage</label>
        <select id="stage"><option>I</option><option>II</option><option>III</option><option>IV</option></select>

        <label>TNM</label>
        <input id="tnm" placeholder="T2N2M1">

        <label>Metastasis</label>
        <input id="metastasis" placeholder="None / bone / liver / brain">
      </div>

      <div class="card">
        <h2>4. Molecular / IHC Profile</h2>
        <label>EGFR</label>
        <select id="egfr">
          <option>Unknown</option><option>Negative</option><option>Exon 19 deletion</option><option>L858R</option><option>T790M</option><option>Exon 20 insertion</option><option>Other EGFR mutation</option>
        </select>

        <label>ALK</label>
        <select id="alk"><option>Unknown</option><option>Negative</option><option>Positive</option></select>

        <label>ROS1</label>
        <select id="ros1"><option>Unknown</option><option>Negative</option><option>Positive</option></select>

        <label>PD-L1</label>
        <select id="pdl1"><option>Unknown</option><option>Negative</option><option>Low</option><option>High</option></select>

        <label>IHC Markers</label>
        <input id="ihc_markers" placeholder="TTF-1+, Napsin A+, p40-">

        <label>Molecular Test Status</label>
        <select id="molecular_test_status">
          <option>Not requested</option><option>Requested</option><option>Completed</option><option>Insufficient material</option>
        </select>
      </div>
    </div>

    <div class="grid">
      <div class="card">
        <h2>Real-Time Digital Twin Prediction</h2>
        <div class="small">Predicted efficiency</div>
        <div class="bar-bg"><div class="bar" id="effBar">0%</div></div>
        <p><b>Response:</b> <span id="predResponse">--</span></p>
        <p><b>Resistance risk:</b> <span id="resRisk">--</span></p>
        <p><b>Confidence:</b> <span id="confidence">--</span></p>
        <div id="reasonPills"></div>
      </div>

      <div class="card">
        <h2>Treatment + Research Output</h2>
        <p><b>Recommended treatment:</b> <span id="recommended">--</span></p>
        <p><b>Tunisia strategy:</b></p>
        <p id="tnStrategy">--</p>
        <p><b>Drug design research:</b></p>
        <p id="drugDesign">--</p>
      </div>
    </div>

    <h2>Saved Hospital Cases</h2>
    <table>
      <thead>
        <tr>
          <th>ID</th><th>Patient</th><th>Histology</th><th>Stage</th><th>EGFR</th><th>Prediction</th><th>Updated By</th><th>Action</th>
        </tr>
      </thead>
      <tbody id="casesBody"></tbody>
    </table>

    <h2>Live JSON Output</h2>
    <pre id="jsonPreview">No prediction yet.</pre>
  </div>
</div>

<script>
let TOKEN = localStorage.getItem("hospital_token") || "";
let CURRENT_USER = null;

function value(id) { return document.getElementById(id).value; }
function setValue(id, v) { document.getElementById(id).value = v ?? ""; }

async function apiFetch(url, options = {}) {
  options.headers = options.headers || {};
  options.headers["Content-Type"] = "application/json";
  if (TOKEN) options.headers["Authorization"] = "Bearer " + TOKEN;

  const res = await fetch(url, options);

  if (res.status === 401) {
    localStorage.removeItem("hospital_token");
    TOKEN = "";
    showLogin();
    throw new Error("Session expired. Please log in again.");
  }

  if (!res.ok) {
    const msg = await res.text();
    throw new Error(msg);
  }

  return await res.json();
}

async function login() {
  try {
    const data = await apiFetch("/api/auth/login", {
      method: "POST",
      body: JSON.stringify({
        username: value("loginUsername"),
        password: value("loginPassword")
      })
    });

    TOKEN = data.token;
    CURRENT_USER = data.user;
    localStorage.setItem("hospital_token", TOKEN);
    showApp();
    await refreshCases();
  } catch (e) {
    document.getElementById("loginError").textContent = e.message;
  }
}

async function logout() {
  try {
    await apiFetch("/api/auth/logout", {method: "POST", body: "{}"});
  } catch(e) {}
  localStorage.removeItem("hospital_token");
  TOKEN = "";
  CURRENT_USER = null;
  showLogin();
}

function showLogin() {
  document.getElementById("loginSection").style.display = "block";
  document.getElementById("appSection").style.display = "none";
}

function showApp() {
  document.getElementById("loginSection").style.display = "none";
  document.getElementById("appSection").style.display = "block";
  document.getElementById("userBadge").textContent =
    CURRENT_USER.username + " (" + CURRENT_USER.role + ")";
}

async function checkSession() {
  if (!TOKEN) {
    showLogin();
    return;
  }

  try {
    CURRENT_USER = await apiFetch("/api/auth/me");
    showApp();
    await refreshCases();
  } catch(e) {
    showLogin();
  }
}

function buildPayload() {
  return {
    patient: {
      patient_id: value("patient_id"),
      age: Number(value("age")),
      sex: value("sex"),
      region: value("region"),
      smoking_status: value("smoking_status"),
      ecog: Number(value("ecog"))
    },
    sample: {
      sample_id: value("sample_id"),
      sample_type: value("sample_type"),
      anatomical_site: value("anatomical_site"),
      reception_date: value("reception_date"),
      specimen_quality: value("specimen_quality"),
      pathologist_comment: value("pathologist_comment")
    },
    pathology: {
      cancer_type: value("cancer_type"),
      histology: value("histology"),
      grade: value("grade"),
      stage: value("stage"),
      tnm: value("tnm"),
      metastasis: value("metastasis")
    },
    molecular: {
      egfr: value("egfr"),
      alk: value("alk"),
      ros1: value("ros1"),
      pdl1: value("pdl1"),
      ihc_markers: value("ihc_markers"),
      molecular_test_status: value("molecular_test_status")
    }
  };
}

function renderPrediction(result) {
  const p = result.prediction;
  const eff = p.predicted_efficiency || 0;

  const bar = document.getElementById("effBar");
  bar.style.width = eff + "%";
  bar.textContent = eff + "%";

  document.getElementById("predResponse").textContent = p.predicted_response;
  document.getElementById("resRisk").textContent = p.resistance_risk + "%";
  document.getElementById("confidence").textContent = p.confidence + "%";
  document.getElementById("recommended").textContent = p.recommended_treatment;
  document.getElementById("tnStrategy").textContent = p.tunisia_strategy;
  document.getElementById("drugDesign").textContent = p.drug_design_research;

  const reasons = (p.reasons || []).map(x => `<span class="pill green">${x}</span>`).join("");
  const missing = (p.missing_data || []).map(x => `<span class="pill orange">Missing: ${x}</span>`).join("");
  document.getElementById("reasonPills").innerHTML = reasons + missing;

  document.getElementById("jsonPreview").textContent = JSON.stringify(result, null, 2);
}

async function predictLive() {
  const payload = buildPayload();
  const result = await apiFetch("/api/predict", {
    method: "POST",
    body: JSON.stringify(payload)
  });
  renderPrediction(result);
}

async function saveCase() {
  const payload = buildPayload();

  if (!payload.patient.patient_id) {
    alert("Anonymous Patient ID is required.");
    return;
  }

  const result = await apiFetch("/api/cases", {
    method: "POST",
    body: JSON.stringify(payload)
  });

  renderPrediction({payload, prediction: result.prediction});
  await refreshCases();
  alert("Case saved to SQLite database.");
}

async function refreshCases() {
  const cases = await apiFetch("/api/cases");
  const body = document.getElementById("casesBody");

  body.innerHTML = cases.map(c => {
    const p = c.payload;
    const pred = c.prediction;

    return `
      <tr>
        <td>${c.id}</td>
        <td><b>${c.patient_id}</b></td>
        <td>${p.pathology?.histology || ""}</td>
        <td>${p.pathology?.stage || ""}</td>
        <td>${p.molecular?.egfr || ""}</td>
        <td>${pred.predicted_efficiency}% - ${pred.predicted_response}</td>
        <td>${c.updated_by || ""}</td>
        <td>
          <button class="secondary" onclick="loadCase(${c.id})">Load</button>
          <button class="purple" onclick="openFHIR(${c.id})">FHIR</button>
          <button class="danger" onclick="deleteCase(${c.id})">Delete</button>
        </td>
      </tr>
    `;
  }).join("");

  await refreshStats();
}

async function refreshStats() {
  const s = await apiFetch("/api/stats");
  document.getElementById("statTotal").textContent = s.total_cases;
  document.getElementById("statEgfr").textContent = s.egfr_mutated_cases;
  document.getElementById("statStage").textContent = s.stage_iv_cases;
  document.getElementById("statHigh").textContent = s.high_predicted_benefit_cases;
}

async function loadCase(id) {
  const c = await apiFetch(`/api/cases/${id}`);
  const p = c.payload;

  setValue("patient_id", p.patient.patient_id);
  setValue("age", p.patient.age);
  setValue("sex", p.patient.sex);
  setValue("region", p.patient.region);
  setValue("smoking_status", p.patient.smoking_status);
  setValue("ecog", p.patient.ecog);

  setValue("sample_id", p.sample.sample_id);
  setValue("sample_type", p.sample.sample_type);
  setValue("anatomical_site", p.sample.anatomical_site);
  setValue("reception_date", p.sample.reception_date);
  setValue("specimen_quality", p.sample.specimen_quality);
  setValue("pathologist_comment", p.sample.pathologist_comment);

  setValue("cancer_type", p.pathology.cancer_type);
  setValue("histology", p.pathology.histology);
  setValue("grade", p.pathology.grade);
  setValue("stage", p.pathology.stage);
  setValue("tnm", p.pathology.tnm);
  setValue("metastasis", p.pathology.metastasis);

  setValue("egfr", p.molecular.egfr);
  setValue("alk", p.molecular.alk);
  setValue("ros1", p.molecular.ros1);
  setValue("pdl1", p.molecular.pdl1);
  setValue("ihc_markers", p.molecular.ihc_markers);
  setValue("molecular_test_status", p.molecular.molecular_test_status);

  renderPrediction({payload: p, prediction: c.prediction});
  window.scrollTo({top: 0, behavior: "smooth"});
}

async function deleteCase(id) {
  if (!confirm("Delete this case? Admin role required.")) return;

  try {
    await apiFetch(`/api/cases/${id}`, {method: "DELETE"});
    await refreshCases();
  } catch(e) {
    alert(e.message);
  }
}

async function seedDemo() {
  try {
    await apiFetch("/api/seed-demo", {method: "POST", body: "{}"});
    await refreshCases();
    alert("Demo cases added.");
  } catch(e) {
    alert(e.message);
  }
}

function openFHIR(id) {
  window.open(`/api/export/fhir/${id}?token_warning=use_docs_with_auth`, "_blank");
  alert("FHIR export requires Authorization in API clients. Use /docs for authenticated testing, or copy token manually.");
}

function openApiDocs() {
  window.open("/docs", "_blank");
}

function openExport() {
  window.open("/api/export/cases", "_blank");
}

function openAudit() {
  window.open("/api/audit", "_blank");
}

let timer = null;
document.querySelectorAll("input, select, textarea").forEach(el => {
  el.addEventListener("change", () => {
    clearTimeout(timer);
    timer = setTimeout(() => {
      const patientId = value("patient_id");
      if (patientId && TOKEN) predictLive().catch(() => {});
    }, 500);
  });
});

checkSession();
</script>
</body>
</html>
'''

(FRONTEND_DIR / "index.html").write_text(index_html, encoding="utf-8")
print("Created frontend:", FRONTEND_DIR / "index.html")

Created frontend: /content/drive/MyDrive/pfe_hospital_anapath_real_system/frontend/index.html


In [ ]:
import subprocess, time, os
from google.colab import output

PROJECT_DIR = "/content/drive/MyDrive/pfe_hospital_anapath_real_system"

!pkill -f "uvicorn" || true

server = subprocess.Popen(
    ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=PROJECT_DIR
)

time.sleep(4)

print("System running on port 8000")
output.serve_kernel_port_as_window(8000)

^C
System running on port 8000
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/pfe_hospital_anapath_real_system")
FRONTEND_DIR = PROJECT_DIR / "frontend"
FRONTEND_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Frontend:", FRONTEND_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project: /content/drive/MyDrive/pfe_hospital_anapath_real_system
Frontend: /content/drive/MyDrive/pfe_hospital_anapath_real_system/frontend


In [ ]:
index_html = r'''
<!DOCTYPE html>
<html lang="fr">
<head>
  <meta charset="UTF-8">
  <title>Système Anapath Digital Twin v0.3</title>
  <style>
    :root {
      --bg: #eef3f8;
      --dark: #0f172a;
      --dark2: #111827;
      --blue: #2563eb;
      --green: #16a34a;
      --red: #dc2626;
      --orange: #ea580c;
      --purple: #7e22ce;
      --border: #dbe3ef;
      --text: #0f172a;
      --muted: #64748b;
      --card: #ffffff;
    }

    * {
      box-sizing: border-box;
    }

    body {
      margin: 0;
      font-family: Arial, sans-serif;
      background: var(--bg);
      color: var(--text);
    }

    .login-page {
      min-height: 100vh;
      display: flex;
      align-items: center;
      justify-content: center;
      background: linear-gradient(135deg, #0f172a, #1e3a8a);
      padding: 30px;
    }

    .login-card {
      background: white;
      width: 440px;
      border-radius: 24px;
      padding: 30px;
      box-shadow: 0 20px 45px rgba(0,0,0,0.25);
    }

    .login-card h1 {
      margin: 0 0 8px;
      color: var(--dark);
      font-size: 27px;
    }

    .login-card p {
      color: var(--muted);
    }

    .app {
      display: none;
      min-height: 100vh;
    }

    .sidebar {
      width: 280px;
      background: var(--dark);
      color: white;
      position: fixed;
      left: 0;
      top: 0;
      bottom: 0;
      padding: 22px 18px;
      overflow-y: auto;
    }

    .brand {
      padding: 8px 10px 24px;
      border-bottom: 1px solid #334155;
      margin-bottom: 18px;
    }

    .brand h1 {
      font-size: 22px;
      margin: 0;
    }

    .brand p {
      font-size: 13px;
      color: #cbd5e1;
      margin-bottom: 0;
    }

    .nav-btn {
      display: block;
      width: 100%;
      text-align: left;
      background: transparent;
      color: #cbd5e1;
      border: 0;
      padding: 13px 14px;
      border-radius: 12px;
      margin-bottom: 8px;
      cursor: pointer;
      font-weight: bold;
      font-size: 14px;
    }

    .nav-btn:hover,
    .nav-btn.active {
      background: #1e293b;
      color: white;
    }

    .user-box {
      background: #111827;
      border: 1px solid #334155;
      border-radius: 16px;
      padding: 14px;
      margin-top: 18px;
      font-size: 13px;
      color: #cbd5e1;
    }

    .main {
      margin-left: 280px;
      padding: 24px;
    }

    .topbar {
      background: white;
      border: 1px solid var(--border);
      border-radius: 22px;
      padding: 20px 24px;
      box-shadow: 0 6px 18px rgba(15,23,42,0.06);
      margin-bottom: 22px;
      display: flex;
      justify-content: space-between;
      gap: 16px;
      align-items: center;
    }

    .topbar h2 {
      margin: 0;
      font-size: 24px;
      color: var(--dark);
    }

    .topbar p {
      margin: 5px 0 0;
      color: var(--muted);
    }

    .alert {
      background: #fff7ed;
      border-left: 5px solid var(--orange);
      padding: 14px;
      border-radius: 14px;
      margin-bottom: 18px;
      color: #7c2d12;
      line-height: 1.5;
    }

    .grid2 {
      display: grid;
      grid-template-columns: repeat(2, 1fr);
      gap: 18px;
      margin-bottom: 18px;
    }

    .grid3 {
      display: grid;
      grid-template-columns: repeat(3, 1fr);
      gap: 18px;
      margin-bottom: 18px;
    }

    .grid4 {
      display: grid;
      grid-template-columns: repeat(4, 1fr);
      gap: 18px;
      margin-bottom: 18px;
    }

    .card {
      background: var(--card);
      border: 1px solid var(--border);
      border-radius: 22px;
      padding: 20px;
      box-shadow: 0 6px 18px rgba(15,23,42,0.06);
    }

    .card h3 {
      margin: 0 0 14px;
      font-size: 19px;
      color: var(--dark);
    }

    label {
      display: block;
      margin-top: 12px;
      margin-bottom: 5px;
      font-weight: bold;
      color: #475569;
      font-size: 13px;
    }

    input, select, textarea {
      width: 100%;
      padding: 10px 11px;
      border-radius: 11px;
      border: 1px solid #cbd5e1;
      background: white;
      font-size: 14px;
    }

    textarea {
      min-height: 82px;
      resize: vertical;
    }

    button {
      border: none;
      padding: 11px 15px;
      border-radius: 12px;
      font-weight: bold;
      cursor: pointer;
      margin: 4px;
    }

    .btn-primary { background: var(--blue); color: white; }
    .btn-success { background: var(--green); color: white; }
    .btn-danger { background: var(--red); color: white; }
    .btn-secondary { background: #e2e8f0; color: var(--dark); }
    .btn-purple { background: var(--purple); color: white; }
    .btn-orange { background: var(--orange); color: white; }

    .stat-value {
      font-size: 34px;
      font-weight: bold;
      color: var(--blue);
      margin-top: 6px;
    }

    .small {
      font-size: 13px;
      color: var(--muted);
    }

    .section {
      display: none;
    }

    .section.active {
      display: block;
    }

    table {
      width: 100%;
      border-collapse: collapse;
      background: white;
      border-radius: 18px;
      overflow: hidden;
      box-shadow: 0 6px 18px rgba(15,23,42,0.06);
    }

    th, td {
      padding: 12px;
      border-bottom: 1px solid #e5e7eb;
      text-align: left;
      vertical-align: top;
      font-size: 14px;
    }

    th {
      background: var(--dark);
      color: white;
    }

    .pill {
      display: inline-block;
      padding: 6px 10px;
      border-radius: 999px;
      color: white;
      background: var(--blue);
      margin: 4px 4px 0 0;
      font-size: 12px;
      font-weight: bold;
    }

    .green { background: var(--green); }
    .orange { background: var(--orange); }
    .red { background: var(--red); }
    .purple { background: var(--purple); }
    .gray { background: #64748b; }

    .bar-bg {
      width: 100%;
      background: #e2e8f0;
      height: 28px;
      border-radius: 99px;
      overflow: hidden;
      margin-top: 10px;
    }

    .bar {
      width: 0%;
      background: var(--blue);
      color: white;
      height: 28px;
      line-height: 28px;
      padding-left: 10px;
      font-weight: bold;
      transition: width .3s ease;
    }

    pre {
      background: #020617;
      color: #d1fae5;
      padding: 16px;
      border-radius: 16px;
      white-space: pre-wrap;
      overflow: auto;
      max-height: 400px;
    }

    .toolbar {
      background: white;
      border: 1px solid var(--border);
      border-radius: 18px;
      padding: 14px;
      margin-bottom: 18px;
      display: flex;
      gap: 10px;
      flex-wrap: wrap;
      align-items: center;
    }

    .toolbar input,
    .toolbar select {
      width: auto;
      min-width: 220px;
      margin: 0;
    }

    .report-box {
      background: #f8fafc;
      border: 1px solid #dbe3ef;
      border-radius: 18px;
      padding: 18px;
      line-height: 1.65;
    }

    .status-draft { color: #b45309; font-weight: bold; }
    .status-progress { color: #2563eb; font-weight: bold; }
    .status-validated { color: #15803d; font-weight: bold; }

    @media(max-width: 1100px) {
      .sidebar {
        position: relative;
        width: 100%;
        height: auto;
      }

      .main {
        margin-left: 0;
      }

      .grid2, .grid3, .grid4 {
        grid-template-columns: 1fr;
      }

      .topbar {
        flex-direction: column;
        align-items: flex-start;
      }
    }
  </style>
</head>

<body>

<div id="loginPage" class="login-page">
  <div class="login-card">
    <h1>Système Anapath Digital Twin</h1>
    <p>Prototype hospitalier dynamique — interface française</p>

    <div class="alert">
      Utiliser uniquement des identifiants anonymisés. Ne pas saisir de nom, CIN, téléphone ou adresse réelle.
    </div>

    <label>Nom d'utilisateur</label>
    <input id="loginUsername" value="admin">

    <label>Mot de passe</label>
    <input id="loginPassword" type="password" value="admin123">

    <button class="btn-primary" onclick="login()" style="width:100%; margin-top:18px;">Connexion</button>

    <p id="loginError" style="color:#dc2626; font-weight:bold;"></p>

    <p class="small">
      Comptes démo :<br>
      admin / admin123<br>
      pathologist / patho123<br>
      oncologist / onco123<br>
      researcher / research123
    </p>
  </div>
</div>

<div id="app" class="app">
  <aside class="sidebar">
    <div class="brand">
      <h1>Anapath Twin</h1>
      <p>Mini-Syslab + Médecine de précision</p>
    </div>

    <button class="nav-btn active" onclick="showSection('dashboard', this)">Tableau de bord</button>
    <button class="nav-btn" onclick="showSection('newcase', this)">Nouveau dossier</button>
    <button class="nav-btn" onclick="showSection('cases', this)">Dossiers enregistrés</button>
    <button class="nav-btn" onclick="showSection('prediction', this)">Prédiction & traitement</button>
    <button class="nav-btn" onclick="showSection('report', this)">Compte rendu</button>
    <button class="nav-btn" onclick="showSection('exports', this)">Exports</button>
    <button class="nav-btn" onclick="showSection('audit', this)">Journal d'audit</button>

    <div class="user-box">
      <b>Utilisateur connecté</b>
      <div id="userBadge">--</div>
      <button class="btn-secondary" onclick="logout()" style="width:100%; margin-top:10px;">Déconnexion</button>
    </div>
  </aside>

  <main class="main">
    <div class="topbar">
      <div>
        <h2 id="pageTitle">Tableau de bord</h2>
        <p>Système dynamique pour l’anapath, la biologie moléculaire et la décision thérapeutique en Tunisie.</p>
      </div>
      <div>
        <button class="btn-primary" onclick="predictLive()">Prédire</button>
        <button class="btn-success" onclick="saveCase()">Enregistrer</button>
        <button class="btn-purple" onclick="seedDemo()">Cas démo</button>
      </div>
    </div>

    <div class="alert">
      <b>Objectif hospitalier :</b> créer un système compatible avec le flux réel d’un service d’anatomopathologie :
      dossier anonymisé → prélèvement → compte rendu → IHC/moléculaire → prédiction → stratégie thérapeutique → export.
    </div>

    <section id="dashboard" class="section active">
      <div class="grid4">
        <div class="card">
          <div class="small">Dossiers enregistrés</div>
          <div class="stat-value" id="statTotal">0</div>
        </div>
        <div class="card">
          <div class="small">Cas EGFR mutés</div>
          <div class="stat-value" id="statEgfr">0</div>
        </div>
        <div class="card">
          <div class="small">Stade IV</div>
          <div class="stat-value" id="statStage">0</div>
        </div>
        <div class="card">
          <div class="small">Bénéfice élevé prédit</div>
          <div class="stat-value" id="statHigh">0</div>
        </div>
      </div>

      <div class="grid2">
        <div class="card">
          <h3>Flux de travail hospitalier</h3>
          <p><span class="pill gray">1. Réception prélèvement</span></p>
          <p><span class="pill gray">2. Diagnostic histologique</span></p>
          <p><span class="pill gray">3. IHC / biologie moléculaire</span></p>
          <p><span class="pill gray">4. Prédiction digital twin</span></p>
          <p><span class="pill gray">5. Stratégie thérapeutique adaptée à la Tunisie</span></p>
        </div>

        <div class="card">
          <h3>Statut du dossier courant</h3>
          <p><b>ID patient :</b> <span id="currentPatientLabel">Aucun</span></p>
          <p><b>Statut :</b> <span id="currentStatusLabel">Non enregistré</span></p>
          <p><b>Dernière prédiction :</b> <span id="currentPredictionLabel">--</span></p>
        </div>
      </div>
    </section>

    <section id="newcase" class="section">
      <div class="grid2">
        <div class="card">
          <h3>1. Registre patient anonymisé</h3>

          <label>ID patient anonymisé</label>
          <input id="patient_id" placeholder="HM-TN-LC-0001">

          <label>Âge</label>
          <input id="age" type="number" placeholder="61">

          <label>Sexe</label>
          <select id="sex">
            <option>Homme</option>
            <option>Femme</option>
          </select>

          <label>Région</label>
          <input id="region" placeholder="Tunis / Monastir / Sousse">

          <label>Statut tabagique</label>
          <select id="smoking_status">
            <option>Inconnu</option>
            <option>Non-fumeur</option>
            <option>Ancien fumeur</option>
            <option>Fumeur actuel</option>
          </select>

          <label>ECOG</label>
          <select id="ecog">
            <option value="0">0 - Activité normale</option>
            <option value="1">1 - Activité limitée</option>
            <option value="2">2 - Ambulatoire >50%</option>
            <option value="3">3 - Soins personnels limités</option>
            <option value="4">4 - Alité</option>
          </select>
        </div>

        <div class="card">
          <h3>2. Prélèvement anapath</h3>

          <label>ID prélèvement</label>
          <input id="sample_id" placeholder="ANA-2026-0001">

          <label>Type de prélèvement</label>
          <select id="sample_type">
            <option>Biopsie bronchique</option>
            <option>Biopsie scanno-guidée</option>
            <option>Pièce opératoire</option>
            <option>Bloc FFPE</option>
            <option>Cytologie</option>
          </select>

          <label>Site anatomique</label>
          <input id="anatomical_site" placeholder="Lobe supérieur / ganglion / plèvre">

          <label>Date de réception</label>
          <input id="reception_date" type="date">

          <label>Qualité du prélèvement</label>
          <select id="specimen_quality">
            <option>Bonne</option>
            <option>Matériel limité</option>
            <option>Nécrotique</option>
            <option>Insuffisant pour test moléculaire</option>
          </select>

          <label>Statut du dossier</label>
          <select id="workflow_status">
            <option>Brouillon</option>
            <option>En cours d'analyse</option>
            <option>En attente IHC</option>
            <option>En attente moléculaire</option>
            <option>Validé</option>
          </select>
        </div>
      </div>

      <div class="grid2">
        <div class="card">
          <h3>3. Compte rendu anatomopathologique</h3>

          <label>Type de cancer</label>
          <select id="cancer_type">
            <option>CBNPC</option>
            <option>CBPC</option>
            <option>Autre</option>
          </select>

          <label>Histologie</label>
          <select id="histology">
            <option>Adénocarcinome</option>
            <option>Carcinome épidermoïde</option>
            <option>Carcinome à grandes cellules</option>
            <option>CBNPC NOS</option>
            <option>Carcinome à petites cellules</option>
          </select>

          <label>Grade</label>
          <select id="grade">
            <option>Inconnu</option>
            <option>Bien différencié</option>
            <option>Moyennement différencié</option>
            <option>Peu différencié</option>
          </select>

          <label>Stade</label>
          <select id="stage">
            <option>I</option>
            <option>II</option>
            <option>III</option>
            <option>IV</option>
          </select>

          <label>TNM</label>
          <input id="tnm" placeholder="T2N2M1">

          <label>Métastases</label>
          <input id="metastasis" placeholder="Aucune / os / foie / cerveau">

          <label>Conclusion anapath</label>
          <textarea id="pathologist_comment" placeholder="Conclusion du compte rendu..."></textarea>
        </div>

        <div class="card">
          <h3>4. Profil IHC / moléculaire</h3>

          <label>EGFR</label>
          <select id="egfr">
            <option>Inconnu</option>
            <option>Négatif</option>
            <option>Délétion exon 19</option>
            <option>L858R</option>
            <option>T790M</option>
            <option>Insertion exon 20</option>
            <option>Autre mutation EGFR</option>
          </select>

          <label>ALK</label>
          <select id="alk">
            <option>Inconnu</option>
            <option>Négatif</option>
            <option>Positif</option>
          </select>

          <label>ROS1</label>
          <select id="ros1">
            <option>Inconnu</option>
            <option>Négatif</option>
            <option>Positif</option>
          </select>

          <label>PD-L1</label>
          <select id="pdl1">
            <option>Inconnu</option>
            <option>Négatif</option>
            <option>Faible</option>
            <option>Élevé</option>
          </select>

          <label>Marqueurs IHC</label>
          <input id="ihc_markers" placeholder="TTF-1+, Napsin A+, p40-">

          <label>Statut du test moléculaire</label>
          <select id="molecular_test_status">
            <option>Non demandé</option>
            <option>Demandé</option>
            <option>Terminé</option>
            <option>Matériel insuffisant</option>
          </select>
        </div>
      </div>
    </section>

    <section id="cases" class="section">
      <div class="toolbar">
        <input id="searchInput" placeholder="Rechercher par ID, histologie, EGFR..." oninput="renderCasesTable()">
        <select id="stageFilter" onchange="renderCasesTable()">
          <option value="">Tous les stades</option>
          <option>I</option>
          <option>II</option>
          <option>III</option>
          <option>IV</option>
        </select>
        <select id="egfrFilter" onchange="renderCasesTable()">
          <option value="">Tous EGFR</option>
          <option>Délétion exon 19</option>
          <option>L858R</option>
          <option>T790M</option>
          <option>Insertion exon 20</option>
          <option>Négatif</option>
          <option>Inconnu</option>
        </select>
        <button class="btn-secondary" onclick="refreshCases()">Actualiser</button>
      </div>

      <table>
        <thead>
          <tr>
            <th>ID</th>
            <th>Patient</th>
            <th>Statut</th>
            <th>Histologie</th>
            <th>Stade</th>
            <th>EGFR</th>
            <th>Prédiction</th>
            <th>Mis à jour par</th>
            <th>Actions</th>
          </tr>
        </thead>
        <tbody id="casesBody"></tbody>
      </table>
    </section>

    <section id="prediction" class="section">
      <div class="grid2">
        <div class="card">
          <h3>Prédiction Digital Twin en temps réel</h3>
          <div class="small">Efficacité prédite</div>
          <div class="bar-bg"><div class="bar" id="effBar">0%</div></div>
          <p><b>Réponse prédite :</b> <span id="predResponse">--</span></p>
          <p><b>Risque de résistance :</b> <span id="resRisk">--</span></p>
          <p><b>Confiance :</b> <span id="confidence">--</span></p>
          <div id="reasonPills"></div>
        </div>

        <div class="card">
          <h3>Stratégie thérapeutique</h3>
          <p><b>Traitement recommandé :</b> <span id="recommended">--</span></p>
          <p><b>Adaptation au contexte tunisien :</b></p>
          <p id="tnStrategy">--</p>
          <p><b>Module recherche drug design :</b></p>
          <p id="drugDesign">--</p>
        </div>
      </div>

      <div class="card">
        <h3>Sortie JSON dynamique</h3>
        <pre id="jsonPreview">Aucune prédiction pour le moment.</pre>
      </div>
    </section>

    <section id="report" class="section">
      <div class="card">
        <h3>Compte rendu synthétique généré</h3>
        <div id="reportBox" class="report-box">
          Aucun dossier chargé.
        </div>
        <br>
        <button class="btn-secondary" onclick="generateReport()">Générer le compte rendu</button>
        <button class="btn-primary" onclick="window.print()">Imprimer / Exporter PDF</button>
      </div>
    </section>

    <section id="exports" class="section">
      <div class="grid2">
        <div class="card">
          <h3>Exports hospitaliers</h3>
          <button class="btn-secondary" onclick="exportDataset()">Exporter dataset JSON</button>
          <button class="btn-secondary" onclick="exportCurrentCase()">Exporter dossier courant JSON</button>
          <button class="btn-secondary" onclick="copyCurrentJSON()">Copier JSON courant</button>
          <p class="small">
            Les exports servent à entraîner les futurs modèles sur les données tunisiennes anonymisées.
          </p>
        </div>

        <div class="card">
          <h3>FHIR / interopérabilité</h3>
          <p>
            Pour chaque dossier, le système peut générer un format proche de FHIR DiagnosticReport.
            Dans la version hospitalière réelle, cette couche permettra l’intégration avec les systèmes d’information.
          </p>
          <p id="fhirHint" class="small">Chargez un dossier puis utilisez le bouton FHIR dans la liste des dossiers.</p>
        </div>
      </div>
    </section>

    <section id="audit" class="section">
      <div class="card">
        <h3>Journal d’audit</h3>
        <button class="btn-secondary" onclick="loadAudit()">Charger le journal</button>
        <pre id="auditBox">Réservé au rôle administrateur.</pre>
      </div>
    </section>

  </main>
</div>

<script>
let TOKEN = localStorage.getItem("hospital_token") || "";
let CURRENT_USER = null;
let ALL_CASES = [];
let CURRENT_RESULT = null;
let CURRENT_CASE_ID = null;

const mapToBackend = {
  "Homme": "Male",
  "Femme": "Female",
  "Inconnu": "Unknown",
  "Négatif": "Negative",
  "Positif": "Positive",
  "Délétion exon 19": "Exon 19 deletion",
  "Insertion exon 20": "Exon 20 insertion",
  "Autre mutation EGFR": "Other EGFR mutation",
  "Adénocarcinome": "Adenocarcinoma",
  "Carcinome épidermoïde": "Squamous cell carcinoma",
  "Carcinome à grandes cellules": "Large cell carcinoma",
  "Carcinome à petites cellules": "Small cell carcinoma",
  "CBNPC": "NSCLC",
  "CBPC": "SCLC",
  "Bonne": "Good",
  "Matériel limité": "Limited material",
  "Nécrotique": "Necrotic",
  "Insuffisant pour test moléculaire": "Insufficient for molecular testing",
  "Non demandé": "Not requested",
  "Demandé": "Requested",
  "Terminé": "Completed",
  "Matériel insuffisant": "Insufficient material",
  "Non-fumeur": "Never smoker",
  "Ancien fumeur": "Former smoker",
  "Fumeur actuel": "Current smoker",
  "Bien différencié": "Well differentiated",
  "Moyennement différencié": "Moderately differentiated",
  "Peu différencié": "Poorly differentiated",
  "Faible": "Low",
  "Élevé": "High"
};

const mapToFrench = {
  "Male": "Homme",
  "Female": "Femme",
  "Unknown": "Inconnu",
  "Negative": "Négatif",
  "Positive": "Positif",
  "Exon 19 deletion": "Délétion exon 19",
  "Exon 20 insertion": "Insertion exon 20",
  "Other EGFR mutation": "Autre mutation EGFR",
  "Adenocarcinoma": "Adénocarcinome",
  "Squamous cell carcinoma": "Carcinome épidermoïde",
  "Large cell carcinoma": "Carcinome à grandes cellules",
  "Small cell carcinoma": "Carcinome à petites cellules",
  "NSCLC": "CBNPC",
  "SCLC": "CBPC",
  "Good": "Bonne",
  "Limited material": "Matériel limité",
  "Necrotic": "Nécrotique",
  "Insufficient for molecular testing": "Insuffisant pour test moléculaire",
  "Not requested": "Non demandé",
  "Requested": "Demandé",
  "Completed": "Terminé",
  "Insufficient material": "Matériel insuffisant",
  "Never smoker": "Non-fumeur",
  "Former smoker": "Ancien fumeur",
  "Current smoker": "Fumeur actuel",
  "Well differentiated": "Bien différencié",
  "Moderately differentiated": "Moyennement différencié",
  "Poorly differentiated": "Peu différencié",
  "Low": "Faible",
  "High": "Élevé"
};

function be(v) { return mapToBackend[v] || v; }
function fr(v) { return mapToFrench[v] || v || ""; }

function value(id) {
  return document.getElementById(id).value;
}

function setValue(id, v) {
  const el = document.getElementById(id);
  if (!el) return;
  el.value = fr(v);
}

async function apiFetch(url, options = {}) {
  options.headers = options.headers || {};
  options.headers["Content-Type"] = "application/json";
  if (TOKEN) options.headers["Authorization"] = "Bearer " + TOKEN;

  const res = await fetch(url, options);

  if (res.status === 401) {
    localStorage.removeItem("hospital_token");
    TOKEN = "";
    showLogin();
    throw new Error("Session expirée. Reconnectez-vous.");
  }

  if (!res.ok) {
    throw new Error(await res.text());
  }

  return await res.json();
}

async function login() {
  try {
    const data = await apiFetch("/api/auth/login", {
      method: "POST",
      body: JSON.stringify({
        username: value("loginUsername"),
        password: value("loginPassword")
      })
    });

    TOKEN = data.token;
    CURRENT_USER = data.user;
    localStorage.setItem("hospital_token", TOKEN);

    showApp();
    await refreshCases();
  } catch(e) {
    document.getElementById("loginError").textContent = e.message;
  }
}

async function logout() {
  try {
    await apiFetch("/api/auth/logout", {method: "POST", body: "{}"});
  } catch(e) {}

  localStorage.removeItem("hospital_token");
  TOKEN = "";
  CURRENT_USER = null;
  showLogin();
}

function showLogin() {
  document.getElementById("loginPage").style.display = "flex";
  document.getElementById("app").style.display = "none";
}

function showApp() {
  document.getElementById("loginPage").style.display = "none";
  document.getElementById("app").style.display = "block";
  document.getElementById("userBadge").textContent =
    CURRENT_USER.username + " — " + CURRENT_USER.role;
}

async function checkSession() {
  if (!TOKEN) {
    showLogin();
    return;
  }

  try {
    CURRENT_USER = await apiFetch("/api/auth/me");
    showApp();
    await refreshCases();
  } catch(e) {
    showLogin();
  }
}

function showSection(id, btn) {
  document.querySelectorAll(".section").forEach(s => s.classList.remove("active"));
  document.getElementById(id).classList.add("active");

  document.querySelectorAll(".nav-btn").forEach(b => b.classList.remove("active"));
  if (btn) btn.classList.add("active");

  const titles = {
    dashboard: "Tableau de bord",
    newcase: "Nouveau dossier",
    cases: "Dossiers enregistrés",
    prediction: "Prédiction & traitement",
    report: "Compte rendu",
    exports: "Exports",
    audit: "Journal d'audit"
  };

  document.getElementById("pageTitle").textContent = titles[id] || "Système";
}

function buildPayload() {
  return {
    patient: {
      patient_id: value("patient_id"),
      age: Number(value("age")),
      sex: be(value("sex")),
      region: value("region"),
      smoking_status: be(value("smoking_status")),
      ecog: Number(value("ecog"))
    },
    sample: {
      sample_id: value("sample_id"),
      sample_type: be(value("sample_type")),
      anatomical_site: value("anatomical_site"),
      reception_date: value("reception_date"),
      specimen_quality: be(value("specimen_quality")),
      pathologist_comment: value("pathologist_comment")
    },
    pathology: {
      cancer_type: be(value("cancer_type")),
      histology: be(value("histology")),
      grade: be(value("grade")),
      stage: value("stage"),
      tnm: value("tnm"),
      metastasis: value("metastasis")
    },
    molecular: {
      egfr: be(value("egfr")),
      alk: be(value("alk")),
      ros1: be(value("ros1")),
      pdl1: be(value("pdl1")),
      ihc_markers: value("ihc_markers"),
      molecular_test_status: be(value("molecular_test_status"))
    },
    workflow: {
      status: value("workflow_status"),
      language: "fr",
      hospital_context: "Tunisie - Service d'anatomopathologie"
    }
  };
}

function renderPrediction(result) {
  CURRENT_RESULT = result;

  const p = result.prediction;
  const eff = p.predicted_efficiency || 0;

  const bar = document.getElementById("effBar");
  bar.style.width = eff + "%";
  bar.textContent = eff + "%";

  document.getElementById("predResponse").textContent = translatePrediction(p.predicted_response);
  document.getElementById("resRisk").textContent = p.resistance_risk + "%";
  document.getElementById("confidence").textContent = p.confidence + "%";
  document.getElementById("recommended").textContent = p.recommended_treatment;
  document.getElementById("tnStrategy").textContent = p.tunisia_strategy;
  document.getElementById("drugDesign").textContent = p.drug_design_research;

  const reasons = (p.reasons || []).map(x => `<span class="pill green">${x}</span>`).join("");
  const missing = (p.missing_data || []).map(x => `<span class="pill orange">Donnée manquante : ${x}</span>`).join("");

  document.getElementById("reasonPills").innerHTML = reasons + missing;
  document.getElementById("jsonPreview").textContent = JSON.stringify(result, null, 2);

  document.getElementById("currentPatientLabel").textContent =
    result.payload.patient.patient_id || "Aucun";
  document.getElementById("currentStatusLabel").textContent =
    result.payload.workflow?.status || "Brouillon";
  document.getElementById("currentPredictionLabel").textContent =
    eff + "% — " + translatePrediction(p.predicted_response);

  generateReport();
}

function translatePrediction(x) {
  if (x === "High predicted benefit") return "Bénéfice élevé prédit";
  if (x === "Intermediate predicted benefit") return "Bénéfice intermédiaire prédit";
  if (x === "Low predicted benefit") return "Bénéfice faible prédit";
  return x || "--";
}

async function predictLive() {
  const payload = buildPayload();

  if (!payload.patient.patient_id) {
    alert("Veuillez saisir un ID patient anonymisé.");
    return;
  }

  const result = await apiFetch("/api/predict", {
    method: "POST",
    body: JSON.stringify(payload)
  });

  renderPrediction(result);
  showSection("prediction", document.querySelectorAll(".nav-btn")[3]);
}

async function saveCase() {
  const payload = buildPayload();

  if (!payload.patient.patient_id) {
    alert("ID patient anonymisé obligatoire.");
    return;
  }

  const result = await apiFetch("/api/cases", {
    method: "POST",
    body: JSON.stringify(payload)
  });

  renderPrediction({payload, prediction: result.prediction});
  await refreshCases();

  alert("Dossier enregistré dans la base SQLite.");
}

async function seedDemo() {
  try {
    await apiFetch("/api/seed-demo", {method: "POST", body: "{}"});
    await refreshCases();
    alert("Cas démo ajoutés.");
  } catch(e) {
    alert(e.message);
  }
}

async function refreshCases() {
  ALL_CASES = await apiFetch("/api/cases");
  renderCasesTable();
  await refreshStats();
}

function renderCasesTable() {
  const search = document.getElementById("searchInput")?.value?.toLowerCase() || "";
  const stageFilter = document.getElementById("stageFilter")?.value || "";
  const egfrFilter = document.getElementById("egfrFilter")?.value || "";

  let cases = ALL_CASES.filter(c => {
    const p = c.payload;
    const allText = JSON.stringify(c).toLowerCase();

    const matchSearch = !search || allText.includes(search);
    const matchStage = !stageFilter || p.pathology?.stage === stageFilter;
    const matchEgfr = !egfrFilter || fr(p.molecular?.egfr) === egfrFilter;

    return matchSearch && matchStage && matchEgfr;
  });

  const body = document.getElementById("casesBody");

  body.innerHTML = cases.map(c => {
    const p = c.payload;
    const pred = c.prediction;
    const status = p.workflow?.status || "Brouillon";
    let statusClass = "status-draft";
    if (status.includes("cours") || status.includes("attente")) statusClass = "status-progress";
    if (status.includes("Validé")) statusClass = "status-validated";

    return `
      <tr>
        <td>${c.id}</td>
        <td><b>${c.patient_id}</b><br><span class="small">${fr(p.patient?.sex)}, ${p.patient?.age || ""} ans</span></td>
        <td><span class="${statusClass}">${status}</span></td>
        <td>${fr(p.pathology?.histology)}</td>
        <td>${p.pathology?.stage || ""}</td>
        <td>${fr(p.molecular?.egfr)}</td>
        <td>${pred.predicted_efficiency}%<br>${translatePrediction(pred.predicted_response)}</td>
        <td>${c.updated_by || ""}</td>
        <td>
          <button class="btn-secondary" onclick="loadCase(${c.id})">Charger</button>
          <button class="btn-purple" onclick="openFHIR(${c.id})">FHIR</button>
          <button class="btn-danger" onclick="deleteCase(${c.id})">Supprimer</button>
        </td>
      </tr>
    `;
  }).join("");
}

async function refreshStats() {
  const s = await apiFetch("/api/stats");
  document.getElementById("statTotal").textContent = s.total_cases;
  document.getElementById("statEgfr").textContent = s.egfr_mutated_cases;
  document.getElementById("statStage").textContent = s.stage_iv_cases;
  document.getElementById("statHigh").textContent = s.high_predicted_benefit_cases;
}

async function loadCase(id) {
  const c = await apiFetch(`/api/cases/${id}`);
  CURRENT_CASE_ID = id;

  const p = c.payload;

  setValue("patient_id", p.patient.patient_id);
  setValue("age", p.patient.age);
  setValue("sex", p.patient.sex);
  setValue("region", p.patient.region);
  setValue("smoking_status", p.patient.smoking_status);
  setValue("ecog", p.patient.ecog);

  setValue("sample_id", p.sample.sample_id);
  setValue("sample_type", p.sample.sample_type);
  setValue("anatomical_site", p.sample.anatomical_site);
  setValue("reception_date", p.sample.reception_date);
  setValue("specimen_quality", p.sample.specimen_quality);
  setValue("pathologist_comment", p.sample.pathologist_comment);

  setValue("cancer_type", p.pathology.cancer_type);
  setValue("histology", p.pathology.histology);
  setValue("grade", p.pathology.grade);
  setValue("stage", p.pathology.stage);
  setValue("tnm", p.pathology.tnm);
  setValue("metastasis", p.pathology.metastasis);

  setValue("egfr", p.molecular.egfr);
  setValue("alk", p.molecular.alk);
  setValue("ros1", p.molecular.ros1);
  setValue("pdl1", p.molecular.pdl1);
  setValue("ihc_markers", p.molecular.ihc_markers);
  setValue("molecular_test_status", p.molecular.molecular_test_status);

  setValue("workflow_status", p.workflow?.status || "Brouillon");

  renderPrediction({payload: p, prediction: c.prediction});
  showSection("newcase", document.querySelectorAll(".nav-btn")[1]);
}

async function deleteCase(id) {
  if (!confirm("Supprimer ce dossier ? Rôle admin requis.")) return;

  try {
    await apiFetch(`/api/cases/${id}`, {method: "DELETE"});
    await refreshCases();
  } catch(e) {
    alert(e.message);
  }
}

function generateReport() {
  const payload = CURRENT_RESULT?.payload || buildPayload();
  const pred = CURRENT_RESULT?.prediction;

  if (!payload.patient.patient_id) {
    document.getElementById("reportBox").innerHTML = "Aucun dossier chargé.";
    return;
  }

  const report = `
    <h3>Compte rendu synthétique — Digital Twin Anapath</h3>
    <p><b>ID patient anonymisé :</b> ${payload.patient.patient_id}</p>
    <p><b>Prélèvement :</b> ${payload.sample.sample_id || "--"} — ${fr(payload.sample.sample_type)}</p>
    <p><b>Diagnostic :</b> ${fr(payload.pathology.histology)} — ${fr(payload.pathology.cancer_type)} — Stade ${payload.pathology.stage}</p>
    <p><b>TNM :</b> ${payload.pathology.tnm || "--"} | <b>Métastases :</b> ${payload.pathology.metastasis || "--"}</p>
    <p><b>IHC :</b> ${payload.molecular.ihc_markers || "--"}</p>
    <p><b>Profil moléculaire :</b> EGFR ${fr(payload.molecular.egfr)}, ALK ${fr(payload.molecular.alk)}, ROS1 ${fr(payload.molecular.ros1)}, PD-L1 ${fr(payload.molecular.pdl1)}</p>
    <p><b>Conclusion anapath :</b> ${payload.sample.pathologist_comment || "--"}</p>
    <hr>
    <p><b>Prédiction Digital Twin :</b> ${pred ? translatePrediction(pred.predicted_response) : "--"}</p>
    <p><b>Efficacité prédite :</b> ${pred ? pred.predicted_efficiency + "%" : "--"}</p>
    <p><b>Risque de résistance :</b> ${pred ? pred.resistance_risk + "%" : "--"}</p>
    <p><b>Traitement recommandé :</b> ${pred ? pred.recommended_treatment : "--"}</p>
    <p><b>Stratégie adaptée à la Tunisie :</b> ${pred ? pred.tunisia_strategy : "--"}</p>
    <p><b>Module drug design :</b> ${pred ? pred.drug_design_research : "--"}</p>
    <p class="small"><b>Note :</b> Ce système est un prototype de support décisionnel. Il ne remplace pas la décision médicale multidisciplinaire.</p>
  `;

  document.getElementById("reportBox").innerHTML = report;
}

function exportCurrentCase() {
  const payload = CURRENT_RESULT || {payload: buildPayload()};
  const dataStr = "data:text/json;charset=utf-8," + encodeURIComponent(JSON.stringify(payload, null, 2));
  const a = document.createElement("a");
  a.href = dataStr;
  a.download = (payload.payload?.patient?.patient_id || "dossier") + "_digital_twin.json";
  a.click();
}

async function exportDataset() {
  try {
    const data = await apiFetch("/api/export/cases");
    const dataStr = "data:text/json;charset=utf-8," + encodeURIComponent(JSON.stringify(data, null, 2));
    const a = document.createElement("a");
    a.href = dataStr;
    a.download = "dataset_anapath_digital_twin.json";
    a.click();
  } catch(e) {
    alert(e.message);
  }
}

async function copyCurrentJSON() {
  const text = JSON.stringify(CURRENT_RESULT || {payload: buildPayload()}, null, 2);
  await navigator.clipboard.writeText(text);
  alert("JSON copié.");
}

function openFHIR(id) {
  alert("Pour l'export FHIR avec authentification, utilisez Swagger /docs ou l'export dataset. Le backend possède déjà /api/export/fhir/{id}.");
}

async function loadAudit() {
  try {
    const audit = await apiFetch("/api/audit");
    document.getElementById("auditBox").textContent = JSON.stringify(audit, null, 2);
  } catch(e) {
    document.getElementById("auditBox").textContent = e.message;
  }
}

function openApiDocs() {
  window.open("/docs", "_blank");
}

document.querySelectorAll("input, select, textarea").forEach(el => {
  el.addEventListener("change", () => {
    const pid = document.getElementById("patient_id")?.value;
    if (!pid || !TOKEN) return;

    clearTimeout(window.__predictTimer);
    window.__predictTimer = setTimeout(() => {
      apiFetch("/api/predict", {
        method: "POST",
        body: JSON.stringify(buildPayload())
      })
      .then(renderPrediction)
      .catch(() => {});
    }, 600);
  });
});

checkSession();
</script>

</body>
</html>
'''

(FRONTEND_DIR / "index.html").write_text(index_html, encoding="utf-8")
print("Interface française professionnelle créée :", FRONTEND_DIR / "index.html")

Interface française professionnelle créée : /content/drive/MyDrive/pfe_hospital_anapath_real_system/frontend/index.html


In [ ]:
import subprocess, time
from google.colab import output

PROJECT_DIR = "/content/drive/MyDrive/pfe_hospital_anapath_real_system"

!pkill -f "uvicorn" || true

server = subprocess.Popen(
    ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=PROJECT_DIR
)

time.sleep(4)

print("Système Anapath Digital Twin v0.3 lancé.")
output.serve_kernel_port_as_window(8000)

^C
Système Anapath Digital Twin v0.3 lancé.
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [ ]:
from pathlib import Path
from datetime import datetime
import shutil

PROJECT_DIR = Path("/content/drive/MyDrive/pfe_hospital_anapath_real_system")
APP_PATH = PROJECT_DIR / "app.py"

if not APP_PATH.exists():
    raise Exception("app.py introuvable. Vérifie PROJECT_DIR.")

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_path = PROJECT_DIR / f"app.py.backup_fr_{timestamp}"
shutil.copy2(APP_PATH, backup_path)
print("Backup backend créé :", backup_path)

txt = APP_PATH.read_text(encoding="utf-8")

replacements = {
    # Réponses prédictives
    "High predicted benefit": "Bénéfice élevé prédit",
    "Intermediate predicted benefit": "Bénéfice intermédiaire prédit",
    "Low predicted benefit": "Bénéfice faible prédit",

    # Recommandations
    "Osimertinib": "Osimertinib",
    "Advanced EGFR resistance strategy": "Stratégie avancée pour résistance EGFR",
    "Specific EGFR exon 20 strategy / local systemic therapy": "Stratégie spécifique EGFR exon 20 / traitement systémique local",
    "Immunotherapy + chemotherapy if accessible": "Immunothérapie + chimiothérapie si accessible",
    "Complete molecular testing first": "Compléter d’abord les tests moléculaires",

    # Raisons biologiques
    "EGFR sensitizing mutation detected": "Mutation EGFR sensibilisante détectée",
    "EGFR resistance mutation detected": "Mutation de résistance EGFR détectée",
    "EGFR exon 20 insertion requires specific strategy": "L’insertion EGFR exon 20 nécessite une stratégie spécifique",
    "EGFR negative profile": "Profil EGFR négatif",
    "Adenocarcinoma supports molecular profiling priority": "L’adénocarcinome justifie une priorité au profilage moléculaire",
    "Metastatic disease increases progression risk": "La maladie métastatique augmente le risque de progression",
    "Reduced ECOG increases treatment risk": "Un score ECOG altéré augmente le risque thérapeutique",
    "High PD-L1 may support immunotherapy strategy if accessible": "Un PD-L1 élevé peut soutenir une stratégie d’immunothérapie si accessible",
    "Insufficient sample limits precision-medicine confidence": "Un prélèvement insuffisant limite la fiabilité de la médecine de précision",

    # Données manquantes
    "EGFR mutation result": "Résultat mutationnel EGFR",
    "PD-L1 result": "Résultat PD-L1",
    "ALK status": "Statut ALK",
    "ROS1 status": "Statut ROS1",
    "Complete molecular testing": "Tests moléculaires complets",

    # Stratégies thérapeutiques
    "Best global option: Osimertinib. Tunisia layer: verify hospital pharmacy access. ":
        "Meilleure option selon les recommandations internationales : Osimertinib. Couche Tunisie : vérifier la disponibilité auprès de la pharmacie hospitalière. ",

    "If unavailable, consider erlotinib/gefitinib/afatinib or platinum-based chemotherapy according to local protocol.":
        "Si indisponible, discuter erlotinib/géfitinib/afatinib ou une chimiothérapie à base de sels de platine selon le protocole local.",

    "Resistance profile: verify access to appropriate EGFR-targeted therapy; request MET amplification/resistance workup if possible.":
        "Profil de résistance : vérifier l’accès à une thérapie ciblée EGFR adaptée ; demander si possible une recherche d’amplification MET et un bilan de résistance.",

    "Rare EGFR alteration: verify targeted therapy availability. If unavailable, use local systemic protocol and consider research referral.":
        "Altération EGFR rare : vérifier la disponibilité d’une thérapie ciblée spécifique. Si indisponible, utiliser le protocole systémique local et envisager une orientation vers la recherche.",

    "EGFR negative with high PD-L1: immunotherapy + chemotherapy may be globally indicated. ":
        "EGFR négatif avec PD-L1 élevé : immunothérapie + chimiothérapie peut être indiquée selon les recommandations internationales. ",

    "Tunisia access must be verified; if unavailable, use chemotherapy-based local protocol.":
        "L’accès en Tunisie doit être vérifié ; si indisponible, utiliser un protocole local basé sur la chimiothérapie.",

    "Incomplete profile: request EGFR, ALK, ROS1 and PD-L1 testing if possible. ":
        "Profil incomplet : demander si possible les tests EGFR, ALK, ROS1 et PD-L1. ",

    "Use pathology, stage and local protocols while waiting.":
        "Utiliser l’histologie, le stade et les protocoles locaux en attendant les résultats.",

    # Drug design
    "Research only: EGFR inhibitor scaffold prioritization using PyMOL, AutoDock Vina, RDKit and ADMET filtering.":
        "Recherche uniquement : priorisation de scaffolds inhibiteurs d’EGFR à l’aide de PyMOL, AutoDock Vina, RDKit et filtrage ADMET.",

    "Research only: prioritize resistant EGFR conformations and possible EGFR/MET combination strategies.":
        "Recherche uniquement : prioriser les conformations résistantes d’EGFR et les stratégies potentielles de combinaison EGFR/MET.",

    "Research only: exon 20 insertion should trigger docking/scaffold screening.":
        "Recherche uniquement : l’insertion exon 20 doit déclencher un criblage par docking et une recherche de scaffolds candidats.",

    "Research only: expand molecular profiling before rational docking.":
        "Recherche uniquement : élargir le profilage moléculaire avant tout docking rationnel.",

    "Research only: insufficient molecular data for rational drug design.":
        "Recherche uniquement : données moléculaires insuffisantes pour un drug design rationnel."
}

for old, new in replacements.items():
    txt = txt.replace(old, new)

APP_PATH.write_text(txt, encoding="utf-8")
print("Backend traduit en français.")

Backup backend créé : /content/drive/MyDrive/pfe_hospital_anapath_real_system/app.py.backup_fr_20260514_151546
Backend traduit en français.


In [ ]:
from pathlib import Path
from datetime import datetime
import shutil
import re

PROJECT_DIR = Path("/content/drive/MyDrive/pfe_hospital_anapath_real_system")
INDEX_PATH = PROJECT_DIR / "frontend" / "index.html"

if not INDEX_PATH.exists():
    raise Exception("index.html introuvable.")

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_path = INDEX_PATH.with_name(f"index.html.backup_fr_{timestamp}")
shutil.copy2(INDEX_PATH, backup_path)
print("Backup frontend créé :", backup_path)

html = INDEX_PATH.read_text(encoding="utf-8")

# Remplacer la fonction translatePrediction par une version plus complète
new_translate_block = r'''
function translatePrediction(x) {
  if (x === "High predicted benefit") return "Bénéfice élevé prédit";
  if (x === "Intermediate predicted benefit") return "Bénéfice intermédiaire prédit";
  if (x === "Low predicted benefit") return "Bénéfice faible prédit";
  return x || "--";
}

function translateTreatment(x) {
  const d = {
    "Advanced EGFR resistance strategy": "Stratégie avancée pour résistance EGFR",
    "Specific EGFR exon 20 strategy / local systemic therapy": "Stratégie spécifique EGFR exon 20 / traitement systémique local",
    "Immunotherapy + chemotherapy if accessible": "Immunothérapie + chimiothérapie si accessible",
    "Complete molecular testing first": "Compléter d’abord les tests moléculaires"
  };
  return d[x] || x || "--";
}

function translateText(x) {
  const d = {
    "EGFR sensitizing mutation detected": "Mutation EGFR sensibilisante détectée",
    "EGFR resistance mutation detected": "Mutation de résistance EGFR détectée",
    "EGFR exon 20 insertion requires specific strategy": "L’insertion EGFR exon 20 nécessite une stratégie spécifique",
    "EGFR negative profile": "Profil EGFR négatif",
    "Adenocarcinoma supports molecular profiling priority": "L’adénocarcinome justifie une priorité au profilage moléculaire",
    "Metastatic disease increases progression risk": "La maladie métastatique augmente le risque de progression",
    "Reduced ECOG increases treatment risk": "Un score ECOG altéré augmente le risque thérapeutique",
    "High PD-L1 may support immunotherapy strategy if accessible": "Un PD-L1 élevé peut soutenir une stratégie d’immunothérapie si accessible",
    "Insufficient sample limits precision-medicine confidence": "Un prélèvement insuffisant limite la fiabilité de la médecine de précision",

    "Best global option: Osimertinib. Tunisia layer: verify hospital pharmacy access. If unavailable, consider erlotinib/gefitinib/afatinib or platinum-based chemotherapy according to local protocol.":
      "Meilleure option selon les recommandations internationales : Osimertinib. Couche Tunisie : vérifier la disponibilité auprès de la pharmacie hospitalière. Si indisponible, discuter erlotinib/géfitinib/afatinib ou une chimiothérapie à base de sels de platine selon le protocole local.",

    "EGFR negative with high PD-L1: immunotherapy + chemotherapy may be globally indicated. Tunisia access must be verified; if unavailable, use chemotherapy-based local protocol.":
      "EGFR négatif avec PD-L1 élevé : immunothérapie + chimiothérapie peut être indiquée selon les recommandations internationales. L’accès en Tunisie doit être vérifié ; si indisponible, utiliser un protocole local basé sur la chimiothérapie.",

    "Research only: expand molecular profiling before rational docking.":
      "Recherche uniquement : élargir le profilage moléculaire avant tout docking rationnel.",

    "Research only: insufficient molecular data for rational drug design.":
      "Recherche uniquement : données moléculaires insuffisantes pour un drug design rationnel."
  };
  return d[x] || x || "--";
}

function translateMissing(x) {
  const d = {
    "EGFR mutation result": "Résultat mutationnel EGFR",
    "PD-L1 result": "Résultat PD-L1",
    "ALK status": "Statut ALK",
    "ROS1 status": "Statut ROS1",
    "Complete molecular testing": "Tests moléculaires complets"
  };
  return d[x] || x || "--";
}

function roleFr(role) {
  const d = {
    "admin": "Administrateur",
    "pathologist": "Anatomopathologiste",
    "oncologist": "Oncologue",
    "researcher": "Chercheur"
  };
  return d[role] || role;
}

function localizeForDisplay(obj) {
  if (Array.isArray(obj)) return obj.map(localizeForDisplay);
  if (obj && typeof obj === "object") {
    const out = {};
    Object.keys(obj).forEach(k => {
      out[k] = localizeForDisplay(obj[k]);
    });
    return out;
  }
  if (typeof obj === "string") {
    return translateTreatment(translateText(translatePrediction(obj)));
  }
  return obj;
}
'''

html = re.sub(
    r'function translatePrediction\(x\) \{.*?return x \|\| "--";\s*\}',
    new_translate_block,
    html,
    flags=re.DOTALL
)

# Corriger les affichages principaux
html = html.replace(
    'document.getElementById("recommended").textContent = p.recommended_treatment;',
    'document.getElementById("recommended").textContent = translateTreatment(p.recommended_treatment);'
)

html = html.replace(
    'document.getElementById("tnStrategy").textContent = p.tunisia_strategy;',
    'document.getElementById("tnStrategy").textContent = translateText(p.tunisia_strategy);'
)

html = html.replace(
    'document.getElementById("drugDesign").textContent = p.drug_design_research;',
    'document.getElementById("drugDesign").textContent = translateText(p.drug_design_research);'
)

html = html.replace(
    'const reasons = (p.reasons || []).map(x => `<span class="pill green">${x}</span>`).join("");',
    'const reasons = (p.reasons || []).map(x => `<span class="pill green">${translateText(x)}</span>`).join("");'
)

html = html.replace(
    'const missing = (p.missing_data || []).map(x => `<span class="pill orange">Donnée manquante : ${x}</span>`).join("");',
    'const missing = (p.missing_data || []).map(x => `<span class="pill orange">Donnée manquante : ${translateMissing(x)}</span>`).join("");'
)

html = html.replace(
    'document.getElementById("jsonPreview").textContent = JSON.stringify(result, null, 2);',
    'document.getElementById("jsonPreview").textContent = JSON.stringify(localizeForDisplay(result), null, 2);'
)

html = html.replace(
    'CURRENT_USER.username + " — " + CURRENT_USER.role;',
    'CURRENT_USER.username + " — " + roleFr(CURRENT_USER.role);'
)

# Corriger le compte rendu généré
html = html.replace(
    '${pred ? pred.recommended_treatment : "--"}',
    '${pred ? translateTreatment(pred.recommended_treatment) : "--"}'
)

html = html.replace(
    '${pred ? pred.tunisia_strategy : "--"}',
    '${pred ? translateText(pred.tunisia_strategy) : "--"}'
)

html = html.replace(
    '${pred ? pred.drug_design_research : "--"}',
    '${pred ? translateText(pred.drug_design_research) : "--"}'
)

INDEX_PATH.write_text(html, encoding="utf-8")
print("Frontend corrigé : affichage 100% français.")

Backup frontend créé : /content/drive/MyDrive/pfe_hospital_anapath_real_system/frontend/index.html.backup_fr_20260514_151615
Frontend corrigé : affichage 100% français.


In [ ]:
import subprocess, time
from google.colab import output

PROJECT_DIR = "/content/drive/MyDrive/pfe_hospital_anapath_real_system"

!pkill -f "uvicorn" || true

server = subprocess.Popen(
    ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=PROJECT_DIR
)

time.sleep(4)

print("Système relancé en français.")
output.serve_kernel_port_as_window(8000)

^C
Système relancé en français.
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, textwrap, json, shutil
from datetime import datetime

PROJECT_DIR = Path("/content/drive/MyDrive/pfe_hospital_anapath_real_system")
DEPLOY_DIR = PROJECT_DIR / "deployment"
DEPLOY_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Deployment folder:", DEPLOY_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project: /content/drive/MyDrive/pfe_hospital_anapath_real_system
Deployment folder: /content/drive/MyDrive/pfe_hospital_anapath_real_system/deployment


In [ ]:
requirements = """
fastapi==0.115.6
uvicorn[standard]==0.34.0
python-multipart==0.0.20
"""

(PROJECT_DIR / "requirements.txt").write_text(requirements.strip(), encoding="utf-8")

print("requirements.txt created")

requirements.txt created


In [ ]:
run_windows = r'''
@echo off
title AnapathTwin Hospital System

echo ==========================================
echo   AnapathTwin Hospital System
echo   Systeme hospitalier local - Anapath
echo ==========================================

cd /d "%~dp0"

if not exist ".venv" (
    echo Creation environnement Python...
    python -m venv .venv
)

call .venv\Scripts\activate

echo Installation / verification dependances...
pip install -r requirements.txt

echo Lancement du serveur hospitalier local...
echo.
echo Ouvrir dans le navigateur:
echo http://127.0.0.1:8000
echo.
echo Pour acces reseau local:
echo http://ADRESSE_IP_DU_PC:8000
echo.

uvicorn app:app --host 0.0.0.0 --port 8000

pause
'''

(PROJECT_DIR / "START_ANAPATHTWIN_WINDOWS.bat").write_text(run_windows, encoding="utf-8")

print("Windows launcher created")

Windows launcher created


In [ ]:
run_linux = r'''
#!/bin/bash

echo "=========================================="
echo "  AnapathTwin Hospital System"
echo "  Systeme hospitalier local - Anapath"
echo "=========================================="

cd "$(dirname "$0")"

if [ ! -d ".venv" ]; then
    echo "Creation environnement Python..."
    python3 -m venv .venv
fi

source .venv/bin/activate

echo "Installation / verification dependances..."
pip install -r requirements.txt

echo "Lancement du serveur hospitalier local..."
echo "Ouvrir: http://127.0.0.1:8000"

uvicorn app:app --host 0.0.0.0 --port 8000
'''

linux_path = PROJECT_DIR / "START_ANAPATHTWIN_LINUX.sh"
linux_path.write_text(run_linux, encoding="utf-8")

print("Linux launcher created")

Linux launcher created


In [ ]:
backup_script = r'''
from pathlib import Path
from datetime import datetime
import shutil
import os

PROJECT_DIR = Path(__file__).resolve().parent
DB_PATH = PROJECT_DIR / "data" / "hospital_twin.db"
BACKUP_DIR = PROJECT_DIR / "backups"

BACKUP_DIR.mkdir(exist_ok=True)

if not DB_PATH.exists():
    print("Database not found:", DB_PATH)
    raise SystemExit(1)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_path = BACKUP_DIR / f"hospital_twin_backup_{timestamp}.db"

shutil.copy2(DB_PATH, backup_path)

print("Backup created:", backup_path)

# Keep only last 30 backups
backups = sorted(BACKUP_DIR.glob("hospital_twin_backup_*.db"))
if len(backups) > 30:
    for old in backups[:-30]:
        old.unlink()
        print("Deleted old backup:", old)
'''

(PROJECT_DIR / "backup_database.py").write_text(backup_script, encoding="utf-8")

print("Backup script created")

Backup script created


In [ ]:
readme = r'''
# AnapathTwin Hospital System

## Objectif

AnapathTwin est un prototype hospitalier pour le service d'anatomopathologie.
Il permet de gérer des dossiers anonymisés de cancer du poumon, intégrer les données anapath, IHC et moléculaires, puis générer une prédiction Digital Twin et une stratégie thérapeutique adaptée au contexte tunisien.

## Mode de déploiement recommandé pour le pilote

Déploiement local dans le réseau interne de l'hôpital.

Ne pas utiliser ngrok, Colab ou un cloud public pour des données réelles de patients.

## Lancement sur Windows

1. Copier le dossier complet sur le PC hospitalier.
2. Double-cliquer sur:

START_ANAPATHTWIN_WINDOWS.bat

3. Ouvrir dans Chrome:

http://127.0.0.1:8000

Pour accès depuis d'autres postes du service:

http://ADRESSE_IP_DU_PC:8000

## Comptes de démonstration

admin / admin123
pathologist / patho123
oncologist / onco123
researcher / research123

Ces mots de passe doivent être changés avant tout pilote réel.

## Règles importantes

- Utiliser uniquement des IDs anonymisés.
- Ne pas saisir de nom, CIN, téléphone ou adresse.
- Obtenir l'accord du service et de l'administration avant utilisation réelle.
- Prévoir validation médicale et conformité protection des données.

## Sauvegarde

Exécuter régulièrement:

python backup_database.py

La base est stockée dans:

data/hospital_twin.db

Les sauvegardes sont stockées dans:

backups/

## Roadmap

v0.4 : système local stable
v0.5 : import Excel/CSV des archives anapath
v0.6 : comptes utilisateurs configurables
v0.7 : PostgreSQL
v0.8 : déploiement intranet hospitalier
v1.0 : validation clinique rétrospective
'''

(PROJECT_DIR / "README_DEPLOIEMENT_HOPITAL.md").write_text(readme, encoding="utf-8")

print("README created")

README created


In [ ]:
import shutil
from pathlib import Path
from datetime import datetime

PROJECT_DIR = Path("/content/drive/MyDrive/pfe_hospital_anapath_real_system")
EXPORT_DIR = Path("/content/drive/MyDrive/AnapathTwin_Exports")
EXPORT_DIR.mkdir(exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
zip_base = EXPORT_DIR / f"AnapathTwin_Hospital_System_v0_4_{timestamp}"

# Remove useless backups from package? Keep project as is for now.
shutil.make_archive(str(zip_base), "zip", PROJECT_DIR)

print("ZIP created:")
print(str(zip_base) + ".zip")

ZIP created:
/content/drive/MyDrive/AnapathTwin_Exports/AnapathTwin_Hospital_System_v0_4_20260514_1526.zip


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime
import shutil, os

PROJECT_DIR = Path("/content/drive/MyDrive/pfe_hospital_anapath_real_system")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)

# ----------------------------
# 1. Professional app launcher
# ----------------------------

start_app = r'''
@echo off
setlocal

title AnapathTwin Hospital App
cd /d "%~dp0"

if not exist logs mkdir logs

echo ==========================================
echo   AnapathTwin Hospital App
echo   Application hospitaliere locale
echo ==========================================
echo.

echo Verification du serveur...

powershell -NoProfile -ExecutionPolicy Bypass -Command "try { Invoke-WebRequest -UseBasicParsing 'http://127.0.0.1:8000/api/health' -TimeoutSec 1 | Out-Null; exit 0 } catch { exit 1 }" >nul 2>nul

if not errorlevel 1 goto OPEN_APP

echo Serveur non actif. Demarrage...

if not exist ".venv\Scripts\python.exe" (
    echo Creation environnement Python...
    python -m venv .venv
)

if not exist ".venv\Scripts\python.exe" (
    echo ERREUR: Python est introuvable ou l'environnement n'a pas pu etre cree.
    echo Installez Python puis relancez.
    pause
    exit /b
)

echo Installation / verification des dependances...
".venv\Scripts\python.exe" -m pip install -r requirements.txt > logs\install.log 2>&1

if errorlevel 1 (
    echo ERREUR pendant l'installation des dependances.
    echo Voir logs\install.log
    pause
    exit /b
)

echo Lancement du serveur en arriere-plan...
start "AnapathTwin Server" /min cmd /c ""%CD%\.venv\Scripts\python.exe" -m uvicorn app:app --host 127.0.0.1 --port 8000 > "%CD%\logs\server.log" 2>&1"

echo Attente du serveur...
for /l %%i in (1,1,30) do (
    powershell -NoProfile -ExecutionPolicy Bypass -Command "try { Invoke-WebRequest -UseBasicParsing 'http://127.0.0.1:8000/api/health' -TimeoutSec 1 | Out-Null; exit 0 } catch { exit 1 }" >nul 2>nul
    if not errorlevel 1 goto OPEN_APP
    timeout /t 1 >nul
)

echo ERREUR: Le serveur n'a pas demarre.
echo Voir logs\server.log
pause
exit /b

:OPEN_APP
echo Serveur actif.
echo Ouverture de l'application...

set "APP_URL=http://127.0.0.1:8000"
set "BROWSER="

if exist "%ProgramFiles(x86)%\Microsoft\Edge\Application\msedge.exe" set "BROWSER=%ProgramFiles(x86)%\Microsoft\Edge\Application\msedge.exe"
if exist "%ProgramFiles%\Microsoft\Edge\Application\msedge.exe" set "BROWSER=%ProgramFiles%\Microsoft\Edge\Application\msedge.exe"
if exist "%ProgramFiles%\Google\Chrome\Application\chrome.exe" set "BROWSER=%ProgramFiles%\Google\Chrome\Application\chrome.exe"
if exist "%ProgramFiles(x86)%\Google\Chrome\Application\chrome.exe" set "BROWSER=%ProgramFiles(x86)%\Google\Chrome\Application\chrome.exe"

if defined BROWSER (
    start "" "%BROWSER%" --app="%APP_URL%" --window-size=1450,900
) else (
    start "" "%APP_URL%"
)

echo.
echo Application ouverte.
echo Ne supprimez pas ce dossier: il contient la base de donnees locale.
echo.
exit /b
'''

(PROJECT_DIR / "ANAPATHTWIN_APP_START.bat").write_text(start_app, encoding="utf-8")


# ----------------------------
# 2. Stop script
# ----------------------------

stop_app = r'''
@echo off
title Stop AnapathTwin Hospital App

echo Arret du serveur AnapathTwin sur le port 8000...

for /f "tokens=5" %%a in ('netstat -aon ^| findstr :8000 ^| findstr LISTENING') do (
    taskkill /F /PID %%a
)

echo Serveur arrete.
pause
'''

(PROJECT_DIR / "ANAPATHTWIN_APP_STOP.bat").write_text(stop_app, encoding="utf-8")


# ----------------------------
# 3. Desktop shortcut creator
# ----------------------------

shortcut_script = r'''
@echo off
cd /d "%~dp0"

echo Creation du raccourci bureau...

powershell -NoProfile -ExecutionPolicy Bypass -Command "$WshShell = New-Object -ComObject WScript.Shell; $Shortcut = $WshShell.CreateShortcut([Environment]::GetFolderPath('Desktop') + '\AnapathTwin Hospital App.lnk'); $Shortcut.TargetPath = '%~dp0ANAPATHTWIN_APP_START.bat'; $Shortcut.WorkingDirectory = '%~dp0'; $Shortcut.IconLocation = $env:SystemRoot + '\System32\shell32.dll,167'; $Shortcut.Save()"

echo Raccourci cree sur le bureau:
echo AnapathTwin Hospital App
pause
'''

(PROJECT_DIR / "INSTALL_DESKTOP_SHORTCUT.bat").write_text(shortcut_script, encoding="utf-8")


# ----------------------------
# 4. README app
# ----------------------------

readme = r'''
ANAPATHTWIN HOSPITAL APP

Mode app local pour Windows.

FICHIERS IMPORTANTS

ANAPATHTWIN_APP_START.bat
- Lance le serveur FastAPI automatiquement
- Ouvre l'interface comme une application desktop

ANAPATHTWIN_APP_STOP.bat
- Arrete le serveur local

INSTALL_DESKTOP_SHORTCUT.bat
- Cree un raccourci sur le bureau

UTILISATION

1. Double-cliquer sur INSTALL_DESKTOP_SHORTCUT.bat une seule fois.
2. Lancer l'application depuis le bureau:
   AnapathTwin Hospital App
3. Login:
   admin / admin123

IMPORTANT

- Ne pas saisir de vraies identites patients.
- Utiliser uniquement des IDs anonymises.
- Ne pas exposer cette version sur internet.
- Pour un pilote hopital: deploiement local/intranet uniquement.

DONNEES

La base locale est:
data/hospital_twin.db

Les logs sont:
logs/server.log
logs/install.log
'''

(PROJECT_DIR / "README_APP_WINDOWS.txt").write_text(readme, encoding="utf-8")


# ----------------------------
# 5. Zip export
# ----------------------------

EXPORT_DIR = Path("/content/drive/MyDrive/AnapathTwin_Exports")
EXPORT_DIR.mkdir(exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
zip_base = EXPORT_DIR / f"AnapathTwin_Hospital_App_v0_5_{timestamp}"

shutil.make_archive(str(zip_base), "zip", PROJECT_DIR)

print("App files created successfully.")
print("ZIP created:")
print(str(zip_base) + ".zip")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project: /content/drive/MyDrive/pfe_hospital_anapath_real_system
App files created successfully.
ZIP created:
/content/drive/MyDrive/AnapathTwin_Exports/AnapathTwin_Hospital_App_v0_5_20260514_1603.zip


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime
import shutil

PROJECT_DIR = Path("/content/drive/MyDrive/pfe_hospital_anapath_real_system")
EXPORT_DIR = Path("/content/drive/MyDrive/AnapathTwin_Exports")

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)

# --------------------------------------------------
# 1. Hidden app launcher
# --------------------------------------------------

hidden_vbs = r'''
Set fso = CreateObject("Scripting.FileSystemObject")
base = fso.GetParentFolderName(WScript.ScriptFullName)

Set shell = CreateObject("WScript.Shell")
cmd = "cmd /c """ & base & "\ANAPATHTWIN_APP_START_BACKGROUND.bat"""
shell.Run cmd, 0, False
'''

(PROJECT_DIR / "ANAPATHTWIN_APP_LAUNCHER.vbs").write_text(hidden_vbs, encoding="utf-8")


# --------------------------------------------------
# 2. Background launcher
# --------------------------------------------------

background_bat = r'''
@echo off
cd /d "%~dp0"

if not exist logs mkdir logs

echo [%date% %time%] Starting AnapathTwin App >> logs\launcher.log

powershell -NoProfile -ExecutionPolicy Bypass -Command "try { Invoke-WebRequest -UseBasicParsing 'http://127.0.0.1:8000/api/health' -TimeoutSec 1 | Out-Null; exit 0 } catch { exit 1 }" >nul 2>nul

if not errorlevel 1 goto OPEN_APP

if not exist ".venv\Scripts\python.exe" (
    python -m venv .venv >> logs\launcher.log 2>&1
)

if not exist ".venv\Scripts\python.exe" (
    echo Python or virtual environment error >> logs\launcher.log
    start notepad logs\launcher.log
    exit /b
)

".venv\Scripts\python.exe" -m pip install -r requirements.txt >> logs\install.log 2>&1

start "AnapathTwin Server" /min cmd /c ""%CD%\.venv\Scripts\python.exe" -m uvicorn app:app --host 127.0.0.1 --port 8000 > "%CD%\logs\server.log" 2> "%CD%\logs\server_error.log""

for /l %%i in (1,1,30) do (
    powershell -NoProfile -ExecutionPolicy Bypass -Command "try { Invoke-WebRequest -UseBasicParsing 'http://127.0.0.1:8000/api/health' -TimeoutSec 1 | Out-Null; exit 0 } catch { exit 1 }" >nul 2>nul
    if not errorlevel 1 goto OPEN_APP
    timeout /t 1 >nul
)

echo Server failed to start. Check logs\server_error.log >> logs\launcher.log
start notepad logs\server_error.log
exit /b

:OPEN_APP
set "APP_URL=http://127.0.0.1:8000"
set "BROWSER="

if exist "%ProgramFiles(x86)%\Microsoft\Edge\Application\msedge.exe" set "BROWSER=%ProgramFiles(x86)%\Microsoft\Edge\Application\msedge.exe"
if exist "%ProgramFiles%\Microsoft\Edge\Application\msedge.exe" set "BROWSER=%ProgramFiles%\Microsoft\Edge\Application\msedge.exe"
if exist "%ProgramFiles%\Google\Chrome\Application\chrome.exe" set "BROWSER=%ProgramFiles%\Google\Chrome\Application\chrome.exe"
if exist "%ProgramFiles(x86)%\Google\Chrome\Application\chrome.exe" set "BROWSER=%ProgramFiles(x86)%\Google\Chrome\Application\chrome.exe"

if defined BROWSER (
    start "" "%BROWSER%" --app="%APP_URL%" --window-size=1450,900
) else (
    start "" "%APP_URL%"
)

exit /b
'''

(PROJECT_DIR / "ANAPATHTWIN_APP_START_BACKGROUND.bat").write_text(background_bat, encoding="utf-8")


# --------------------------------------------------
# 3. Visible rescue launcher
# --------------------------------------------------

visible_bat = r'''
@echo off
title AnapathTwin Hospital App - Mode visible
cd /d "%~dp0"

if not exist logs mkdir logs

echo ==========================================
echo   AnapathTwin Hospital App
echo   Mode visible / diagnostic
echo ==========================================
echo.

if not exist ".venv\Scripts\python.exe" (
    echo Creation environnement Python...
    python -m venv .venv
)

if not exist ".venv\Scripts\python.exe" (
    echo ERREUR: Python introuvable.
    pause
    exit /b
)

echo Installation / verification dependances...
".venv\Scripts\python.exe" -m pip install -r requirements.txt

echo.
echo Lancement du serveur...
echo Ouvrir:
echo http://127.0.0.1:8000
echo.

start http://127.0.0.1:8000

".venv\Scripts\python.exe" -m uvicorn app:app --host 127.0.0.1 --port 8000

pause
'''

(PROJECT_DIR / "ANAPATHTWIN_APP_START_VISIBLE.bat").write_text(visible_bat, encoding="utf-8")


# --------------------------------------------------
# 4. Stop app script
# --------------------------------------------------

stop_bat = r'''
@echo off
title Stop AnapathTwin Hospital App

echo Arret du serveur AnapathTwin...

for /f "tokens=5" %%a in ('netstat -aon ^| findstr :8000 ^| findstr LISTENING') do (
    taskkill /F /PID %%a
)

echo Serveur arrete.
pause
'''

(PROJECT_DIR / "ANAPATHTWIN_APP_STOP.bat").write_text(stop_bat, encoding="utf-8")


# --------------------------------------------------
# 5. Health check
# --------------------------------------------------

health_bat = r'''
@echo off
title AnapathTwin Health Check

echo Verification du serveur AnapathTwin...
echo.

powershell -NoProfile -ExecutionPolicy Bypass -Command "try { Invoke-WebRequest -UseBasicParsing 'http://127.0.0.1:8000/api/health' -TimeoutSec 3 } catch { Write-Host 'Serveur non joignable'; exit 1 }"

echo.
pause
'''

(PROJECT_DIR / "ANAPATHTWIN_HEALTH_CHECK.bat").write_text(health_bat, encoding="utf-8")


# --------------------------------------------------
# 6. Desktop shortcut installer
# --------------------------------------------------

shortcut_bat = r'''
@echo off
cd /d "%~dp0"

echo Creation du raccourci bureau...

powershell -NoProfile -ExecutionPolicy Bypass -Command "$WshShell = New-Object -ComObject WScript.Shell; $Shortcut = $WshShell.CreateShortcut([Environment]::GetFolderPath('Desktop') + '\AnapathTwin Hospital App.lnk'); $Shortcut.TargetPath = '%~dp0ANAPATHTWIN_APP_LAUNCHER.vbs'; $Shortcut.WorkingDirectory = '%~dp0'; $Shortcut.IconLocation = $env:SystemRoot + '\System32\shell32.dll,167'; $Shortcut.Save()"

echo.
echo Raccourci cree sur le bureau:
echo AnapathTwin Hospital App
echo.
pause
'''

(PROJECT_DIR / "INSTALL_APP_DESKTOP_SHORTCUT.bat").write_text(shortcut_bat, encoding="utf-8")


# --------------------------------------------------
# 7. App README
# --------------------------------------------------

readme = r'''
ANAPATHTWIN HOSPITAL APP

Application hospitaliere locale pour le service d'anatomopathologie.

LANCEMENT NORMAL

1. Double-cliquer sur:
   INSTALL_APP_DESKTOP_SHORTCUT.bat

2. Un raccourci sera cree sur le bureau:
   AnapathTwin Hospital App

3. Double-cliquer sur ce raccourci.
   Le serveur demarre automatiquement.
   L'application s'ouvre comme une application desktop.

COMPTES DEMO

admin / admin123
pathologist / patho123
oncologist / onco123
researcher / research123

ARRET

Double-cliquer sur:
ANAPATHTWIN_APP_STOP.bat

SI L'APPLICATION NE S'OUVRE PAS

Double-cliquer sur:
ANAPATHTWIN_APP_START_VISIBLE.bat

Cela affiche les erreurs.

FICHIERS IMPORTANTS

data/hospital_twin.db
Base de donnees locale SQLite.

logs/server.log
Logs du serveur.

logs/server_error.log
Erreurs serveur.

backup_database.py
Script de sauvegarde.

SECURITE

- Ne pas saisir de vraies identites.
- Utiliser uniquement des IDs anonymises.
- Ne pas exposer cette application sur Internet.
- Pour le pilote hopital: utiliser seulement en intranet/local.
'''

(PROJECT_DIR / "README_APP_HOPITAL.txt").write_text(readme, encoding="utf-8")


# --------------------------------------------------
# 8. Create ZIP package
# --------------------------------------------------

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
zip_base = EXPORT_DIR / f"AnapathTwin_Hospital_App_v0_6_{timestamp}"

shutil.make_archive(str(zip_base), "zip", PROJECT_DIR)

print("Application package created:")
print(str(zip_base) + ".zip")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project: /content/drive/MyDrive/pfe_hospital_anapath_real_system
Application package created:
/content/drive/MyDrive/AnapathTwin_Exports/AnapathTwin_Hospital_App_v0_6_20260514_1633.zip


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime
import shutil
import re

PROJECT_DIR = Path("/content/drive/MyDrive/pfe_hospital_anapath_real_system")
EXPORT_DIR = Path("/content/drive/MyDrive/AnapathTwin_Exports")

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

APP_PATH = PROJECT_DIR / "app.py"
FRONTEND_DIR = PROJECT_DIR / "frontend"
DATA_DIR = PROJECT_DIR / "data"

if not APP_PATH.exists():
    raise Exception("app.py introuvable.")

FRONTEND_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

print("Project:", PROJECT_DIR)

# --------------------------------------------------
# 1. Patch app.py for desktop executable mode
# --------------------------------------------------

txt = APP_PATH.read_text(encoding="utf-8")

backup_path = PROJECT_DIR / f"app.py.backup_desktop_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
shutil.copy2(APP_PATH, backup_path)
print("Backup backend:", backup_path)

# Add sys import if missing
if "import sys" not in txt:
    txt = txt.replace("import time", "import time\nimport sys")

runtime_block = r'''
def get_runtime_dir():
    """
    Folder where database/logs are stored.
    In development: project folder.
    In packaged app: folder containing the .exe.
    """
    if getattr(sys, "frozen", False):
        return Path(sys.executable).resolve().parent
    return Path(__file__).resolve().parent

def get_resource_dir():
    """
    Folder where bundled frontend resources are stored.
    In PyInstaller: sys._MEIPASS.
    In development: project folder.
    """
    if hasattr(sys, "_MEIPASS"):
        return Path(sys._MEIPASS)
    return Path(__file__).resolve().parent

PROJECT_DIR = get_runtime_dir()
RESOURCE_DIR = get_resource_dir()
DB_PATH = PROJECT_DIR / "data" / "hospital_twin.db"
FRONTEND_DIR = RESOURCE_DIR / "frontend"
'''

# Replace old path block if present
txt = re.sub(
    r'PROJECT_DIR\s*=\s*Path\(__file__\)\.resolve\(\)\.parent\s*\nDB_PATH\s*=\s*PROJECT_DIR\s*/\s*"data"\s*/\s*"hospital_twin\.db"\s*\nFRONTEND_DIR\s*=\s*PROJECT_DIR\s*/\s*"frontend"',
    runtime_block.strip(),
    txt
)

# If replacement did not happen, insert before app = FastAPI
if "def get_runtime_dir()" not in txt:
    txt = txt.replace(
        "app = FastAPI(",
        runtime_block + "\n\napp = FastAPI("
    )

APP_PATH.write_text(txt, encoding="utf-8")
print("app.py patched for desktop executable mode.")


# --------------------------------------------------
# 2. Create desktop_app.py
# --------------------------------------------------

desktop_app = r'''
import os
import sys
import time
import socket
import threading
from pathlib import Path

import uvicorn
import webview

from app import app


APP_TITLE = "AnapathTwin Hospital App"
HOST = "127.0.0.1"
PORT = 8000
URL = f"http://{HOST}:{PORT}"


def get_runtime_dir():
    if getattr(sys, "frozen", False):
        return Path(sys.executable).resolve().parent
    return Path(__file__).resolve().parent


def ensure_dirs():
    base = get_runtime_dir()
    (base / "data").mkdir(exist_ok=True)
    (base / "logs").mkdir(exist_ok=True)
    (base / "backups").mkdir(exist_ok=True)


def is_port_open(host=HOST, port=PORT):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(0.5)
        return sock.connect_ex((host, port)) == 0


def run_server():
    log_dir = get_runtime_dir() / "logs"
    log_dir.mkdir(exist_ok=True)

    uvicorn.run(
        app,
        host=HOST,
        port=PORT,
        log_level="warning",
        access_log=False
    )


def wait_for_server(timeout=30):
    start = time.time()
    while time.time() - start < timeout:
        if is_port_open():
            return True
        time.sleep(0.5)
    return False


def main():
    ensure_dirs()

    if not is_port_open():
        server_thread = threading.Thread(target=run_server, daemon=True)
        server_thread.start()

    if not wait_for_server():
        webview.create_window(
            "Erreur AnapathTwin",
            html="""
            <html>
            <body style='font-family:Arial;padding:30px'>
              <h2>Erreur de démarrage</h2>
              <p>Le serveur local n'a pas pu démarrer.</p>
              <p>Vérifiez que le port 8000 n'est pas déjà utilisé.</p>
            </body>
            </html>
            """,
            width=700,
            height=400
        )
        webview.start()
        return

    webview.create_window(
        APP_TITLE,
        URL,
        width=1450,
        height=900,
        min_size=(1100, 700),
        confirm_close=True
    )

    webview.start()

    # When app window closes, terminate process cleanly
    os._exit(0)


if __name__ == "__main__":
    main()
'''

(PROJECT_DIR / "desktop_app.py").write_text(desktop_app, encoding="utf-8")
print("desktop_app.py created.")


# --------------------------------------------------
# 3. Desktop requirements
# --------------------------------------------------

requirements_desktop = r'''
fastapi==0.115.6
uvicorn[standard]==0.34.0
python-multipart==0.0.20
pywebview
pyinstaller
'''

(PROJECT_DIR / "requirements_desktop.txt").write_text(requirements_desktop.strip(), encoding="utf-8")
print("requirements_desktop.txt created.")


# --------------------------------------------------
# 4. Build script for Windows EXE
# --------------------------------------------------

build_bat = r'''
@echo off
title Build AnapathTwin Hospital App EXE

cd /d "%~dp0"

echo ==========================================
echo   Build AnapathTwin Hospital App
echo   Creation de l'application Windows
echo ==========================================
echo.

if not exist logs mkdir logs

echo Verification Python...
python --version
IF ERRORLEVEL 1 (
    echo ERREUR: Python introuvable. Installez Python 3.11 ou 3.12 puis relancez.
    pause
    exit /b
)

echo.
echo Creation environnement de build...
if not exist ".venv_build\Scripts\python.exe" (
    python -m venv .venv_build
)

call .venv_build\Scripts\activate

echo.
echo Installation dependances desktop...
python -m pip install --upgrade pip
python -m pip install -r requirements_desktop.txt

IF ERRORLEVEL 1 (
    echo ERREUR installation dependances.
    pause
    exit /b
)

echo.
echo Nettoyage anciens builds...
if exist build rmdir /s /q build
if exist dist rmdir /s /q dist

echo.
echo Creation EXE avec PyInstaller...
pyinstaller ^
  --noconfirm ^
  --clean ^
  --windowed ^
  --onedir ^
  --name "AnapathTwin_Hospital_App" ^
  --add-data "frontend;frontend" ^
  desktop_app.py

IF ERRORLEVEL 1 (
    echo.
    echo ERREUR pendant la creation EXE.
    echo Essayez avec Python 3.11 ou 3.12 si vous utilisez Python 3.13.
    pause
    exit /b
)

echo.
echo Preparation dossier final...
if not exist "dist\AnapathTwin_Hospital_App\data" mkdir "dist\AnapathTwin_Hospital_App\data"
if not exist "dist\AnapathTwin_Hospital_App\logs" mkdir "dist\AnapathTwin_Hospital_App\logs"
if not exist "dist\AnapathTwin_Hospital_App\backups" mkdir "dist\AnapathTwin_Hospital_App\backups"

copy README_REAL_APP.txt "dist\AnapathTwin_Hospital_App\README_REAL_APP.txt" >nul

echo.
echo ==========================================
echo BUILD TERMINE
echo ==========================================
echo.
echo Votre application est ici:
echo dist\AnapathTwin_Hospital_App\AnapathTwin_Hospital_App.exe
echo.
echo Testez-la maintenant.
echo.
pause
'''

(PROJECT_DIR / "BUILD_WINDOWS_EXE.bat").write_text(build_bat, encoding="utf-8")
print("BUILD_WINDOWS_EXE.bat created.")


# --------------------------------------------------
# 5. Development run script
# --------------------------------------------------

run_dev = r'''
@echo off
title Run AnapathTwin Desktop App - Dev Mode

cd /d "%~dp0"

if not exist ".venv_build\Scripts\python.exe" (
    python -m venv .venv_build
)

call .venv_build\Scripts\activate

python -m pip install -r requirements_desktop.txt

python desktop_app.py

pause
'''

(PROJECT_DIR / "RUN_DESKTOP_APP_DEV.bat").write_text(run_dev, encoding="utf-8")


# --------------------------------------------------
# 6. README
# --------------------------------------------------

readme = r'''
ANAPATHTWIN HOSPITAL APP - REAL DESKTOP VERSION

Objectif:
Transformer le système FastAPI + interface française en application Windows exécutable.

ETAPE 1 - Tester en mode desktop dev

Double-cliquer:
RUN_DESKTOP_APP_DEV.bat

Cela ouvre l'application dans une vraie fenêtre desktop.

ETAPE 2 - Créer le .EXE

Double-cliquer:
BUILD_WINDOWS_EXE.bat

Après compilation, l'application finale sera ici:

dist\AnapathTwin_Hospital_App\AnapathTwin_Hospital_App.exe

ETAPE 3 - Tester l'application finale

Ouvrir:
dist\AnapathTwin_Hospital_App\AnapathTwin_Hospital_App.exe

Login:
admin / admin123

IMPORTANT POUR HOPITAL

- Utiliser seulement des IDs anonymisés.
- Ne pas saisir nom, CIN, téléphone ou adresse.
- Ne pas exposer l'application sur Internet.
- Pour le pilote: hébergement local/intranet uniquement.

DONNEES

La base SQLite sera créée ici dans le dossier final:
data\hospital_twin.db

Les logs:
logs\

Sauvegardes:
backups\

SI BUILD ECHOUE

1. Vérifier Python:
   python --version

2. Si Python 3.13 pose problème avec PyInstaller:
   Installer Python 3.11 ou 3.12
   Puis relancer BUILD_WINDOWS_EXE.bat

3. Si l'application ne s'ouvre pas:
   Vérifier que le port 8000 n'est pas déjà utilisé.
'''

(PROJECT_DIR / "README_REAL_APP.txt").write_text(readme, encoding="utf-8")


# --------------------------------------------------
# 7. Create builder ZIP
# --------------------------------------------------

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
zip_base = EXPORT_DIR / f"AnapathTwin_Real_Desktop_App_Builder_v0_7_{timestamp}"

shutil.make_archive(str(zip_base), "zip", PROJECT_DIR)

print("Real app builder package created:")
print(str(zip_base) + ".zip")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project: /content/drive/MyDrive/pfe_hospital_anapath_real_system
Backup backend: /content/drive/MyDrive/pfe_hospital_anapath_real_system/app.py.backup_desktop_20260514_171011
app.py patched for desktop executable mode.
desktop_app.py created.
requirements_desktop.txt created.
BUILD_WINDOWS_EXE.bat created.
Real app builder package created:
/content/drive/MyDrive/AnapathTwin_Exports/AnapathTwin_Real_Desktop_App_Builder_v0_7_20260514_1710.zip


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime
import shutil
import re

PROJECT_DIR = Path("/content/drive/MyDrive/pfe_hospital_anapath_real_system")
EXPORT_DIR = Path("/content/drive/MyDrive/AnapathTwin_Exports")
INSTALLER_DIR = PROJECT_DIR / "installer"

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
INSTALLER_DIR.mkdir(parents=True, exist_ok=True)

APP_PATH = PROJECT_DIR / "app.py"
FRONTEND_DIR = PROJECT_DIR / "frontend"

if not APP_PATH.exists():
    raise Exception("app.py introuvable.")
if not FRONTEND_DIR.exists():
    raise Exception("frontend introuvable.")

print("Projet:", PROJECT_DIR)

# ==================================================
# 1. Patch app.py: database/logs outside Program Files
# ==================================================

txt = APP_PATH.read_text(encoding="utf-8")

backup_path = PROJECT_DIR / f"app.py.backup_installer_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
shutil.copy2(APP_PATH, backup_path)
print("Backup backend:", backup_path)

if "import sys" not in txt:
    txt = txt.replace("import time", "import time\nimport sys")

if "import os" not in txt:
    txt = txt.replace("import json", "import json\nimport os")

runtime_block = r'''
def get_appdata_dir():
    """
    Dossier de données local utilisateur.
    Important: en mode installé, on ne stocke pas la base dans Program Files.
    """
    base = os.environ.get("LOCALAPPDATA") or os.environ.get("APPDATA") or str(Path.home())
    return Path(base) / "AnapathTwin_Hospital_App"

def get_runtime_dir():
    """
    Dossier où seront stockés data/logs/backups.
    En développement: dossier projet.
    En application installée: AppData local utilisateur.
    """
    if getattr(sys, "frozen", False):
        return get_appdata_dir()
    return Path(__file__).resolve().parent

def get_resource_dir():
    """
    Dossier des ressources frontend.
    En PyInstaller: sys._MEIPASS.
    En développement: dossier projet.
    """
    if hasattr(sys, "_MEIPASS"):
        return Path(sys._MEIPASS)
    return Path(__file__).resolve().parent

PROJECT_DIR = get_runtime_dir()
RESOURCE_DIR = get_resource_dir()
DB_PATH = PROJECT_DIR / "data" / "hospital_twin.db"
FRONTEND_DIR = RESOURCE_DIR / "frontend"
'''

# Replace previous runtime block if it exists
patterns = [
    r'def get_appdata_dir\(\):.*?FRONTEND_DIR\s*=\s*RESOURCE_DIR\s*/\s*"frontend"',
    r'def get_runtime_dir\(\):.*?FRONTEND_DIR\s*=\s*RESOURCE_DIR\s*/\s*"frontend"',
    r'PROJECT_DIR\s*=\s*Path\(__file__\)\.resolve\(\)\.parent\s*\nDB_PATH\s*=\s*PROJECT_DIR\s*/\s*"data"\s*/\s*"hospital_twin\.db"\s*\nFRONTEND_DIR\s*=\s*PROJECT_DIR\s*/\s*"frontend"'
]

patched = False
for pat in patterns:
    new_txt = re.sub(pat, runtime_block.strip(), txt, flags=re.DOTALL)
    if new_txt != txt:
        txt = new_txt
        patched = True
        break

if not patched and "PROJECT_DIR = get_runtime_dir()" not in txt:
    txt = txt.replace("app = FastAPI(", runtime_block + "\n\napp = FastAPI(")

APP_PATH.write_text(txt, encoding="utf-8")
print("Backend patché pour installation Windows.")

# ==================================================
# 2. Create desktop_app.py
# ==================================================

desktop_app = r'''
import os
import sys
import time
import socket
import threading
from pathlib import Path

import uvicorn
import webview

from app import app


APP_TITLE = "AnapathTwin Hospital App"
HOST = "127.0.0.1"
PORT = 8000
URL = f"http://{HOST}:{PORT}"


def get_appdata_dir():
    base = os.environ.get("LOCALAPPDATA") or os.environ.get("APPDATA") or str(Path.home())
    return Path(base) / "AnapathTwin_Hospital_App"


def get_runtime_dir():
    if getattr(sys, "frozen", False):
        return get_appdata_dir()
    return Path(__file__).resolve().parent


def ensure_dirs():
    base = get_runtime_dir()
    (base / "data").mkdir(parents=True, exist_ok=True)
    (base / "logs").mkdir(parents=True, exist_ok=True)
    (base / "backups").mkdir(parents=True, exist_ok=True)


def is_port_open(host=HOST, port=PORT):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(0.5)
        return sock.connect_ex((host, port)) == 0


def run_server():
    ensure_dirs()
    uvicorn.run(
        app,
        host=HOST,
        port=PORT,
        log_level="warning",
        access_log=False
    )


def wait_for_server(timeout=30):
    start = time.time()
    while time.time() - start < timeout:
        if is_port_open():
            return True
        time.sleep(0.5)
    return False


def main():
    ensure_dirs()

    if not is_port_open():
        server_thread = threading.Thread(target=run_server, daemon=True)
        server_thread.start()

    if not wait_for_server():
        webview.create_window(
            "Erreur AnapathTwin",
            html="""
            <html>
            <body style='font-family:Arial;padding:30px'>
              <h2>Erreur de démarrage</h2>
              <p>Le serveur local n'a pas pu démarrer.</p>
              <p>Vérifiez que le port 8000 n'est pas déjà utilisé.</p>
            </body>
            </html>
            """,
            width=700,
            height=400
        )
        webview.start()
        return

    webview.create_window(
        APP_TITLE,
        URL,
        width=1450,
        height=900,
        min_size=(1100, 700),
        confirm_close=True
    )

    webview.start()
    os._exit(0)


if __name__ == "__main__":
    main()
'''

(PROJECT_DIR / "desktop_app.py").write_text(desktop_app, encoding="utf-8")
print("desktop_app.py créé.")

# ==================================================
# 3. Requirements for desktop build
# ==================================================

requirements_desktop = r'''
fastapi==0.115.6
uvicorn[standard]==0.34.0
python-multipart==0.0.20
pywebview
pyinstaller
'''

(PROJECT_DIR / "requirements_desktop.txt").write_text(requirements_desktop.strip(), encoding="utf-8")
print("requirements_desktop.txt créé.")

# ==================================================
# 4. Backup script for installed app
# ==================================================

backup_installed = r'''
from pathlib import Path
from datetime import datetime
import shutil
import os

base = Path(os.environ.get("LOCALAPPDATA") or os.environ.get("APPDATA") or str(Path.home())) / "AnapathTwin_Hospital_App"
db = base / "data" / "hospital_twin.db"
backup_dir = base / "backups"
backup_dir.mkdir(parents=True, exist_ok=True)

if not db.exists():
    print("Base introuvable:", db)
    raise SystemExit(1)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_path = backup_dir / f"hospital_twin_backup_{timestamp}.db"
shutil.copy2(db, backup_path)

print("Sauvegarde créée:", backup_path)

backups = sorted(backup_dir.glob("hospital_twin_backup_*.db"))
if len(backups) > 30:
    for old in backups[:-30]:
        old.unlink()
'''

(PROJECT_DIR / "backup_installed_database.py").write_text(backup_installed, encoding="utf-8")

backup_bat = r'''
@echo off
title Sauvegarde AnapathTwin
cd /d "%~dp0"

echo Sauvegarde de la base locale AnapathTwin...
python backup_installed_database.py
pause
'''

(PROJECT_DIR / "SAUVEGARDE_BASE_ANAPATHTWIN.bat").write_text(backup_bat, encoding="utf-8")

# ==================================================
# 5. Inno Setup installer script
# ==================================================

iss_script = r'''
#define MyAppName "AnapathTwin Hospital App"
#define MyAppVersion "0.8"
#define MyAppPublisher "PFE Digital Twin - Anapath"
#define MyAppExeName "AnapathTwin_Hospital_App.exe"

[Setup]
AppId={{B92B6F63-56F5-4D3D-9A59-ANAPATHTWIN08}
AppName={#MyAppName}
AppVersion={#MyAppVersion}
AppPublisher={#MyAppPublisher}
DefaultDirName={localappdata}\Programs\AnapathTwin Hospital App
DefaultGroupName=AnapathTwin Hospital App
DisableProgramGroupPage=yes
OutputDir=installer_output
OutputBaseFilename=AnapathTwin_Hospital_App_Setup_v0_8
Compression=lzma
SolidCompression=yes
WizardStyle=modern
PrivilegesRequired=lowest
SetupIconFile=
UninstallDisplayIcon={app}\{#MyAppExeName}
LicenseFile=README_INSTALLER_HOPITAL.txt

[Languages]
Name: "french"; MessagesFile: "compiler:Languages\French.isl"
Name: "english"; MessagesFile: "compiler:Default.isl"

[Tasks]
Name: "desktopicon"; Description: "Créer un raccourci sur le Bureau"; GroupDescription: "Raccourcis:"; Flags: unchecked

[Files]
Source: "dist\AnapathTwin_Hospital_App\*"; DestDir: "{app}"; Flags: ignoreversion recursesubdirs createallsubdirs

[Icons]
Name: "{autoprograms}\AnapathTwin Hospital App"; Filename: "{app}\{#MyAppExeName}"; WorkingDir: "{app}"
Name: "{autodesktop}\AnapathTwin Hospital App"; Filename: "{app}\{#MyAppExeName}"; WorkingDir: "{app}"; Tasks: desktopicon
Name: "{autoprograms}\Dossier données AnapathTwin"; Filename: "{localappdata}\AnapathTwin_Hospital_App"
Name: "{autoprograms}\Dossier installation AnapathTwin"; Filename: "{app}"

[Run]
Filename: "{app}\{#MyAppExeName}"; Description: "Lancer AnapathTwin Hospital App"; Flags: nowait postinstall skipifsilent

[UninstallDelete]
Type: filesandordirs; Name: "{app}"
'''

(INSTALLER_DIR / "AnapathTwin_Hospital_App.iss").write_text(iss_script, encoding="utf-8")
print("Script Inno Setup créé.")

# ==================================================
# 6. Installer README/license
# ==================================================

readme_installer = r'''
ANAPATHTWIN HOSPITAL APP - INSTALLATEUR

Version pilote hospitalière locale.

IMPORTANT

Cette application est un prototype de support décisionnel pour l'anatomopathologie
et la médecine de précision du cancer du poumon.

Elle ne remplace pas:
- la décision médicale,
- la réunion de concertation pluridisciplinaire,
- les protocoles hospitaliers,
- la validation anatomopathologique ou oncologique.

UTILISATION DES DONNEES

Pour la phase pilote:
- utiliser uniquement des identifiants anonymisés,
- ne pas saisir de nom, CIN, téléphone, adresse ou information directement identifiante,
- ne pas exposer l'application sur Internet,
- utiliser uniquement en local ou intranet hospitalier.

STOCKAGE DES DONNEES

La base SQLite est stockée dans le dossier utilisateur local:

%LOCALAPPDATA%\AnapathTwin_Hospital_App\data\hospital_twin.db

Les sauvegardes sont stockées dans:

%LOCALAPPDATA%\AnapathTwin_Hospital_App\backups

COMPTES DEMO

admin / admin123
pathologist / patho123
oncologist / onco123
researcher / research123

Changer les mots de passe avant toute utilisation réelle.
'''

(PROJECT_DIR / "README_INSTALLER_HOPITAL.txt").write_text(readme_installer, encoding="utf-8")

# ==================================================
# 7. Build full EXE + installer script
# ==================================================

build_full = r'''
@echo off
title Build AnapathTwin Installer

cd /d "%~dp0"

echo ==========================================
echo   Build AnapathTwin Hospital Installer
echo ==========================================
echo.

echo Verification Python...
python --version
IF ERRORLEVEL 1 (
    echo ERREUR: Python introuvable.
    echo Installez Python 3.11 ou 3.12 puis relancez.
    pause
    exit /b
)

echo.
echo Creation environnement de build...
if not exist ".venv_build\Scripts\python.exe" (
    python -m venv .venv_build
)

call .venv_build\Scripts\activate

echo.
echo Installation dependances...
python -m pip install --upgrade pip
python -m pip install -r requirements_desktop.txt

IF ERRORLEVEL 1 (
    echo ERREUR installation dependances.
    pause
    exit /b
)

echo.
echo Nettoyage anciens builds...
if exist build rmdir /s /q build
if exist dist rmdir /s /q dist

echo.
echo Creation application Windows...
pyinstaller ^
  --noconfirm ^
  --clean ^
  --windowed ^
  --onedir ^
  --name "AnapathTwin_Hospital_App" ^
  --add-data "frontend;frontend" ^
  desktop_app.py

IF ERRORLEVEL 1 (
    echo.
    echo ERREUR PyInstaller.
    echo Si vous utilisez Python 3.13, essayez Python 3.11 ou 3.12.
    pause
    exit /b
)

echo.
echo Ajout fichiers support...
copy README_INSTALLER_HOPITAL.txt "dist\AnapathTwin_Hospital_App\README_INSTALLER_HOPITAL.txt" >nul
copy backup_installed_database.py "dist\AnapathTwin_Hospital_App\backup_installed_database.py" >nul
copy SAUVEGARDE_BASE_ANAPATHTWIN.bat "dist\AnapathTwin_Hospital_App\SAUVEGARDE_BASE_ANAPATHTWIN.bat" >nul

echo.
echo Recherche Inno Setup...
set "ISCC="

if exist "%ProgramFiles(x86)%\Inno Setup 6\ISCC.exe" set "ISCC=%ProgramFiles(x86)%\Inno Setup 6\ISCC.exe"
if exist "%ProgramFiles%\Inno Setup 6\ISCC.exe" set "ISCC=%ProgramFiles%\Inno Setup 6\ISCC.exe"

if not defined ISCC (
    echo.
    echo ERREUR: Inno Setup 6 introuvable.
    echo Installez Inno Setup 6 puis relancez ce script.
    echo Le fichier .iss est pret dans:
    echo installer\AnapathTwin_Hospital_App.iss
    pause
    exit /b
)

echo Inno Setup trouve:
echo %ISCC%

echo.
echo Creation installateur...
"%ISCC%" "installer\AnapathTwin_Hospital_App.iss"

IF ERRORLEVEL 1 (
    echo.
    echo ERREUR creation installateur.
    pause
    exit /b
)

echo.
echo ==========================================
echo BUILD TERMINE
echo ==========================================
echo.
echo Installateur cree ici:
echo installer\installer_output\AnapathTwin_Hospital_App_Setup_v0_8.exe
echo.
pause
'''

(PROJECT_DIR / "BUILD_CLEAN_INSTALLER_WINDOWS.bat").write_text(build_full, encoding="utf-8")
print("BUILD_CLEAN_INSTALLER_WINDOWS.bat créé.")

# ==================================================
# 8. Test before installer script
# ==================================================

test_desktop = r'''
@echo off
title Test AnapathTwin Desktop App

cd /d "%~dp0"

if not exist ".venv_build\Scripts\python.exe" (
    python -m venv .venv_build
)

call .venv_build\Scripts\activate

python -m pip install -r requirements_desktop.txt

python desktop_app.py

pause
'''

(PROJECT_DIR / "TEST_DESKTOP_APP_BEFORE_INSTALLER.bat").write_text(test_desktop, encoding="utf-8")

# ==================================================
# 9. Zip builder package
# ==================================================

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
zip_base = EXPORT_DIR / f"AnapathTwin_Clean_Installer_Builder_v0_8_{timestamp}"

shutil.make_archive(str(zip_base), "zip", PROJECT_DIR)

print("Package builder installateur créé:")
print(str(zip_base) + ".zip")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Projet: /content/drive/MyDrive/pfe_hospital_anapath_real_system
Backup backend: /content/drive/MyDrive/pfe_hospital_anapath_real_system/app.py.backup_installer_20260514_173839
Backend patché pour installation Windows.
desktop_app.py créé.
requirements_desktop.txt créé.
Script Inno Setup créé.
BUILD_CLEAN_INSTALLER_WINDOWS.bat créé.
Package builder installateur créé:
/content/drive/MyDrive/AnapathTwin_Exports/AnapathTwin_Clean_Installer_Builder_v0_8_20260514_1738.zip
